# AGPQuant v5 — which fidelity strategy works best when it is *trained*, not just swapped?

**Why this notebook exists.** The previous retrieval ablation trained the circuit once (with a pure-fidelity loss, on the 10–50 K-shot patients) and then only swapped the similarity at inference. Every mechanism read the same frozen states, so every mechanism scored about the same (≈ 0.78). This notebook follows the design that produced clearly different outcomes in the classical Q1 notebook: **the similarity is put inside the training loss**, and each trained model is then scored with its own (matched) similarity.

**What is fixed.** The circuit is Q1's `H1_quantum` head: per pathology, a 128→12 angle map, 6 qubits × 3 layers of RY/RZ(x) → RY/RZ(θ) → CNOT ring. It runs on the **pre-adapted** encoder features (Stage A, unchanged recipe). All circuit evaluations run as PennyLane QNodes on `default.qubit` — a statevector simulator, not hardware.

**What varies (the training arms).** The same circuit, same initial weights and same batches, trained on the 1,600-patient preadapt cohort with a different similarity inside the episodic leave-one-out loss:

| Arm | Similarity inside the training loss | Family |
|---|---|---|
| T0_readout | none — BCE + ranking on the ⟨Z⟩ readout (H1 as published) | baseline |
| T1_pure_fidelity | \|⟨ψ\|φ⟩\|² between pure states | quantum |
| T2_mixed_uhlmann | Uhlmann fidelity between six-view mixed states | quantum |
| T3_mixed_bures | Bures distance (ranks like Uhlmann, trains differently) | quantum |
| T4_mixed_hilbert_schmidt | normalised Tr(ρσ) | quantum |
| T5_mixed_trace_distance | ½‖ρ − σ‖₁ | quantum |
| C1 / C2 / C3 angle cosine / Euclidean / Mahalanobis | cosine / Euclidean / Mahalanobis of the 12 input angles (circuit unused) | classical twins |
| C0_raw_feature | cosine, Euclidean and Mahalanobis of the 128-d feature, untrained (the Q1 classical rules) | no circuit |
| P0_prior_only | CheXzero zero-shot alone (reference) | prior |

Every arm × 3 model seeds. Every trained model is scored with every rule (the **cross grid**); the headline is the **matched diagonal**.

**Endpoints.** Primary: retrieval macro mAP of the matched rule (each positive query ranks the full gallery). Secondary: P@k, R@k, MRR, nDCG@10; K-shot prototype classification (AUROC, AUPRC, balanced accuracy) at 1- and 5-shot, with and without the CheXzero prior. Diagnostics: purity of the mixed states, similarity spread, and Kendall τ between the rankings of every pair of rules.

**Ablations** (on the best arm, chosen on SELECT): A1 no entanglement · A2 frozen θ · A3 how the mixture is built (patch-only views; whole + patch pair) · A4 K-shot fine-tuning · A5 original (non-pre-adapted) encoders.

**Complexity (Cell 13).** Operations per pair, whether the rule supports nearest-neighbour indexing, hardware qubits / gates / depth / shots, measured time per pair, and an accuracy-vs-cost figure.

**What runs in PennyLane.** Pure states (`qml.state`), mixed states (a purification circuit read out with `qml.density_matrix`) and the readout (`qml.expval`) are QNodes. Every quantum similarity — inside the training loss and in every score — is a PennyLane quantum-information function: `qml.math.fidelity_statevector`, `qml.math.fidelity` (Uhlmann; Bures from it), `qml.math.trace_distance`, and `qml.math.purity` with `qml.math.einsum` (Hilbert–Schmidt). An independent dense torch implementation re-checks sampled pairs of every scoring call, and simulator circuits (compute–uncompute overlap, 12-qubit Hilbert–Schmidt) measure sampled pairs of every model. Cosine, Euclidean and Mahalanobis are classical by design. Uhlmann fidelity and trace distance have no exact measurement circuit on hardware.

**Review items addressed.** Prior-only baseline (P0) and +prior-vs-P0 tests; quantum scores are the evaluated scores; one protocol for every arm (comparable); FINAL scored once in FULL; "K-shot after supervised target pre-adaptation" wording, with the readout (no supports) reported as shot 0; Mahalanobis reported (C3, C0); 3 seeds with a same-sign rule; CheXzero checkpoint provenance flagged for disclosure.

**Run order.** `RUN_MODE = "SMOKE"` first (≈ 10–20 min, proves every code path), then `"FULL"` in a fresh session. Finished models and results are saved; a stopped run resumes (attach the previous output and set `RESUME_FROM`). Only a completed FULL run is evidence. The circuit choice and the pre-registration hash need Dr. Nimi's confirmation before FULL.

| Cell | Content |
|---|---|
| 1–2 | configuration, helpers |
| 3–4 | quantum core (QNodes, equivalence gate), similarity library and self-tests |
| 5–8 | cohorts, Stage A encoder pre-adaptation, routing and mixture views, CheXzero prior |
| 9 | Stage B — train every arm × seed |
| 10 | Stage C — evaluate every model with every rule |
| 11 | ablations A1–A5 |
| 12 | tables, pre-registered tests, diagnostics, provenance |
| 13 | complexity and efficiency |

**Timed v6:** SMOKE runs a reduced four-arm diagnostic with progress every 30 seconds and a 20-minute cap. FULL runs all nine training arms (two seeds), then attempts optional ablations in the remaining time. It stops before 7 hours, which may leave partial results; inspect the model counts before making scientific claims.


---

## SHARD 3_T5_trace_distance — trace distance (mixed)

Arms in this notebook: `T5_mixed_trace_distance`

Runs standalone: Stage A, training, evaluation, its own ablations A1-A5 and A4, and the full
classification report (prior / no prior, 1-shot / 5-shot, accuracy, AUROC, AUPRC, mAP, nDCG,
bootstrap CIs). Cross-arm families F1-F4 compare arms to each other, so they finish only in a
merge run: attach all five shard outputs and pass them to `RESUME_FROM` as a list.

Unreviewed. The circuit is still `pending_Dr_Nimi`, and the similarity engine changed, so the
pre-registration hash must be regenerated and re-approved before anything here is reported.


## Setup

In [1]:
# =============================================================================
# Cell 1 — Configuration
#   Every experimental setting lives in this cell: paths, run mode, cohorts, the fixed
#   circuit, the training arms, the scoring rules, the ablations, the statistics and the
#   runtime budget. Later cells read these names and never introduce new settings.
# =============================================================================
from __future__ import annotations

import os
import time
from pathlib import Path

# Start the clock and a visible heartbeat before heavy imports or checkpoint reads.
import threading
RUN_T0 = time.time()
RUN_STAGE = {'name': 'imports / package checks'}
PROGRESS_LOG = Path('/kaggle/working/agpquant_progress.log') if Path('/kaggle/working').is_dir() else Path('/tmp/agpquant_progress.log')
def progress(message):
    line = f"[AGPQuant t+{(time.time()-RUN_T0)/60:.1f} min] {message}"
    print(line, flush=True)
    try:
        with PROGRESS_LOG.open('a') as handle:
            handle.write(line + '\n')
    except OSError:
        pass
def heartbeat():
    while True:
        time.sleep(30)
        progress(f"HEARTBEAT | stage: {RUN_STAGE['name']} | log: {PROGRESS_LOG}")
threading.Thread(target=heartbeat, daemon=True, name='agpquant-heartbeat').start()
progress('START | heartbeat every 30 seconds. SMOKE stops at 20 minutes; FULL stops before 7 hours.')

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import copy
import hashlib
import importlib
import json
import math
import random
import re
import subprocess
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.stats import kendalltau
from scipy.stats import t as student_t
from sklearn.metrics import average_precision_score, ndcg_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import vit_b_16
from torchvision.transforms import functional as TF


def ensure_module(module_name: str, pip_spec: str):
    """Import a module, installing it with pip first if it is missing.

    Runs at t+0 so a missing package fails immediately, not after the encoder stage.
    Installing needs Kaggle Settings -> Internet -> ON.
    """
    try:
        return importlib.import_module(module_name)
    except ImportError:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_spec],
                                capture_output=True, text=True, timeout=180)
        if result.returncode != 0:
            raise RuntimeError(f"pip could not install {pip_spec}. In Kaggle: Settings -> Internet -> ON, "
                               f"then Run All.\n{result.stderr[-800:]}")
        importlib.invalidate_caches()
        return importlib.import_module(module_name)


progress("Installing or importing PennyLane and open_clip")
PENNYLANE_VERSION = "0.42.3"
qml = ensure_module("pennylane", f"pennylane=={PENNYLANE_VERSION}")
open_clip = ensure_module("open_clip", "open_clip_torch")
from open_clip.model import build_model_from_openai_state_dict  # noqa: E402  (needs the install above)

# ----------------------------------------------------------------------------
# 1. Paths
# ----------------------------------------------------------------------------
STAGE1_ROOT = Path("/kaggle/input/datasets/zarinn/nih-checkpoint-vit/qfid_matched_stage1_nih_only")
T0_WHOLE_PATH = STAGE1_ROOT / "T0_NIH_whole_vit.pt"
T0_PATCH_PATH = STAGE1_ROOT / "T0_NIH_patch_vit.pt"
ROUTING_CSV = STAGE1_ROOT / "per_class_validation_metrics.csv"
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
CHEXZERO_CKPT = Path("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                     "best_64_5e-05_original_22000_0.864.pt")
OUT = Path("/kaggle/working/agpquant_v5_metric_trained_ablation")

# ----------------------------------------------------------------------------
# 2. Run mode — run SMOKE first (minutes, proves every code path), then FULL
# ----------------------------------------------------------------------------
RUN_MODE = "FULL"                        # "SMOKE" -> "FULL"; use a fresh Kaggle run for FULL
SMOKE = RUN_MODE == "SMOKE"

# ----------------------------------------------------------------------------
# 1b. SHARD — this notebook trains and evaluates ONE slice of the arm grid.
#     Run the five shard notebooks in five separate Kaggle sessions, in any order.
#     Each one is self-contained: Stage A, training, evaluation, its own ablations
#     A1-A5 + A4, and the full classification report (prior / no prior, 1-shot /
#     5-shot, accuracy, AUROC, AUPRC, mAP, nDCG, bootstrap CIs).
#     Cross-ARM tests (F1/F2/F3/F4) compare arms against each other, so they can
#     only be completed once every shard has run — see RESUME_FROM below.
# ----------------------------------------------------------------------------
SHARD_NAME = "3_T5_trace_distance"
SHARD_ARMS = ('T5_mixed_trace_distance',)                 # trace distance (mixed)
OUT = Path(f"/kaggle/working/agpquant_v6_shard_{SHARD_NAME}")
USE_LOWRANK = True     # exact rank-V mixed-state similarity in EVALUATION (see the patch cell)
SEED = 42
CLASSES = ("Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion")
N_CLASSES = len(CLASSES)
LABEL_COLS = tuple(f"label_{i}" for i in range(N_CLASSES))

# ----------------------------------------------------------------------------
# 3. Cohorts (patients). All cohorts are patient-disjoint.
#    preadapt : trains the encoders (Stage A) and then the circuit (Stage B)
#    support  : the only source of K-shot supports (never trains anything else)
#    gallery  : the fixed retrieval gallery
#    select   : epoch selection, beta tuning, best-arm choice — never reported as a result
#    dev / expert_valid / final : report splits (final is scored once, in FULL only)
# ----------------------------------------------------------------------------
if SMOKE:
    CAP = dict(preadapt=32, support=64, gallery=24, select=24, dev=24, final=24)
else:
    CAP = dict(preadapt=320, support=320, gallery=96, select=80, dev=80, final=80)
SPLIT_FRACTIONS = {"train": 0.55, "select": 0.15, "dev": 0.15, "final": 0.15}
USE_EXPERT_VALID = False     # optional expert split would multiply the expensive full cross-grid
RUN_FINAL_TEST = not SMOKE
GALLERY_MIN_PER_LABEL = 2 if SMOKE else 12    # known positives AND negatives per pathology in the gallery
EVAL_MIN_PER_LABEL = 2 if SMOKE else 8       # same floor for select / dev / final (v9 DEV had 3 Atelectasis negatives)

# ----------------------------------------------------------------------------
# 4. Stage A — supervised encoder pre-adaptation (unchanged recipe from the previous notebook)
# ----------------------------------------------------------------------------
ADAPT_EPOCHS = 1 if SMOKE else 2
ADAPT_BLOCKS = 2
LR_BACKBONE, LR_HEAD = 1e-5, 1e-4
LAMBDA_ANCHOR = 0.1
BATCH_WHOLE, BATCH_PATCH = (8, 4) if SMOKE else (32, 8)
SMOKE_MAX_STEPS = 2
ENCODER_SETS = ("original", "preadapted")
MAIN_ENCODER_SET = "preadapted"               # the circuit is trained on pre-adapted features
ROUTING_SOURCE = "nih_val_csv"                # "nih_val_csv" | "select" | "equal"
ROUTING_DEADBAND = 0.03
TARGET_CROP_BOXES = [(.05, .05, .55, .95), (.45, .05, .95, .95), (.10, .02, .90, .58),
                     (.10, .42, .90, .98), (.18, .18, .82, .82), (.28, .08, .72, .92)]
FEATURE_DIM = 128
EVAL_BATCH, WORKERS = 16, 0                   # workers=0 avoids Kaggle loader deadlocks

# ----------------------------------------------------------------------------
# 5. The fixed circuit — Q1's H1_quantum head, one copy per pathology
#    Linear(128 -> 12) -> tanh -> x*pi -> 3 x [RY(x) RZ(x) RY(theta) RZ(theta) on 6 qubits; CNOT ring]
#    Pure state   : the circuit applied to the routed feature
#    Mixed state  : the circuit applied to each crop view, mixed with that pathology's crop attention
# ----------------------------------------------------------------------------
N_QUBITS, VQC_LAYERS = 6, 3
STATE_DIM = 2 ** N_QUBITS
N_ANGLES = 2 * N_QUBITS
CIRCUIT_ID = "H1_quantum_6q3l_RY-RZ-reupload_RY-RZ-theta_CNOT-ring"
PL_DEVICE_NAME = "default.qubit"              # PennyLane statevector simulator (not hardware, no shots)
QNODE_TORCH_DEVICE = "auto"                   # "auto" tries CUDA for the QNodes and falls back to CPU
CIRCUIT_APPROVAL = "pending_Dr_Nimi"          # paper owner confirms the circuit before FULL


# ----------------------------------------------------------------------------
# 6. Training arms (Stage B) — the ONLY thing that varies in the main grid is the similarity
#    used inside the training loss. Every arm trains the same H1 circuit on the same patients.
# ----------------------------------------------------------------------------
@dataclass(frozen=True)
class TrainingArm:
    name: str
    family: str             # "readout" | "quantum" | "classical"
    representation: str     # "readout" | "pure" | "mixed" | "angles"
    metric: str | None      # similarity used in the episodic training loss (None for the readout arm)
    matched_rule: str       # scoring rule that reads the same quantity at test time
    description: str


TRAINING_ARMS = {arm.name: arm for arm in [
    TrainingArm("T0_readout", "readout", "readout", None, "readout",
                "H1 as published: BCE + ranking loss on the <Z> readout; no similarity in training"),
    TrainingArm("T1_pure_fidelity", "quantum", "pure", "fidelity", "pure_fidelity",
                "episodic loss on |<psi|phi>|^2 between pure circuit states"),
    TrainingArm("T2_mixed_uhlmann", "quantum", "mixed", "uhlmann", "mixed_uhlmann",
                "episodic loss on Uhlmann fidelity between six-view mixed states"),
    TrainingArm("T3_mixed_bures", "quantum", "mixed", "bures", "mixed_bures",
                "episodic loss on (negative) Bures distance; ranks like Uhlmann, trains differently"),
    TrainingArm("T4_mixed_hilbert_schmidt", "quantum", "mixed", "hilbert_schmidt", "mixed_hilbert_schmidt",
                "episodic loss on normalised Hilbert-Schmidt overlap Tr(rho sigma)/sqrt(Tr rho^2 Tr sigma^2)"),
    TrainingArm("T5_mixed_trace_distance", "quantum", "mixed", "trace_distance", "mixed_trace_distance",
                "episodic loss on (negative) trace distance 0.5 ||rho - sigma||_1"),
    TrainingArm("C1_angle_cosine", "classical", "angles", "cosine", "angle_cosine",
                "classical twin: episodic loss on cosine of the 12 circuit input angles (circuit unused)"),
    TrainingArm("C2_angle_euclidean", "classical", "angles", "euclidean", "angle_euclidean",
                "classical twin: episodic loss on (negative) Euclidean distance of the angles"),
    TrainingArm("C3_angle_mahalanobis", "classical", "angles", "mahalanobis", "angle_mahalanobis",
                "classical twin: episodic loss on (negative) Mahalanobis distance of the angles (shrunk batch covariance)"),
]}


# ----------------------------------------------------------------------------
# 7. Scoring rules (Stage C) — every trained model is scored with every rule (the cross grid).
#    The headline is the matched diagonal: trained with metric m, scored with metric m.
# ----------------------------------------------------------------------------
@dataclass(frozen=True)
class ScoringRule:
    name: str
    family: str
    representation: str     # "readout" | "pure" | "mixed" | "angles"
    metric: str | None
    needs_support: bool     # False only for the readout, which is a trained classifier
    description: str


SCORING_RULES = {rule.name: rule for rule in [
    ScoringRule("readout", "readout", "readout", None, False, "trained <Z> readout logit (classification only)"),
    ScoringRule("pure_fidelity", "quantum", "pure", "fidelity", True, "|<psi|phi>|^2"),
    ScoringRule("mixed_uhlmann", "quantum", "mixed", "uhlmann", True, "(Tr|sqrt(rho) sqrt(sigma)|)^2"),
    ScoringRule("mixed_bures", "quantum", "mixed", "bures", True, "-sqrt(2 - 2 sqrt(F_Uhlmann))"),
    ScoringRule("mixed_hilbert_schmidt", "quantum", "mixed", "hilbert_schmidt", True,
                "Tr(rho sigma) / sqrt(Tr rho^2 Tr sigma^2)"),
    ScoringRule("mixed_trace_distance", "quantum", "mixed", "trace_distance", True, "-0.5 ||rho - sigma||_1"),
    ScoringRule("angle_cosine", "classical", "angles", "cosine", True, "cosine of the circuit input angles"),
    ScoringRule("angle_euclidean", "classical", "angles", "euclidean", True, "-||a - b||_2 of the input angles"),
    ScoringRule("angle_mahalanobis", "classical", "angles", "mahalanobis", True,
                "-(a - b)^T P (a - b), P = shrunk inverse covariance of the support cohort (label-free)"),
]}
RETRIEVAL_RULES = tuple(name for name, rule in SCORING_RULES.items() if rule.needs_support)
QUANTUM_ARMS = tuple(name for name, arm in TRAINING_ARMS.items() if arm.family == "quantum")
MIXED_ARMS = tuple(name for name, arm in TRAINING_ARMS.items() if arm.representation == "mixed")

# Untrained reference rows (not training arms)
REFERENCE_RAW = "C0_raw_feature"                 # 128-d routed feature, no circuit, no training (the Q1 classical rules)
REFERENCE_RAW_RULES = ("angle_cosine", "angle_euclidean", "angle_mahalanobis")   # applied to the raw feature
REFERENCE_PRIOR_ONLY = "P0_prior_only"           # CheXzero zero-shot score alone

# ----------------------------------------------------------------------------
# 8. Stage B hyper-parameters (identical for every arm)
# ----------------------------------------------------------------------------
MODEL_SEEDS = (42,) if SMOKE else (42, 314)
CIRCUIT_EPOCHS = 1 if SMOKE else 4
CIRCUIT_PATIENCE = 3                          # epochs without SELECT improvement before stopping
CIRCUIT_BATCH = 12 if SMOKE else 16          # 64 -> 2,016 distinct pairs per pathology per step
PER_CLASS_QUOTA = 1 if SMOKE else 1           # labelled positives AND negatives per class in every batch
CIRCUIT_LR, CIRCUIT_WEIGHT_DECAY = 1e-2, 1e-3
SCALE_LR = 1e-2                               # temperature and bias of the episodic logit
GRAD_CLIP = 5.0
READOUT_RANK_WEIGHT = 0.30                    # T0 only: H1's original BCE + ranking recipe
MAHALANOBIS_SHRINK = 0.1                      # covariance shrinkage toward (mean variance) x I
SELECT_REFERENCE_SIZE = 12 if SMOKE else 64  # labelled preadapt patients used as supports when scoring SELECT


# ----------------------------------------------------------------------------
# 9. Ablations — each changes ONE thing about the best arm (chosen on SELECT, never on DEV)
# ----------------------------------------------------------------------------
@dataclass(frozen=True)
class Variant:
    name: str
    encoder_set: str = MAIN_ENCODER_SET
    entangle: bool = True
    train_theta: bool = True
    view_builder: str = "routed_views"
    applies_to: str = "best_quantum"         # "all" | "best_quantum" | "best_mixed"
    description: str = ""


MAIN_VARIANT = Variant("main", applies_to="all", description="the main grid")
ABLATIONS = {variant.name: variant for variant in [
    Variant("A1_no_entanglement", entangle=False, description="CNOT ring removed"),
    Variant("A2_frozen_theta", train_theta=False, description="theta frozen at its random start; only the angle map trains"),
    Variant("A3a_patch_only_views", view_builder="patch_only_views", applies_to="best_mixed",
            description="mixed state built from the six crops alone (no shared whole-image component)"),
    Variant("A3b_whole_patch_pair", view_builder="whole_patch_pair", applies_to="best_mixed",
            description="two-component mixture: whole-image state and pooled-patch state"),
    Variant("A5_original_encoder", encoder_set="original", description="NIH-only encoders, no Stage A"),
]}
A4_FINETUNE_SHOTS = (5,)                      # 1-shot has no second same-label support, so no episodic loss exists
A4_FINETUNE_STEPS = 2 if SMOKE else 10
A4_FINETUNE_LR = 5e-3

# ----------------------------------------------------------------------------
# 10. Stage C — evaluation
# ----------------------------------------------------------------------------
SHOTS = (1, 5)
EPISODES = 1 if SMOKE else 4
TUNING_EPISODES = 1 if SMOKE else 2          # separate support draws, used only to tune beta on SELECT
QUERY_CAP = 4 if SMOKE else 24              # positive queries per pathology and split; the gallery is never subsampled
TOP_K = (1, 5, 10)
BETA_GRID = (0.0, 0.25, 0.5, 1.0, 2.0, 4.0)
DIAGNOSTIC_QUERIES = 4 if SMOKE else 8       # queries per pathology used for rank-agreement diagnostics
PAIR_CHUNK = 64                               # query rows per similarity chunk
BOOTSTRAP_REPS = 100 if SMOKE else 500
SIGNFLIP_MC = 20000

# ----------------------------------------------------------------------------
# 11. PennyLane execution and audits
#    States: QNodes (qml.state, qml.density_matrix via a purification circuit, qml.expval).
#    Quantum similarities, in training AND scoring: qml.math quantum-information functions, on DEVICE.
#    Audits: an independent dense torch implementation re-checks sampled pairs of every scoring call;
#    the overlap and 12-qubit Hilbert-Schmidt circuits measure sampled pairs of every model.
# ----------------------------------------------------------------------------
SIMILARITY_ENGINE = "pennylane_qml_math"
PAIR_BATCH = 32 if SMOKE else 128            # (query, reference) pairs per qml.math call; lower if the GPU runs out of memory
AUDIT_PAIRS_PER_CALL = 8
CIRCUIT_AUDIT_PAIRS = 4
AUDIT_TOLERANCE = 5e-4

# ----------------------------------------------------------------------------
# 12. Runtime budget (Kaggle GPU sessions stop at 12 h)
# ----------------------------------------------------------------------------
MAX_RUNTIME_HOURS = (20.0 / 60.0) if SMOKE else 6.80                     # hard stop inside any long loop
STAGE_STOP_HOURS = (19.0 / 60.0) if SMOKE else 6.60                       # no new model is trained or evaluated after this
SMOKE_BUDGET_MINUTES = 20             # enforced by MAX_RUNTIME_HOURS
# To continue a stopped run in a NEW Kaggle session, attach the previous version's output as an input
# and point RESUME_FROM at it; cached features, trained circuits and finished results are then reused.
# Continuing THIS shard: attach its previous output and give that one path.
# MERGE run: attach all five shard outputs and pass them as a list — Cell 12 then sees
# every arm and the cross-arm families F1-F4 complete.
RESUME_FROM = [Path("/kaggle/input/datasets/omarradee/trace-output/agpquant_v6_shard_3_T5_trace_distance")]
if not RESUME_FROM[0].is_dir():                  # dataset not attached: fall back to auto-detect below
    print(f"  !! {RESUME_FROM[0]} not found; searching /kaggle/input instead")
    RESUME_FROM = None
# Auto-detect this shard's earlier output if it is attached as an input (depth <= 4, never the dataset tree).
_resume_hits = [p for pattern in ("*/", "*/*/", "*/*/*/", "*/*/*/*/")
                for p in Path("/kaggle/input").glob(pattern + OUT.name) if p.is_dir()]
if RESUME_FROM is None and _resume_hits:
    RESUME_FROM = _resume_hits
print("RESUME_FROM:", RESUME_FROM)
if RESUME_FROM is None:
    print("  !! No earlier output attached: Stage A and training will run from scratch (~30 min). "
          "To reuse Version 3's work, Add Input -> Your Work -> the 'trace' notebook output.")
else:
    _hits = [q.name for src in (RESUME_FROM if isinstance(RESUME_FROM, (list, tuple)) else [RESUME_FROM])
             for q in Path(src).rglob("*.pt")]
    print(f"  resuming: {len(_hits)} saved checkpoints found in the attached output")

# ----------------------------------------------------------------------------
# 13. Pre-registration — fixed before FULL; record the printed hash with Dr. Nimi
# ----------------------------------------------------------------------------
PRIMARY_SPLIT = "final" if RUN_FINAL_TEST else "dev"
PREREGISTRATION = {
    "question": ("When the same H1 circuit is trained with each similarity inside its training loss, "
                 "which similarity retrieves same-pathology patients best, and does any quantum similarity "
                 "beat its classical twin?"),
    "fixed_circuit": CIRCUIT_ID,
    "quantum_backend": f"pennylane=={PENNYLANE_VERSION} {PL_DEVICE_NAME} (statevector simulator)",
    "training_arms": list(TRAINING_ARMS),
    "scoring_rules": list(SCORING_RULES),
    "primary_endpoint": "retrieval macro mAP of the matched rule (arm trained with m, scored with m)",
    "primary_cell": {"split": PRIMARY_SPLIT, "encoder_set": MAIN_ENCODER_SET},
    "families": {
        "F1_quantum_vs_classical": "each quantum arm (matched) vs C1_angle_cosine (matched)",
        "F2_training_effect": "each quantum rule: arm trained with it vs T0_readout, both scored with it",
        "F3_no_circuit_control": "each arm (matched) vs cosine of the raw 128-d feature (C0, untrained)",
        "F4_quantum_pairwise": "all pairs of quantum arms (matched) — exploratory",
        "F5_prior": "each matched arm +prior vs P0_prior_only (K-shot classification AUROC)",
    },
    "test": ("two-sided paired two-level bootstrap (model seeds x query patients, gallery fixed) on the "
             "macro-mAP difference, Holm-corrected within each family; a difference counts only if the "
             "Holm p < alpha AND every model seed shows the same sign"),
    "alpha": 0.05,
    "model_seeds": list(MODEL_SEEDS),
    "best_arm_selection": "highest SELECT retrieval mAP (matched rule), mean over seeds — never DEV/FINAL",
    "gallery_min_known_per_label": GALLERY_MIN_PER_LABEL,
    "exploratory_only": ["cross-grid off-diagonal cells", "ablations A1-A5", "diagnostics"],
}

# ----------------------------------------------------------------------------
# 14. Device and determinism
# ----------------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU (Settings -> Accelerator -> GPU T4), then Run All.")
DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

for required_input in (T0_WHOLE_PATH, T0_PATCH_PATH, ROUTING_CSV, CHEXZERO_CKPT):
    if not required_input.is_file():
        raise FileNotFoundError(f"missing input: {required_input}")
progress("Configuration and checkpoint paths validated")
print(f"GPU: {torch.cuda.get_device_name(0)} | RUN_MODE: {RUN_MODE} | circuit approval: {CIRCUIT_APPROVAL}")
print(f"Circuit: {CIRCUIT_ID} | PennyLane {qml.__version__} on {PL_DEVICE_NAME} | open_clip {open_clip.__version__}")
print(f"Arms: {len(TRAINING_ARMS)} x seeds {MODEL_SEEDS} | scoring rules: {len(SCORING_RULES)} | "
      f"ablations: {list(ABLATIONS)} + A4 K-shot fine-tuning")
if qml.__version__ != PENNYLANE_VERSION:
    print(f"NOTE: expected PennyLane {PENNYLANE_VERSION}, found {qml.__version__} (the Cell 3 gate still decides)")


[AGPQuant t+0.0 min] START | heartbeat every 30 seconds. SMOKE stops at 20 minutes; FULL stops before 7 hours.
[AGPQuant t+0.2 min] Installing or importing PennyLane and open_clip


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:196: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


[AGPQuant t+0.5 min] HEARTBEAT | stage: imports / package checks | log: /kaggle/working/agpquant_progress.log
RESUME_FROM: [PosixPath('/kaggle/input/datasets/omarradee/trace-output/agpquant_v6_shard_3_T5_trace_distance')]
  resuming: 5 saved checkpoints found in the attached output
[AGPQuant t+0.9 min] Configuration and checkpoint paths validated
GPU: Tesla T4 | RUN_MODE: FULL | circuit approval: pending_Dr_Nimi
Circuit: H1_quantum_6q3l_RY-RZ-reupload_RY-RZ-theta_CNOT-ring | PennyLane 0.42.3 on default.qubit | open_clip 3.3.0
Arms: 9 x seeds (42, 314) | scoring rules: 9 | ablations: ['A1_no_entanglement', 'A2_frozen_theta', 'A3a_patch_only_views', 'A3b_whole_patch_pair', 'A5_original_encoder'] + A4 K-shot fine-tuning


In [2]:
# =============================================================================
# Cell 2 — Shared helpers: seeding, hashing, runtime guard, losses, metrics, statistics
# =============================================================================

# ---- seeding ----
def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(SEED)


# ---- hashing and files ----
def sha_of(obj) -> str:
    """Stable SHA-256 of any JSON-serialisable object (used for fingerprints)."""
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()


def file_sha256(path: Path, block: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(block):
            digest.update(chunk)
    return digest.hexdigest()


def tensor_sha256(tensor: torch.Tensor) -> str:
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()


def atomic_torch_save(obj, path: Path) -> None:
    """Write to a temporary file and rename, so a stopped session never leaves a half-written file."""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(obj, temporary)
    os.replace(temporary, path)


_RESUME_INDEX: dict[str, Path] | None = None


def find_saved_file(name: str) -> Path | None:
    """A file this notebook wrote earlier: in this session's run folder first, then in RESUME_FROM."""
    global _RESUME_INDEX
    local = RUN_DIR / name
    if local.is_file():
        return local
    for match in RUN_DIR.rglob(name):
        return match
    if RESUME_FROM is None:
        return None
    if _RESUME_INDEX is None:                                  # index the attached output(s) once
        sources = RESUME_FROM if isinstance(RESUME_FROM, (list, tuple)) else [RESUME_FROM]
        _RESUME_INDEX = {}
        for source in sources:
            _RESUME_INDEX.update({path.name: path for path in Path(source).rglob("*") if path.is_file()})
    return _RESUME_INDEX.get(name)


RUN_DIR = OUT / ("smoke" if SMOKE else "full")
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = RUN_DIR / "models"          # trained circuits (Stage B)
RESULT_DIR = RUN_DIR / "results"        # per-model evaluation results (Stage C)
for directory in (MODEL_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
PREREG_SHA = sha_of(PREREGISTRATION)
(RUN_DIR / "preregistration.json").write_text(json.dumps({**PREREGISTRATION, "sha256": PREREG_SHA}, indent=2))
print("Pre-registration sha256:", PREREG_SHA)


# ---- runtime guard ----
RUN_STAGE["name"] = "helpers and runtime guard"


def hours_used() -> float:
    return (time.time() - RUN_T0) / 3600


def elapsed() -> str:
    return f"t+{hours_used() * 60:.1f} min"


def enforce_runtime_budget(stage: str) -> None:
    """Hard stop inside long loops; everything finished so far is already on disk."""
    RUN_STAGE["name"] = stage
    if hours_used() >= MAX_RUNTIME_HOURS:
        progress(f"TIME LIMIT reached in {stage}; run stopped intentionally. Share {PROGRESS_LOG} and Kaggle output.")
        raise RuntimeError(f"Runtime guard stopped at {stage}: {hours_used():.2f} h >= {MAX_RUNTIME_HOURS} h. "
                           "Re-run the notebook: finished models and results are reused.")


def stage_budget_left() -> bool:
    """False once no new model should be started."""
    return hours_used() < STAGE_STOP_HOURS


# ---- losses ----
def masked_bce(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Binary cross-entropy over known labels only (NaN = unknown)."""
    known = torch.isfinite(labels)
    per_entry = F.binary_cross_entropy_with_logits(logits.float(), torch.nan_to_num(labels, nan=0.0), reduction="none")
    return (per_entry * known).sum() / known.sum().clamp_min(1)


def pairwise_rank_loss(scores: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Mean softplus(-(positive - negative)) over all positive/negative pairs, per class."""
    losses = []
    for class_index in range(N_CLASSES):
        known = torch.isfinite(labels[:, class_index])
        positives = scores[known & (labels[:, class_index] >= 0.5), class_index]
        negatives = scores[known & (labels[:, class_index] < 0.5), class_index]
        if len(positives) and len(negatives):
            losses.append(F.softplus(-(positives[:, None] - negatives[None, :])).mean())
    return torch.stack(losses).mean() if losses else scores.sum() * 0


# ---- classification metrics ----
def two_class_mask(y: np.ndarray, score: np.ndarray, class_index: int) -> np.ndarray | None:
    """Rows where the label is known and the score finite, or None if only one label value is present."""
    known = np.isfinite(y[:, class_index]) & np.isfinite(score[:, class_index])
    if known.sum() == 0 or len(np.unique(y[known, class_index])) < 2:
        return None
    return known


def macro_auc(y: np.ndarray, score: np.ndarray) -> float:
    y, score = np.asarray(y, float), np.asarray(score, float)
    values = []
    for class_index in range(y.shape[1]):
        known = two_class_mask(y, score, class_index)
        if known is not None:
            values.append(roc_auc_score(y[known, class_index], score[known, class_index]))
    return float(np.mean(values)) if values else np.nan


def classification_metrics(y: np.ndarray, score: np.ndarray, threshold: np.ndarray | None = None) -> dict:
    """Per-pathology AUROC, AUPRC, accuracy and balanced accuracy (score >= threshold is positive), plus macro means."""
    threshold = np.zeros((1, N_CLASSES)) if threshold is None else threshold
    row, aurocs, auprcs, accuracies, balanced = {}, [], [], [], []
    for class_index, name in enumerate(CLASSES):
        known = np.isfinite(y[:, class_index]) & np.isfinite(score[:, class_index])
        if not known.any():
            continue
        truth = y[known, class_index].astype(int)
        predicted = (score[known, class_index] >= threshold[0, class_index]).astype(int)
        row[f"accuracy__{name}"] = float((predicted == truth).mean())
        accuracies.append(row[f"accuracy__{name}"])
        if 0 < truth.sum() < len(truth):
            row[f"auroc__{name}"] = float(roc_auc_score(truth, score[known, class_index]))
            row[f"auprc__{name}"] = float(average_precision_score(truth, score[known, class_index]))
            sensitivity = (predicted[truth == 1] == 1).mean()
            specificity = (predicted[truth == 0] == 0).mean()
            row[f"balanced_accuracy__{name}"] = float(0.5 * (sensitivity + specificity))
            aurocs.append(row[f"auroc__{name}"])
            auprcs.append(row[f"auprc__{name}"])
            balanced.append(row[f"balanced_accuracy__{name}"])
    row["macro_auroc"] = float(np.mean(aurocs)) if aurocs else np.nan
    row["macro_auprc"] = float(np.mean(auprcs)) if auprcs else np.nan
    row["macro_accuracy"] = float(np.mean(accuracies)) if accuracies else np.nan
    row["macro_balanced_accuracy"] = float(np.mean(balanced)) if balanced else np.nan
    return row


def youden_thresholds(y: np.ndarray, score: np.ndarray) -> np.ndarray:
    """Per-class operating point maximising sensitivity + specificity - 1 (computed on SELECT only)."""
    thresholds = np.zeros((1, N_CLASSES), dtype=np.float32)
    for class_index in range(N_CLASSES):
        known = two_class_mask(y, score, class_index)
        if known is None:
            continue
        fpr, tpr, candidates = roc_curve(y[known, class_index].astype(int), score[known, class_index])
        finite = np.isfinite(candidates)
        if finite.any():
            best = np.flatnonzero(finite)[np.argmax((tpr - fpr)[finite])]
            thresholds[0, class_index] = candidates[best]
    return thresholds


# ---- retrieval metrics (one value per query) ----
RETRIEVAL_METRICS = ("map", "mrr") + tuple(f"p_at_{k}" for k in TOP_K) + tuple(f"r_at_{k}" for k in TOP_K) + ("ndcg_at_10",)


def per_query_retrieval(similarity: np.ndarray, relevance: np.ndarray) -> dict[str, np.ndarray]:
    """similarity (Q, R), larger = more similar; relevance (R,) in {0, 1}. Ties keep gallery order."""
    similarity = np.asarray(similarity, np.float64)
    relevance = np.asarray(relevance, np.int8)
    n_relevant, gallery_size = int(relevance.sum()), similarity.shape[1]
    assert n_relevant > 0 and gallery_size == len(relevance)
    ranked = relevance[np.argsort(-similarity, axis=1, kind="stable")].astype(np.float64)
    hits_so_far = np.cumsum(ranked, axis=1)
    ranks = np.arange(1, gallery_size + 1)
    out = {"map": (hits_so_far / ranks * ranked).sum(1) / n_relevant,
           "mrr": 1.0 / (ranked.argmax(1) + 1)}
    for k in TOP_K:
        kk = min(k, gallery_size)
        out[f"p_at_{k}"] = hits_so_far[:, kk - 1] / kk
        out[f"r_at_{k}"] = hits_so_far[:, kk - 1] / n_relevant
    kk = min(10, gallery_size)
    discount = 1 / np.log2(np.arange(2, kk + 2))
    out["ndcg_at_10"] = (ranked[:, :kk] * discount).sum(1) / discount[:min(n_relevant, kk)].sum()
    return {name: values.astype(np.float32) for name, values in out.items()}


# ---- statistics ----
def holm(p_values) -> np.ndarray:
    """Holm step-down adjustment; NaN p-values stay NaN and do not count toward the family size."""
    p = np.asarray(p_values, float)
    adjusted = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    running_max = 0.0
    for rank, index in enumerate(valid[np.argsort(p[valid], kind="stable")]):
        running_max = max(running_max, (len(valid) - rank) * p[index])
        adjusted[index] = min(1.0, running_max)
    return adjusted


def ci95_t(values) -> float:
    """Half-width of a Student-t 95% confidence interval of the mean."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    if len(v) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v)))


def signflip_p(differences, seed: int) -> float:
    """Two-sided paired sign-flip permutation p-value (exact for <= 16 pairs, Monte Carlo above)."""
    d = np.asarray(differences, float)
    d = d[np.isfinite(d)]
    if len(d) < 2:
        return np.nan
    observed = abs(d.mean())
    if len(d) <= 16:
        signs = np.asarray(list(product((1.0, -1.0), repeat=len(d))))
        return float((np.abs((signs * d).mean(1)) >= observed - 1e-12).mean())
    flips = np.random.default_rng(seed).choice((-1.0, 1.0), size=(SIGNFLIP_MC, len(d)))
    return float((1 + (np.abs((flips * d).mean(1)) >= observed - 1e-12).sum()) / (1 + SIGNFLIP_MC))


def two_level_bootstrap(per_class: list[np.ndarray], query_rows: list[np.ndarray], n_patients: int,
                        reps: int, seed: int) -> np.ndarray:
    """Bootstrap distribution of a macro mean of per-query values.

    per_class  : one (U, Q_c) array per pathology; U = replication units (model seeds or episodes)
    query_rows : one (Q_c,) array per pathology with each query's patient index in its split
    Units and patients are resampled jointly across pathologies; the gallery is held fixed.
    """
    rng = np.random.default_rng(seed)
    n_units = per_class[0].shape[0]
    draws = []
    for start in range(0, reps, 500):
        batch = min(500, reps - start)
        unit_weights = rng.multinomial(n_units, np.full(n_units, 1 / n_units), size=batch).astype(float)
        patient_weights = rng.multinomial(n_patients, np.full(n_patients, 1 / n_patients), size=batch).astype(float)
        class_means = []
        for values, rows in zip(per_class, query_rows):
            query_weights = patient_weights[:, rows]
            numerator = ((unit_weights @ values) * query_weights).sum(1)
            denominator = unit_weights.sum(1) * query_weights.sum(1)
            class_means.append(np.where(denominator > 0, numerator / np.maximum(denominator, 1e-12), np.nan))
        draws.append(np.nanmean(np.stack(class_means, 1), 1))
    return np.concatenate(draws)


def bootstrap_p(draws: np.ndarray) -> float:
    """Two-sided bootstrap p-value for 'the mean difference is zero'."""
    d = draws[np.isfinite(draws)]
    n = len(d)
    return float(min(1.0, 2 * min((1 + (d <= 0).sum()) / (n + 1), (1 + (d >= 0).sum()) / (n + 1))))


def stable_seed(*parts) -> int:
    """Deterministic integer seed from any labels (so every test is reproducible)."""
    return int(sha_of(parts)[:8], 16)


print("Cell 2 ready.", elapsed())


Pre-registration sha256: 520e215f55c4afeb8d75eb00b8c5b31b8212e10e17ca5e3f6d61056a8215f70c
Cell 2 ready. t+0.9 min


## Quantum core and similarity library
Everything here is data-free and self-tested before any GPU-heavy stage.

In [3]:
# =============================================================================
# Cell 3 — Quantum core: the H1 circuit as PennyLane QNodes, the model, and the equivalence gate
#   Every circuit evaluation in this notebook (training, SELECT scoring, encoding every split,
#   ablations) goes through a QNode on PennyLane's default.qubit statevector simulator.
#   A small hand-written torch simulator exists ONLY to cross-check PennyLane inside the gate
#   below; any other call raises. QSTATS counts every QNode call for the provenance record.
# =============================================================================
QSTATS = {"qnode_calls": 0, "qnode_samples": 0, "torch_reference_calls": 0,
          "audit_pairs": 0, "audit_max_gap": 0.0, "circuit_audit_pairs": 0, "circuit_audit_max_gap": 0.0}
_TORCH_REFERENCE = {"allowed": False}
MAX_VIEW_SLOTS = 8                               # 3 ancilla qubits index up to 8 mixture components
ANCILLA_A = list(range(N_QUBITS, N_QUBITS + 3))
ANCILLA_B = list(range(N_QUBITS + 3, N_QUBITS + 6))
SYSTEM_WIRES = list(range(N_QUBITS))


def feature_index(layer: int, qubit: int, axis: int, n_features: int = N_ANGLES) -> int:
    """Which of the 12 input angles feeds (layer, qubit, axis); axis 0 -> RY, axis 1 -> RZ."""
    return (2 * layer * N_QUBITS + qubit + axis * N_QUBITS) % n_features


def apply_h1_gates(angles_pi, theta, entangle: bool = True, adjoint: bool = False, control_wires=None) -> None:
    """Queue H1's gates on the system qubits.

    angles_pi     : (..., 12) input angles already multiplied by pi, or (..., 8, 12) when control_wires is set
    theta         : (VQC_LAYERS, N_QUBITS, 2) trainable angles (identical in every branch)
    control_wires : when given, branch k of that ancilla register applies the circuit to angles_pi[..., k, :].
                    Only the data rotations differ between branches, so they become multiplexed rotations
                    (qml.SelectPauliRot); theta rotations and the CNOT ring stay uncontrolled. The result is
                    exactly sum_k |k><k| (x) U(x_k).
    adjoint       : queue the inverse circuit (used by the overlap and Hilbert-Schmidt circuits)
    """
    steps = []
    for layer in range(VQC_LAYERS):
        for qubit in range(N_QUBITS):
            steps.append(("data", "Y", feature_index(layer, qubit, 0), qubit))
            steps.append(("data", "Z", feature_index(layer, qubit, 1), qubit))
            steps.append(("theta", "Y", (layer, qubit, 0), qubit))
            steps.append(("theta", "Z", (layer, qubit, 1), qubit))
        if entangle:
            steps.extend(("cnot", None, None, qubit) for qubit in range(N_QUBITS))
    sign = -1.0 if adjoint else 1.0
    for kind, axis, index, qubit in (reversed(steps) if adjoint else steps):
        if kind == "cnot":
            qml.CNOT(wires=[qubit, (qubit + 1) % N_QUBITS])
        elif kind == "theta":
            rotation = qml.RY if axis == "Y" else qml.RZ
            rotation(sign * theta[index], wires=qubit)
        elif control_wires is None:
            rotation = qml.RY if axis == "Y" else qml.RZ
            rotation(sign * angles_pi[..., index], wires=qubit)
        else:
            qml.SelectPauliRot(sign * angles_pi[..., index], control_wires=control_wires,
                               target_wire=qubit, rot_axis=axis)


_QNODES = {}


def h1_qnode(kind: str, entangle: bool = True):
    """Cached QNodes (module-level, so models can be deep-copied safely).

    state   : H1 state                                   -> qml.state()
    expval  : H1 readout                                 -> [<Z_w>]
    overlap : U(a) then U(b)^dagger                      -> P(|0..0>) = |<b|a>|^2
    mixing  : purification of sum_k w_k |psi_k><psi_k|   -> qml.density_matrix of the 6 system qubits
    hs      : U_rho (ancillas A) then U_sigma^dagger (ancillas B) -> P(system |0..0>) = Tr(rho sigma)
    """
    key = (kind, bool(entangle))
    if key in _QNODES:
        return _QNODES[key]
    if kind == "state":
        def circuit(angles_pi, theta):
            apply_h1_gates(angles_pi, theta, entangle)
            return qml.state()
        n_wires = N_QUBITS
    elif kind == "expval":
        def circuit(angles_pi, theta):
            apply_h1_gates(angles_pi, theta, entangle)
            return [qml.expval(qml.PauliZ(wire)) for wire in SYSTEM_WIRES]
        n_wires = N_QUBITS
    elif kind == "overlap":
        def circuit(angles_a, angles_b, theta):
            apply_h1_gates(angles_a, theta, entangle)
            apply_h1_gates(angles_b, theta, entangle, adjoint=True)
            return qml.probs(wires=SYSTEM_WIRES)
        n_wires = N_QUBITS
    elif kind == "mixing":
        def circuit(amplitudes, view_angles, theta):
            qml.StatePrep(amplitudes, wires=ANCILLA_A)
            apply_h1_gates(view_angles, theta, entangle, control_wires=ANCILLA_A)
            return qml.density_matrix(wires=SYSTEM_WIRES)
        n_wires = N_QUBITS + 3
    elif kind == "hs":
        def circuit(amplitudes_ab, view_angles_a, view_angles_b, theta):
            qml.StatePrep(amplitudes_ab, wires=ANCILLA_A + ANCILLA_B)
            apply_h1_gates(view_angles_a, theta, entangle, control_wires=ANCILLA_A)
            apply_h1_gates(view_angles_b, theta, entangle, adjoint=True, control_wires=ANCILLA_B)
            return qml.probs(wires=SYSTEM_WIRES)
        n_wires = N_QUBITS + 6
    else:
        raise ValueError(kind)
    _QNODES[key] = qml.QNode(circuit, qml.device(PL_DEVICE_NAME, wires=n_wires), interface="torch", diff_method="backprop")
    return _QNODES[key]


QNODE_DEVICE = torch.device("cpu")               # replaced by choose_qnode_device() below


def run_qnode(kind: str, entangle: bool, *args):
    """Execute a QNode on QNODE_DEVICE and count it. Tensor moves are differentiable."""
    enforce_runtime_budget(f"QNode {kind}")
    batch_size = len(args[0])
    if batch_size == 0:
        raise ValueError(f"empty batch sent to the {kind} QNode")
    moved = [argument.to(QNODE_DEVICE) for argument in args]
    result = h1_qnode(kind, entangle)(*moved)
    QSTATS["qnode_calls"] += 1
    QSTATS["qnode_samples"] += batch_size
    return result


def pad_views(view_angles: torch.Tensor, view_weights: torch.Tensor):
    """(P, V, 12) angles in [-1, 1] and (P, V) weights -> angles*pi padded to 8 slots and sqrt-weight
    amplitudes for the 3 ancilla qubits. Differentiable in the angles."""
    n, n_views, _ = view_angles.shape
    padding = MAX_VIEW_SLOTS - n_views
    angles = torch.cat([view_angles.double() * math.pi, view_angles.new_zeros(n, padding, N_ANGLES).double()], 1)
    weights = view_weights.double() / view_weights.double().sum(-1, keepdim=True)
    amplitudes = torch.cat([weights.sqrt(), weights.new_zeros(n, padding)], 1).to(torch.complex128)
    return angles, amplitudes.detach()


class H1Circuit(nn.Module):
    """One pathology's H1 circuit: theta (VQC_LAYERS, N_QUBITS, 2), optionally frozen."""

    def __init__(self, entangle: bool = True, train_theta: bool = True):
        super().__init__()
        self.entangle = bool(entangle)
        initial_theta = torch.randn(VQC_LAYERS, N_QUBITS, 2) * 0.05
        if train_theta:
            self.theta = nn.Parameter(initial_theta)
        else:
            self.register_buffer("theta", initial_theta)

    def states(self, angles: torch.Tensor) -> torch.Tensor:
        """(B, 12) angles in [-1, 1] -> (B, 64) complex64 statevectors, on the angles' device."""
        state = run_qnode("state", self.entangle, angles * math.pi, self.theta)
        return state.reshape(len(angles), STATE_DIM).to(device=angles.device, dtype=torch.complex64)

    def mixed_densities(self, view_angles: torch.Tensor, view_weights: torch.Tensor) -> torch.Tensor:
        """(B, V, 12), (B, V) -> (B, 64, 64) rho = sum_k w_k |psi_k><psi_k|, prepared by the purification
        circuit and read out with qml.density_matrix (3 ancillas index the views)."""
        angles, amplitudes = pad_views(view_angles, view_weights)
        rho = run_qnode("mixing", self.entangle, amplitudes, angles, self.theta.double())
        return rho.reshape(len(view_angles), STATE_DIM, STATE_DIM).to(device=view_angles.device, dtype=torch.complex64)

    def z_expectations(self, angles: torch.Tensor) -> torch.Tensor:
        """(B, 12) -> (B, N_QUBITS) <Z_w> readout."""
        values = run_qnode("expval", self.entangle, angles * math.pi, self.theta)
        return torch.stack([v.reshape(len(angles)) for v in values], -1).to(device=angles.device, dtype=angles.dtype)


class H1Retriever(nn.Module):
    """Five pathology-specific H1 circuits (Q1's QuantumHead) plus the episodic logit scale.

    angle map  : Linear(128 -> 12) + tanh, one per pathology
    circuit    : H1Circuit, one per pathology
    readout    : Linear(6 -> 1) on <Z>, one per pathology (trained only by T0_readout)
    scale      : temperature and per-class bias turning a similarity margin into a logit (training only)
    """

    def __init__(self, variant: Variant = MAIN_VARIANT):
        super().__init__()
        self.variant_name = variant.name
        self.angle_maps = nn.ModuleList([nn.Linear(FEATURE_DIM, N_ANGLES) for _ in CLASSES])
        self.circuits = nn.ModuleList([H1Circuit(variant.entangle, variant.train_theta) for _ in CLASSES])
        self.readouts = nn.ModuleList([nn.Linear(N_QUBITS, 1) for _ in CLASSES])
        self.log_temperature = nn.Parameter(torch.zeros(()))
        self.class_bias = nn.Parameter(torch.zeros(N_CLASSES))

    def angles(self, features: torch.Tensor) -> torch.Tensor:
        """(N, 5, 128) -> (N, 5, 12) in [-1, 1]."""
        return torch.stack([torch.tanh(self.angle_maps[c](features[:, c])) for c in range(N_CLASSES)], 1)

    def view_angles(self, views: torch.Tensor) -> torch.Tensor:
        """(N, 5, V, 128) -> (N, 5, V, 12); each pathology's views go through that pathology's angle map."""
        return torch.stack([torch.tanh(self.angle_maps[c](views[:, c])) for c in range(N_CLASSES)], 1)

    def pure_states(self, angles: torch.Tensor) -> torch.Tensor:
        """(N, 5, 12) -> (N, 5, 64)."""
        return torch.stack([self.circuits[c].states(angles[:, c]) for c in range(N_CLASSES)], 1)

    def mixed_densities(self, view_angles: torch.Tensor, view_weights: torch.Tensor) -> torch.Tensor:
        """(N, 5, V, 12), (N, 5, V) -> (N, 5, 64, 64) density matrices from the mixing QNode."""
        return torch.stack([self.circuits[c].mixed_densities(view_angles[:, c], view_weights[:, c]) for c in range(N_CLASSES)], 1)

    def readout_logits(self, angles: torch.Tensor) -> torch.Tensor:
        """(N, 5, 12) -> (N, 5) readout logit through qml.expval."""
        return torch.cat([self.readouts[c](self.circuits[c].z_expectations(angles[:, c])) for c in range(N_CLASSES)], 1)

    def thetas(self) -> torch.Tensor:
        return torch.stack([circuit.theta.detach().cpu() for circuit in self.circuits])


# ---- torch reference simulator (equivalence gate ONLY) ----
def _ry(angle):
    c, s = torch.cos(angle / 2), torch.sin(angle / 2)
    return torch.stack([c, -s, s, c], -1).reshape(angle.shape + (2, 2)).to(torch.complex64)


def _rz(angle):
    zero = torch.zeros_like(angle)
    minus = torch.complex(torch.cos(-angle / 2), torch.sin(-angle / 2))
    plus = torch.complex(torch.cos(angle / 2), torch.sin(angle / 2))
    return torch.stack([minus, torch.complex(zero, zero), torch.complex(zero, zero), plus], -1).reshape(angle.shape + (2, 2))


def _apply_one_qubit(state, unitary, qubit):
    batch = len(state)
    tensor = state.view((batch,) + (2,) * N_QUBITS).movedim(qubit + 1, -1)
    return torch.einsum("b...j,bij->b...i", tensor, unitary).movedim(-1, qubit + 1).reshape(batch, -1)


def _apply_cnot(state, control, target):
    permutation = torch.tensor([i ^ (1 << (N_QUBITS - 1 - target)) if (i >> (N_QUBITS - 1 - control)) & 1 else i
                                for i in range(STATE_DIM)], device=state.device)
    return state[:, permutation]


def torch_reference_states(angles: torch.Tensor, theta: torch.Tensor, entangle: bool = True) -> torch.Tensor:
    """H1 in plain torch; raises unless called from inside the equivalence gate."""
    if not _TORCH_REFERENCE["allowed"]:
        raise RuntimeError("torch simulator called outside the equivalence gate -- PennyLane lock violated")
    QSTATS["torch_reference_calls"] += 1
    batch, angles_pi = len(angles), angles * math.pi
    state = torch.zeros(batch, STATE_DIM, dtype=torch.complex64)
    state[:, 0] = 1
    for layer in range(VQC_LAYERS):
        for qubit in range(N_QUBITS):
            state = _apply_one_qubit(state, _ry(angles_pi[:, feature_index(layer, qubit, 0)]), qubit)
            state = _apply_one_qubit(state, _rz(angles_pi[:, feature_index(layer, qubit, 1)]), qubit)
            state = _apply_one_qubit(state, _ry(theta[layer, qubit, 0].expand(batch)), qubit)
            state = _apply_one_qubit(state, _rz(theta[layer, qubit, 1].expand(batch)), qubit)
        if entangle:
            for qubit in range(N_QUBITS):
                state = _apply_cnot(state, qubit, (qubit + 1) % N_QUBITS)
    return state


def z_from_states(states: torch.Tensor) -> torch.Tensor:
    probabilities = states.abs().square().view((len(states),) + (2,) * N_QUBITS)
    values = []
    for qubit in range(N_QUBITS):
        marginal = probabilities.sum(dim=tuple(axis + 1 for axis in range(N_QUBITS) if axis != qubit))
        values.append(marginal[:, 0] - marginal[:, 1])
    return torch.stack(values, 1)


# ---- choose where the QNodes run ----
def choose_qnode_device() -> torch.device:
    """Run the state QNode on CPU and (if allowed) CUDA; keep CUDA only if it agrees and is faster."""
    generator = torch.Generator().manual_seed(SEED)
    theta = torch.randn(VQC_LAYERS, N_QUBITS, 2, generator=generator) * 0.1
    angles = torch.rand(1920, N_ANGLES, generator=generator) * 2 - 1
    timings, outputs = {}, {}
    candidates = ["cpu"] if QNODE_TORCH_DEVICE == "cpu" else ["cpu", "cuda"]
    for name in candidates:
        try:
            device = torch.device(name)
            angle_input = angles.to(device).requires_grad_(True)
            theta_input = theta.to(device).requires_grad_(True)
            start = time.time()
            state = h1_qnode("state")(angle_input * math.pi, theta_input)
            state.abs().square()[:, 0].sum().backward()
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings[name] = time.time() - start
            outputs[name] = state.detach().cpu().to(torch.complex64)
        except Exception as error:                                   # noqa: BLE001 — any failure means "use CPU"
            print(f"  [qnode device] {name} unavailable for default.qubit: {type(error).__name__}: {error}")
    if "cuda" in outputs and (outputs["cuda"] - outputs["cpu"]).abs().max() < 1e-5 and timings["cuda"] < timings["cpu"]:
        chosen = torch.device("cuda")
    else:
        chosen = torch.device("cpu")
    print(f"  [qnode device] forward+backward of 1,920 states: "
          + ", ".join(f"{k} {v:.2f}s" for k, v in timings.items()) + f" -> QNodes run on {chosen}")
    return chosen


# ---- the mandatory gate ----
def pennylane_equivalence_gate() -> None:
    """PennyLane states and readout == torch reference; circuit diagram; gradients reach theta and the angle map."""
    generator = torch.Generator().manual_seed(SEED)
    seed_all(SEED)
    probe = H1Retriever(MAIN_VARIANT)
    features = F.normalize(torch.randn(8, N_CLASSES, FEATURE_DIM, generator=generator), dim=-1)
    with torch.no_grad():
        angles = probe.angles(features)
        pennylane_states = probe.pure_states(angles)
        pennylane_z = torch.stack([probe.circuits[c].z_expectations(angles[:, c]) for c in range(N_CLASSES)], 1)
        _TORCH_REFERENCE["allowed"] = True
        try:
            reference = torch.stack([torch_reference_states(angles[:, c], probe.circuits[c].theta)
                                     for c in range(N_CLASSES)], 1)
        finally:
            _TORCH_REFERENCE["allowed"] = False
    fidelity = (pennylane_states.conj() * reference).sum(-1).abs().square()
    z_gap = float((z_from_states(reference.reshape(-1, STATE_DIM)).reshape(pennylane_z.shape) - pennylane_z).abs().max())
    print(f"  [equivalence] PennyLane {PL_DEVICE_NAME} vs torch reference: min state fidelity "
          f"{float(fidelity.min()):.7f}, max |d<Z>| {z_gap:.2e}")
    if float(fidelity.min()) < 1 - 1e-5 or z_gap > 1e-4:
        raise RuntimeError("PennyLane and the torch reference disagree -- do not trust any quantum number.")
    diagram = qml.draw(h1_qnode("state"), decimals=2)(angles[0, 0].to(QNODE_DEVICE) * math.pi,
                                                       probe.circuits[0].theta.detach().to(QNODE_DEVICE))
    print("  [circuit] one pathology's H1 circuit as executed by PennyLane:\n" + diagram)
    probe.zero_grad()
    loss = probe.readout_logits(probe.angles(features)).sum() + probe.pure_states(probe.angles(features))[..., 0].abs().sum()
    loss.backward()
    for name, gradient in (("theta", probe.circuits[0].theta.grad), ("angle map", probe.angle_maps[0].weight.grad)):
        if gradient is None or not torch.isfinite(gradient).all() or float(gradient.abs().sum()) == 0:
            raise RuntimeError(f"No gradient reached the {name} through the PennyLane QNodes.")
    per_class = N_ANGLES * (FEATURE_DIM + 1) + VQC_LAYERS * N_QUBITS * 2 + N_QUBITS + 1
    print(f"  [gradient] theta and angle map receive finite, non-zero gradients through the QNodes")
    print(f"  parameters per pathology: angle map {N_ANGLES * (FEATURE_DIM + 1)}, theta {VQC_LAYERS * N_QUBITS * 2}, "
          f"readout {N_QUBITS + 1} -> {per_class} ({N_CLASSES * per_class:,} for five pathologies)")


QNODE_DEVICE = choose_qnode_device()
pennylane_equivalence_gate()
if QSTATS["torch_reference_calls"] != N_CLASSES:
    raise RuntimeError(f"torch reference ran {QSTATS['torch_reference_calls']} times; expected {N_CLASSES} (the gate only)")
print("Cell 3 ready.", elapsed())


  [qnode device] forward+backward of 1,920 states: cpu 0.47s, cuda 2.55s -> QNodes run on cpu
  [equivalence] PennyLane default.qubit vs torch reference: min state fidelity 0.9999975, max |d<Z>| 1.07e-06
  [circuit] one pathology's H1 circuit as executed by PennyLane:
0: ──RY(-0.57)──RZ(0.03)───RY(-0.00)──RZ(0.08)──╭●─────────────╭X──RY(-0.57)──RZ(0.03)─ ···
1: ──RY(-0.33)──RZ(0.12)───RY(-0.02)──RZ(-0.03)─╰X─╭●──────────│───RY(-0.33)──RZ(0.12)─ ···
2: ──RY(-0.07)──RZ(0.07)───RY(0.03)───RZ(-0.02)────╰X─╭●───────│───RY(-0.07)──RZ(0.07)─ ···
3: ──RY(-0.01)──RZ(0.04)───RY(0.07)───RZ(0.01)────────╰X─╭●────│───RY(-0.01)──RZ(0.04)─ ···
4: ──RY(-0.11)──RZ(-0.15)──RY(-0.03)──RZ(0.00)───────────╰X─╭●─│───RY(-0.11)──RZ(-0.15) ···
5: ──RY(0.45)───RZ(0.02)───RY(0.06)───RZ(0.04)──────────────╰X─╰●──RY(0.45)───RZ(0.02)─ ···

0: ··· ──RY(0.01)───RZ(-0.05)─╭●─────────────╭X──RY(-0.57)──RZ(0.03)───RY(-0.01)──RZ(-0.04)─╭● ···
1: ··· ──RY(0.06)───RZ(-0.04)─╰X─╭●──────────│───RY(-0.33)──RZ(0.12)───RY(0.01)

In [4]:
# =============================================================================
# Cell 4 — Similarity library (PennyLane), audits and self-tests
#   Representations, all produced by the QNodes of Cell 3 (one per pathology):
#     pure     (N, 64)       statevector of the routed feature                      qml.state()
#     density  (N, 64, 64)   mixed state of the crop views                          qml.density_matrix()
#     angles   (N, 12)       circuit input angles (classical rules)
#   Every QUANTUM similarity — in the training loss and in every score — is computed with PennyLane's
#   quantum-information functions:
#     pure fidelity      qml.math.fidelity_statevector
#     Uhlmann fidelity   qml.math.fidelity                (Bures = sqrt(2 - 2 sqrt(F)) of the same call)
#     trace distance     qml.math.trace_distance
#     Hilbert-Schmidt    qml.math.einsum Tr(rho sigma) / sqrt(qml.math.purity x qml.math.purity)
#   Classical rules (cosine, Euclidean, Mahalanobis) are plain torch by design.
#   An independent dense torch implementation re-checks sampled pairs of every scoring call, and simulator
#   circuits (compute-uncompute overlap, 12-qubit Hilbert-Schmidt) measure sampled pairs of every model.
# =============================================================================
REPRESENTATION_KEY = {"pure": "pure", "mixed": "density", "angles": "angles", "readout": "readout"}
NOT_ROW_DATA = ("precision",)                     # dictionary entries that are not indexed by patient
EPSILON = 1e-12
_ALL_WIRES = list(range(N_QUBITS))


def shrunk_precision(vectors: torch.Tensor, shrink: float = MAHALANOBIS_SHRINK) -> torch.Tensor:
    """Inverse of a covariance shrunk toward (mean variance) x I; vectors (N, D) -> (D, D)."""
    x = vectors.double()
    centred = x - x.mean(0, keepdim=True)
    covariance = centred.T @ centred / max(1, len(x) - 1)
    identity = torch.eye(x.shape[1], dtype=x.dtype, device=x.device)
    target = covariance.diagonal().mean().clamp_min(1e-6) * identity
    return torch.linalg.inv((1 - shrink) * covariance + shrink * target).float()


def class_precisions(angles: torch.Tensor) -> torch.Tensor:
    """(N, 5, D) -> (5, D, D): one label-free precision per pathology."""
    return torch.stack([shrunk_precision(angles[:, c]) for c in range(angles.shape[1])])


def density_purity(density: torch.Tensor) -> torch.Tensor:
    """Tr(rho^2) with qml.math.purity; 1 for a pure state."""
    return qml.math.real(qml.math.purity(density, _ALL_WIRES, check_state=False, c_dtype="complex128")).float()


def paired_similarity(metric: str, representation: str, a: torch.Tensor, b: torch.Tensor,
                      precision: torch.Tensor | None = None, purity_a=None, purity_b=None) -> torch.Tensor:
    """(P,) similarity of the pairs (a[i], b[i]); larger = more similar. Differentiable."""
    if representation == "angles":
        if metric == "cosine":
            return (F.normalize(a.float(), dim=-1) * F.normalize(b.float(), dim=-1)).sum(-1)
        difference = a.float() - b.float()
        if metric == "euclidean":
            return -difference.square().sum(-1).add(EPSILON).sqrt()
        if metric == "mahalanobis":
            return -torch.einsum("pd,de,pe->p", difference, precision.to(difference), difference)
    if representation == "pure" and metric == "fidelity":
        return qml.math.real(qml.math.fidelity_statevector(a, b, check_state=False, c_dtype="complex128")).float()
    if representation == "mixed":
        if metric in ("uhlmann", "bures"):
            fidelity = qml.math.real(qml.math.fidelity(a, b, check_state=False, c_dtype="complex128")).clamp(0, 1)
            if metric == "uhlmann":
                return fidelity.float()
            return (-torch.sqrt((2 - 2 * torch.sqrt(fidelity + EPSILON)).clamp_min(0) + EPSILON)).float()
        if metric == "trace_distance":
            return (-qml.math.real(qml.math.trace_distance(a, b, check_state=False, c_dtype="complex128"))).float()
        if metric == "hilbert_schmidt":
            overlap = qml.math.real(qml.math.einsum("bij,bji->b", a.to(torch.complex128), b.to(torch.complex128)))
            purity_a = density_purity(a) if purity_a is None else purity_a
            purity_b = density_purity(b) if purity_b is None else purity_b
            return (overlap.float() / torch.sqrt(purity_a * purity_b).clamp_min(1e-10)).clamp(0, 1)
    raise ValueError(f"no similarity for metric={metric!r} on representation={representation!r}")


def similarity_matrix(metric: str, representation: str, query: torch.Tensor, reference: torch.Tensor,
                      precision: torch.Tensor | None = None) -> torch.Tensor:
    """(Q, R) similarity for every query/reference pair of ONE pathology, PAIR_BATCH pairs per PennyLane call."""
    if representation == "angles":
        if metric == "cosine":
            return F.normalize(query.float(), dim=-1) @ F.normalize(reference.float(), dim=-1).T
        if metric == "euclidean":
            return -torch.cdist(query.double(), reference.double()).float()
    purities = {}
    if representation == "mixed" and metric == "hilbert_schmidt":        # once per patient, not once per pair
        purities = {"query": density_purity(query), "reference": density_purity(reference)}
    n_query, n_reference = len(query), len(reference)
    rows_per_call = max(1, PAIR_BATCH // n_reference)
    blocks = []
    for start in range(0, n_query, rows_per_call):
        enforce_runtime_budget(f"similarity {representation}/{metric} {start}/{n_query}")
        block = query[start:start + rows_per_call]
        n_block = len(block)
        a = block[:, None].expand(n_block, n_reference, *block.shape[1:]).reshape(n_block * n_reference, *block.shape[1:])
        b = reference[None].expand(n_block, n_reference, *reference.shape[1:]).reshape(n_block * n_reference, *reference.shape[1:])
        extra = {}
        if purities:
            extra = {"purity_a": purities["query"][start:start + n_block, None].expand(-1, n_reference).reshape(-1),
                     "purity_b": purities["reference"][None].expand(n_block, -1).reshape(-1)}
        blocks.append(paired_similarity(metric, representation, a, b, precision, **extra).reshape(n_block, n_reference))
    return torch.cat(blocks)


# ---- independent dense torch reference (audit only; never produces a reported number) ----
def _matrix_sqrt(matrix: torch.Tensor) -> torch.Tensor:
    eigenvalues, eigenvectors = torch.linalg.eigh(matrix.to(torch.complex128))
    return (eigenvectors * eigenvalues.clamp_min(0).sqrt().to(eigenvectors.dtype)[..., None, :]) @ eigenvectors.conj().transpose(-1, -2)


def dense_reference_similarity(metric: str, representation: str, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if representation == "pure":
        return (a.conj().to(torch.complex128) * b.to(torch.complex128)).sum(-1).abs().square().float()
    a, b = a.to(torch.complex128), b.to(torch.complex128)
    if metric in ("uhlmann", "bures"):
        root = _matrix_sqrt(a)
        middle = root @ b @ root
        fidelity = torch.linalg.eigvalsh((middle + middle.conj().transpose(-1, -2)) / 2).clamp_min(0).sqrt().sum(-1).square().clamp(0, 1)
        return fidelity.float() if metric == "uhlmann" else (-torch.sqrt((2 - 2 * fidelity.sqrt()).clamp_min(0))).float()
    if metric == "trace_distance":
        return (-0.5 * torch.linalg.eigvalsh(a - b).abs().sum(-1)).float()
    if metric == "hilbert_schmidt":
        trace = lambda x, y: torch.einsum("bij,bji->b", x, y).real
        return (trace(a, b) / torch.sqrt(trace(a, a) * trace(b, b))).clamp(0, 1).float()
    raise ValueError(metric)


def audit_scores(metric: str, representation: str, query: torch.Tensor, reference: torch.Tensor,
                 scores: torch.Tensor, seed: int) -> float:
    """Recompute sampled pairs with the dense torch reference; abort if PennyLane's value disagrees."""
    if representation == "angles":
        return 0.0
    rng = np.random.default_rng(seed)
    n_pairs = min(AUDIT_PAIRS_PER_CALL, len(query) * len(reference))
    q_index, r_index = rng.integers(0, len(query), n_pairs), rng.integers(0, len(reference), n_pairs)
    expected = dense_reference_similarity(metric, representation, query[q_index].cpu(), reference[r_index].cpu())
    gap = float((scores[q_index, r_index].cpu() - expected).abs().max())
    QSTATS["audit_pairs"] += n_pairs
    QSTATS["audit_max_gap"] = max(QSTATS["audit_max_gap"], gap)
    if gap > AUDIT_TOLERANCE:
        raise RuntimeError(f"{representation}/{metric}: qml.math and the dense reference differ by {gap:.2e}")
    return gap


def score_matrix(rule_name: str, query: dict, reference: dict, class_index: int, audit_seed: int = 0) -> np.ndarray:
    """(Q, R) scores of one rule for one pathology, computed on DEVICE, returned as numpy float32."""
    rule = SCORING_RULES[rule_name]
    key = REPRESENTATION_KEY[rule.representation]
    q_all, r_all = query[key][:, class_index], reference[key][:, class_index]
    precision = reference["precision"][class_index].to(DEVICE) if rule.metric == "mahalanobis" else None
    with torch.no_grad():
        scores = similarity_matrix(rule.metric, rule.representation, q_all.to(DEVICE), r_all.to(DEVICE), precision).float().cpu()
    audit_scores(rule.metric, rule.representation, q_all, r_all, scores, audit_seed)
    return scores.numpy()


# ---- circuit-level measurements on sampled pairs ----
def circuit_overlap(angles_a: torch.Tensor, angles_b: torch.Tensor, theta: torch.Tensor, entangle: bool) -> torch.Tensor:
    """|<psi(a)|psi(b)>|^2 measured as P(|0..0>) of the compute-uncompute circuit, pair by pair."""
    probabilities = run_qnode("overlap", entangle, angles_a.double().cpu() * math.pi, angles_b.double().cpu() * math.pi,
                              theta.double().cpu())
    return probabilities.reshape(len(angles_a), -1)[:, 0].float().cpu()


def circuit_hilbert_schmidt_overlap(views_a, weights_a, views_b, weights_b, theta, entangle) -> torch.Tensor:
    """Tr(rho sigma) measured as P(system |0..0>) of the 12-qubit circuit, pair by pair."""
    angles_a, amplitudes_a = pad_views(views_a.detach().cpu(), weights_a.detach().cpu())
    angles_b, amplitudes_b = pad_views(views_b.detach().cpu(), weights_b.detach().cpu())
    joint = torch.einsum("pa,pb->pab", amplitudes_a, amplitudes_b).reshape(len(angles_a), -1)
    probabilities = run_qnode("hs", entangle, joint, angles_a, angles_b, theta.double().cpu())
    return probabilities.reshape(len(angles_a), -1)[:, 0].float().cpu()


def circuit_audit(encoded: dict, thetas: torch.Tensor, entangle: bool, seed: int) -> float:
    """Measure sampled pure-fidelity and Tr(rho sigma) pairs with circuits and compare with qml.math."""
    rng = np.random.default_rng(seed)
    n, worst = len(encoded["angles"]), 0.0
    for class_index in range(N_CLASSES):
        i = torch.as_tensor(rng.integers(0, n, CIRCUIT_AUDIT_PAIRS))
        j = torch.as_tensor(rng.integers(0, n, CIRCUIT_AUDIT_PAIRS))
        theta = thetas[class_index]
        if "pure" in encoded:
            measured = circuit_overlap(encoded["angles"][i, class_index], encoded["angles"][j, class_index], theta, entangle)
            expected = paired_similarity("fidelity", "pure", encoded["pure"][i, class_index], encoded["pure"][j, class_index]).cpu()
            worst = max(worst, float((measured - expected).abs().max()))
            QSTATS["circuit_audit_pairs"] += CIRCUIT_AUDIT_PAIRS
        if "density" in encoded:
            measured = circuit_hilbert_schmidt_overlap(encoded["view_angles"][i, class_index], encoded["view_weights"][i, class_index],
                                                   encoded["view_angles"][j, class_index], encoded["view_weights"][j, class_index],
                                                   theta, entangle)
            rho_i, rho_j = encoded["density"][i, class_index], encoded["density"][j, class_index]
            expected = qml.math.real(qml.math.einsum("bij,bji->b", rho_i.to(torch.complex128), rho_j.to(torch.complex128))).float()
            worst = max(worst, float((measured - expected).abs().max()))
            QSTATS["circuit_audit_pairs"] += CIRCUIT_AUDIT_PAIRS
    QSTATS["circuit_audit_max_gap"] = max(QSTATS["circuit_audit_max_gap"], worst)
    if worst > AUDIT_TOLERANCE:
        raise RuntimeError(f"simulator circuits disagree with qml.math by {worst:.2e}")
    return worst


def mean_kendall_tau(scores_a: np.ndarray, scores_b: np.ndarray) -> float:
    """Mean over queries of Kendall's tau between two rankings of the same gallery (1 = identical order)."""
    values = [kendalltau(a, b).statistic for a, b in zip(scores_a, scores_b)]
    values = [v for v in values if np.isfinite(v)]
    return float(np.mean(values)) if values else np.nan


# ---- data-free self-tests ----
def run_similarity_self_tests() -> None:
    generator = torch.Generator().manual_seed(SEED)
    seed_all(SEED + 1)
    circuit = H1Circuit()
    views = torch.rand(6, N_VIEWS_DEFAULT, N_ANGLES, generator=generator) * 2 - 1
    weights = torch.softmax(torch.randn(6, N_VIEWS_DEFAULT, generator=generator), 1)
    with torch.no_grad():
        view_states = circuit.states(views.reshape(-1, N_ANGLES)).reshape(6, N_VIEWS_DEFAULT, STATE_DIM).to(torch.complex128)
        density = circuit.mixed_densities(views, weights).to(torch.complex128)
    weighted = torch.einsum("nk,nki,nkj->nij", weights.to(torch.complex128), view_states, view_states.conj())
    assert (density - weighted).abs().max() < 1e-4, "mixing circuit = weighted sum of the view states"
    assert torch.allclose(density.diagonal(dim1=-2, dim2=-1).sum(-1).real, torch.ones(6, dtype=torch.float64), atol=1e-5), "trace one"
    pure = view_states[:, 0]
    ii, jj = torch.tensor([0, 1, 2, 4, 3]), torch.tensor([1, 3, 5, 0, 3])
    for rule in SCORING_RULES.values():                                  # 1. qml.math = dense definitions
        if rule.representation in ("pure", "mixed"):
            data = pure if rule.representation == "pure" else density
            pennylane = paired_similarity(rule.metric, rule.representation, data[ii], data[jj])
            matrix = similarity_matrix(rule.metric, rule.representation, data, data)[ii, jj]
            reference = dense_reference_similarity(rule.metric, rule.representation, data[ii], data[jj])
            assert (pennylane - reference).abs().max() < 2e-4, f"{rule.name}: qml.math vs dense definition"
            assert (matrix - pennylane).abs().max() < 2e-4, f"{rule.name}: matrix vs paired form"
    uhlmann = similarity_matrix("uhlmann", "mixed", density, density)    # 2. Bures ranks like Uhlmann
    bures = similarity_matrix("bures", "mixed", density, density)
    order = lambda m: torch.argsort(m, dim=1, descending=True, stable=True)
    assert torch.equal(order(uhlmann), order(bures)), "Bures ranking"
    for arm in TRAINING_ARMS.values():                                   # 3. finite gradients through qml.math and the QNodes
        if arm.metric is None:
            continue
        angles = (torch.rand(5, N_ANGLES, generator=generator) * 2 - 1).requires_grad_(True)
        if arm.representation == "angles":
            data, precision = angles, shrunk_precision(angles.detach())
        elif arm.representation == "pure":
            data, precision = circuit.states(angles), None
        else:
            data, precision = circuit.mixed_densities(angles[:, None].expand(-1, 2, -1), torch.full((5, 2), 0.5)), None
        paired_similarity(arm.metric, arm.representation, data[[0, 1, 2]], data[[3, 4, 0]], precision).sum().backward()
        assert torch.isfinite(angles.grad).all(), f"{arm.name}: non-finite gradient"
    rng = np.random.default_rng(SEED)                                    # 4. retrieval metrics vs scikit-learn
    sim = rng.normal(size=(7, 40))
    relevance = (rng.random(40) < 0.3).astype(np.int8)
    relevance[:2] = 1
    per_query = per_query_retrieval(sim, relevance)
    for q in range(7):
        assert abs(per_query["map"][q] - average_precision_score(relevance, sim[q])) < 1e-6, "AP"
        assert abs(per_query["ndcg_at_10"][q] - ndcg_score(relevance[None], sim[q][None], k=10)) < 1e-6, "nDCG@10"
    assert np.allclose(holm([0.01, 0.04, 0.03, 0.005]), [0.03, 0.06, 0.06, 0.02]), "Holm"
    arrays = [rng.normal(size=(5, 9)), rng.normal(size=(5, 4))]
    draws = two_level_bootstrap(arrays, [np.arange(9), np.arange(3, 7)], 12, 2000, SEED)
    assert abs(np.nanmean(draws) - np.mean([a.mean() for a in arrays])) < 0.05, "bootstrap centring"
    assert signflip_p(np.ones(8), SEED) == 2 / 256, "sign-flip"
    theta = circuit.theta.detach()                                       # 5. measurement circuits
    overlap = circuit_overlap(views[:3, 0], views[3:, 0], theta, True)
    assert (overlap - paired_similarity("fidelity", "pure", pure[:3], pure[3:])).abs().max() < 1e-4, "overlap circuit"
    hs = circuit_hilbert_schmidt_overlap(views[:3], weights[:3], views[3:], weights[3:], theta, True)
    assert (hs - torch.einsum("pij,pji->p", density[:3], density[3:]).real.float()).abs().max() < 1e-4, "12-qubit HS circuit"
    print("  Self-tests passed: mixing circuit = weighted views, qml.math = dense definitions for every quantum rule, "
          "Bures ranks like Uhlmann, finite gradients through QNodes and qml.math, overlap and 12-qubit circuits, "
          "AP/nDCG vs sklearn, Holm, bootstrap, sign-flip.")


N_VIEWS_DEFAULT = len(TARGET_CROP_BOXES)
run_similarity_self_tests()
for counter in QSTATS:
    QSTATS[counter] = 0.0 if counter.endswith("gap") else 0   # the provenance ledger counts real work only
print("Cell 4 ready.", elapsed())


[AGPQuant t+1.0 min] HEARTBEAT | stage: QNode mixing | log: /kaggle/working/agpquant_progress.log
  Self-tests passed: mixing circuit = weighted views, qml.math = dense definitions for every quantum rule, Bures ranks like Uhlmann, finite gradients through QNodes and qml.math, overlap and 12-qubit circuits, AP/nDCG vs sklearn, Holm, bootstrap, sign-flip.
Cell 4 ready. t+1.1 min


## Data, Stage A encoders, routing and prior

In [5]:
# =============================================================================
# Cell 5 — Patient-disjoint CheXpert cohorts
#   preadapt, support : label-balanced (training and K-shot draws need positives of every pathology)
#   gallery           : natural prevalence with a floor of known positives AND negatives per pathology
#   select, dev, final: natural prevalence with a smaller floor (so no class has 2-3 known negatives)
#   expert_valid      : CheXpert radiologist-labelled validation patients (FULL only)
#   Each pathology's retrieval chance level is its gallery prevalence, reported with the results.
# =============================================================================
SUPPORT_MIN_PER_LABEL = max(SHOTS) + 1        # > max-shot positives AND negatives, so K-shot draws can vary


def patient_from_path(path) -> str:
    match = re.search(r"(patient\d+)", str(path), re.I)
    return match.group(1).lower() if match else Path(str(path)).parts[-3].lower()


def partition_patients(frame: pd.DataFrame, fractions: dict, seed: int) -> dict[str, pd.DataFrame]:
    """Split patients into train/select/dev/final, stratified by the full label pattern."""
    rng = np.random.default_rng(seed)
    labels = frame[list(LABEL_COLS)].to_numpy(float)
    characters = np.where(~np.isfinite(labels), "u", np.where(labels >= 0.5, "1", "0"))
    patterns = pd.Series(["".join(row) for row in characters], index=frame.index)
    groups = {pattern: rows.to_numpy() for pattern, rows in patterns.groupby(patterns).groups.items()}
    assigned = {name: [] for name in fractions}
    target = {name: fractions[name] * len(frame) for name in fractions}
    for _, rows in sorted(groups.items()):
        for row in rng.permutation(rows):
            fullest_last = min(fractions, key=lambda name: (len(assigned[name]) + 1) / (target[name] + 1e-9))
            assigned[fullest_last].append(row)
    frames = {name: frame.loc[rows].reset_index(drop=True) for name, rows in assigned.items()}
    assert_patient_disjoint(frames)
    return frames


def assert_patient_disjoint(frames: dict[str, pd.DataFrame]) -> None:
    patient_sets = {name: set(f.patient_id.astype(str)) for name, f in frames.items()}
    for a, b in combinations(patient_sets, 2):
        if patient_sets[a] & patient_sets[b]:
            raise RuntimeError(f"patient overlap between {a} and {b}")


def label_poles(labels: np.ndarray, class_index: int):
    """Row masks of known positives and known negatives for one pathology."""
    known = np.isfinite(labels[:, class_index])
    return (("positive", known & (labels[:, class_index] >= 0.5)),
            ("negative", known & (labels[:, class_index] < 0.5)))


def balanced_cap(frame: pd.DataFrame, cap: int, seed: int, min_per_label: int = 2) -> pd.DataFrame:
    """Label-balanced patient sample: about cap/10 known positives and negatives per pathology, then random fill."""
    if cap <= 0 or len(frame) <= cap:
        return frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    labels = frame[list(LABEL_COLS)].to_numpy(float)
    per_pole = max(min_per_label, cap // (2 * N_CLASSES))
    chosen = []
    for class_index in range(N_CLASSES):
        for pole, mask in label_poles(labels, class_index):
            rows = np.flatnonzero(mask)
            if len(rows) < min_per_label:
                raise RuntimeError(f"{CLASSES[class_index]}: only {len(rows)} known {pole} patients, need {min_per_label}")
            chosen += rng.choice(rows, min(per_pole, len(rows)), replace=False).tolist()
    chosen = list(dict.fromkeys(chosen))
    if len(chosen) > cap:
        # Keep the known positive/negative quota intact. The 5-shot support
        # sampler needs at least six of each pole for all five pathologies.
        progress(f"Balanced cohort: quota requires {len(chosen)} patients; expanding cap from {cap}")
        cap = len(chosen)
    remaining = np.setdiff1d(np.arange(len(frame)), np.asarray(chosen, dtype=int))
    chosen += rng.choice(remaining, min(cap - len(chosen), len(remaining)), replace=False).tolist()
    return frame.iloc[sorted(chosen[:cap])].reset_index(drop=True)


def floored_natural_cap(frame: pd.DataFrame, cap: int, seed: int, min_per_label: int, name: str) -> pd.DataFrame:
    """Natural-prevalence sample with a floor of min_per_label known positives AND negatives per pathology."""
    if cap <= 0 or len(frame) <= cap:
        return frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    labels = frame[list(LABEL_COLS)].to_numpy(float)
    chosen: list[int] = []
    for class_index in range(N_CLASSES):
        for pole, mask in label_poles(labels, class_index):
            already = int(mask[np.asarray(chosen, dtype=int)].sum()) if chosen else 0
            available = np.setdiff1d(np.flatnonzero(mask), np.asarray(chosen, dtype=int))
            needed = max(0, min_per_label - already)
            if len(available) < needed:
                raise RuntimeError(f"{name} pool has too few known {pole} {CLASSES[class_index]} patients "
                                   f"({already + len(available)} < {min_per_label})")
            if needed:
                chosen += rng.choice(available, needed, replace=False).tolist()
    if len(chosen) > cap:
        raise RuntimeError(f"{name} floor needs {len(chosen)} patients but CAP['{name}']={cap}")
    remaining = np.setdiff1d(np.arange(len(frame)), np.asarray(chosen, dtype=int))
    chosen += rng.choice(remaining, min(cap - len(chosen), len(remaining)), replace=False).tolist()
    return frame.iloc[sorted(chosen)].reset_index(drop=True)


def split_seed(name: str) -> int:
    return SEED + int(hashlib.sha256(name.encode()).hexdigest()[:6], 16) % 1000


def find_chexpert_csv(name: str) -> Path | None:
    """Direct locations first; the recursive search (slow on Kaggle's network mount) only as a fallback."""
    for candidate in (CHEXPERT_ROOT / name, CHEXPERT_ROOT / "CheXpert-v1.0-small" / name, CHEXPERT_ROOT / "CheXpert-v1.0" / name):
        if candidate.is_file():
            return candidate
    hits = sorted(CHEXPERT_ROOT.rglob(name)) if CHEXPERT_ROOT.is_dir() else []
    return hits[0] if hits else None


def resolve_image_path(raw_path) -> str | None:
    path = Path(str(raw_path))
    candidates = [path, CHEXPERT_ROOT / path]
    if len(path.parts) > 1:
        candidates.append(CHEXPERT_ROOT / Path(*path.parts[1:]))
    candidates.append(CHEXPERT_ROOT / "CheXpert-v1.0-small" / path)
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    return None


def load_chexpert(csv_path: Path) -> pd.DataFrame:
    """Frontal images, one per patient; CheXpert -1 (uncertain) and blanks become NaN (unknown)."""
    frame = pd.read_csv(csv_path)
    if "Frontal/Lateral" in frame:
        frame = frame[frame["Frontal/Lateral"].astype(str).str.lower().eq("frontal")].copy()
    labels = frame[list(CLASSES)].apply(pd.to_numeric, errors="coerce").astype(float).mask(lambda x: x.eq(-1), np.nan)
    for column_index, column in enumerate(LABEL_COLS):
        frame[column] = labels.iloc[:, column_index]
    frame["patient_id"] = frame["Path"].map(patient_from_path)
    frame["_pick"] = [hashlib.sha256(f"{SEED}|{p}|{q}".encode()).hexdigest() for p, q in zip(frame.patient_id, frame.Path)]
    frame = frame.sort_values(["patient_id", "_pick"]).drop_duplicates("patient_id").drop(columns="_pick")
    return frame[np.isfinite(frame[list(LABEL_COLS)].to_numpy(float)).any(1)].reset_index(drop=True)


def attach_paths(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    frame = frame.copy()
    frame["resolved_path"] = [resolve_image_path(p) for p in frame.Path]
    missing = int(frame.resolved_path.isna().sum())
    if missing:
        raise FileNotFoundError(f"{name}: {missing} CheXpert images not found -- check CHEXPERT_ROOT.")
    return frame.reset_index(drop=True)


def without_patients(frame: pd.DataFrame, *used: pd.DataFrame) -> pd.DataFrame:
    used_ids = set().union(*(set(u.patient_id.astype(str)) for u in used))
    return frame[~frame.patient_id.astype(str).isin(used_ids)].reset_index(drop=True)


progress("Reading CheXpert metadata and constructing patient-disjoint cohorts")
train_csv = find_chexpert_csv("train.csv")
if train_csv is None:
    raise FileNotFoundError("CheXpert train.csv not found under CHEXPERT_ROOT.")
FULL_SPLIT = partition_patients(load_chexpert(train_csv), SPLIT_FRACTIONS, SEED)

COHORTS = {}
COHORTS["preadapt"] = attach_paths(balanced_cap(FULL_SPLIT["train"], CAP["preadapt"], split_seed("train")), "preadapt")
support_pool = without_patients(FULL_SPLIT["train"], COHORTS["preadapt"])
COHORTS["support"] = attach_paths(balanced_cap(support_pool, CAP["support"], split_seed("support"),
                                               min_per_label=SUPPORT_MIN_PER_LABEL), "support")
gallery_pool = without_patients(support_pool, COHORTS["support"])
COHORTS["gallery"] = attach_paths(floored_natural_cap(gallery_pool, CAP["gallery"], split_seed("gallery"),
                                                      GALLERY_MIN_PER_LABEL, "gallery"), "gallery")
for split in ("select", "dev") + (("final",) if RUN_FINAL_TEST else ()):
    COHORTS[split] = attach_paths(floored_natural_cap(FULL_SPLIT[split], CAP[split], split_seed(split),
                                                      EVAL_MIN_PER_LABEL, split), split)
valid_csv = find_chexpert_csv("valid.csv") if USE_EXPERT_VALID else None
if valid_csv is not None:
    COHORTS["expert_valid"] = attach_paths(load_chexpert(valid_csv), "expert_valid")
elif USE_EXPERT_VALID:
    print("NOTE: valid.csv not found -- expert-label evaluation skipped.")
assert_patient_disjoint(COHORTS)

YS = {split: COHORTS[split][list(LABEL_COLS)].to_numpy(float) for split in COHORTS}
REPORT_SPLITS = [split for split in ("dev", "expert_valid", "final") if split in COHORTS]
SCORED_SPLITS = ["select"] + REPORT_SPLITS            # splits whose patients are scored against supports / gallery
FEATURE_SPLITS = ["preadapt", "support", "gallery"] + SCORED_SPLITS
if PRIMARY_SPLIT not in REPORT_SPLITS:
    raise RuntimeError(f"pre-registered primary split {PRIMARY_SPLIT} is not available")
for class_index, name in enumerate(CLASSES):
    counts = {split: (int((YS[split][:, class_index] == 1).sum()), int((YS[split][:, class_index] == 0).sum()))
              for split in ("support", "gallery")}
    if min(counts["support"]) < max(SHOTS):
        raise RuntimeError(f"support has {counts['support']} (+/-) {name}; {max(SHOTS)}-shot needs >= {max(SHOTS)} of each")
    if min(counts["gallery"]) < GALLERY_MIN_PER_LABEL:
        raise RuntimeError(f"gallery has {counts['gallery']} (+/-) {name}; need >= {GALLERY_MIN_PER_LABEL} of each")

LABEL_COUNTS = pd.DataFrame([{"split": split, **{name: f"{int((YS[split][:, i] == 1).sum())}+/{int((YS[split][:, i] == 0).sum())}-"
                                                  for i, name in enumerate(CLASSES)}} for split in COHORTS])
LABEL_COUNTS.to_csv(RUN_DIR / "cohort_label_counts.csv", index=False)
print("Cohorts (patients):", {split: len(frame) for split, frame in COHORTS.items()})
print("Known labels (pos/neg):\n" + LABEL_COUNTS.to_string(index=False))
print("Cell 5 ready.", elapsed())


[AGPQuant t+1.1 min] Reading CheXpert metadata and constructing patient-disjoint cohorts
Cohorts (patients): {'preadapt': 320, 'support': 320, 'gallery': 96, 'select': 80, 'dev': 80, 'final': 80}
Known labels (pos/neg):
   split Atelectasis Cardiomegaly Consolidation   Edema Pleural Effusion
preadapt     65+/35-      60+/57-       42+/95- 72+/60-         93+/114-
 support     56+/34-      59+/53-       42+/93- 69+/75-         95+/108-
 gallery     16+/12-      21+/14-       15+/19- 15+/17-          28+/24-
  select      14+/8-      15+/11-        9+/20- 18+/14-          26+/26-
     dev      15+/8-      12+/10-       10+/17- 20+/15-          28+/28-
   final      14+/8-      16+/11-       10+/16- 16+/18-          24+/22-
Cell 5 ready. t+1.3 min


In [6]:
# =============================================================================
# Cell 6 — Stage A: supervised encoder pre-adaptation (whole + patch T0 ViTs) and feature caches
#   whole : full image, masked BCE on its own path_head logits
#   patch : six fixed crops, crop logits pooled per image with log-mean-exp
#   The last ADAPT_BLOCKS blocks + final LayerNorm (backbone LR) and the head (head LR) train, with a
#   cosine anchor to the original embedding; the epoch is chosen by SELECT macro AUROC and the original
#   weights are kept if no epoch beats them. Features are cached for BOTH encoder sets and saved, so a
#   resumed run reuses exactly the same features (and therefore the same trained circuits).
# =============================================================================
class DisentangleHead(nn.Module):
    """Q1's head on the 768-d ViT embedding: 128-d causal feature zc and five pathology logits."""

    def __init__(self, in_dim=768, shared=256, out=FEATURE_DIM, n_classes=N_CLASSES):
        super().__init__()
        self.shared = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, shared), nn.GELU(), nn.Dropout(0.1), nn.LayerNorm(shared))
        self.causal = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.spurious = nn.Sequential(nn.Linear(shared, out), nn.GELU(), nn.LayerNorm(out))
        self.path_head = nn.Linear(out, n_classes)

    def forward(self, hidden):
        causal = F.normalize(self.causal(self.shared(hidden)), dim=-1)
        return {"zc": causal, "logits": self.path_head(causal)}


class ViTBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = vit_b_16(weights=None)
        self.encoder.heads = nn.Identity()
        self.head = DisentangleHead()

    def forward(self, images):
        return self.head(self.encoder(images))


def load_branch(path: Path) -> ViTBranch:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
    state = {k.removeprefix("module.").removeprefix("model."): v for k, v in state.items() if torch.is_tensor(v)}
    model = ViTBranch()
    model.load_state_dict(state, strict=True)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model.to(DEVICE).eval()


IMAGENET_STATS = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
EVAL_TRANSFORM = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(*IMAGENET_STATS)])
TRAIN_TRANSFORM = T.Compose([T.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)), T.ToTensor(), T.Normalize(*IMAGENET_STATS)])
PRE_AUGMENT = T.Compose([T.RandomRotation(5), T.ColorJitter(brightness=0.2, contrast=0.2)])   # no flips (laterality)


def crop_views(image: Image.Image) -> torch.Tensor:
    width, height = image.size
    return torch.stack([EVAL_TRANSFORM(image.crop((int(a * width), int(b * height), int(c * width), int(d * height))))
                        for a, b, c, d in TARGET_CROP_BOXES])


class ChestXrayDataset(Dataset):
    """mode 'whole' -> (3, 224, 224); mode 'patch' -> (6, 3, 224, 224). Labels may contain NaN."""

    def __init__(self, frame: pd.DataFrame, mode: str, train: bool = False):
        self.paths = frame.resolved_path.tolist()
        self.labels = frame[list(LABEL_COLS)].to_numpy(np.float32)
        self.mode, self.train = mode, train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as handle:
            image = handle.convert("RGB")
        if self.train:
            image = PRE_AUGMENT(image)
        if self.mode == "whole":
            pixels = (TRAIN_TRANSFORM if self.train else EVAL_TRANSFORM)(image)
        else:
            pixels = crop_views(image)
        return pixels, torch.from_numpy(self.labels[index])


def log_mean_exp(logits: torch.Tensor) -> torch.Tensor:
    """(N, K, 5) crop logits -> (N, 5) one score per image."""
    return torch.logsumexp(logits.float(), dim=1) - math.log(logits.shape[1])


@torch.inference_mode()
def encode_images(model: ViTBranch, frame: pd.DataFrame, mode: str) -> dict:
    model.eval()
    features, logits = [], []
    progress(f"Encode {mode}: {len(frame)} images ({6 if mode == 'patch' else 1} views each)")
    for batch_number, (pixels, _) in enumerate(DataLoader(ChestXrayDataset(frame, mode), batch_size=EVAL_BATCH, shuffle=False, num_workers=WORKERS), 1):
        enforce_runtime_budget(f"Stage A {mode} encode batch {batch_number}")
        if batch_number == 1 or batch_number % 4 == 0:
            progress(f"Encode {mode}: batch {batch_number}/{math.ceil(len(frame)/EVAL_BATCH)}")
        shape = pixels.shape[:2] if mode == "patch" else pixels.shape[:1]
        batch = pixels.flatten(0, 1) if mode == "patch" else pixels
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            output = model(batch.to(DEVICE))
        features.append(output["zc"].float().view(*shape, -1).cpu())
        logits.append(output["logits"].float().view(*shape, -1).cpu())
    return {"zc": torch.cat(features), "logits": torch.cat(logits)}


def branch_scores(encoded: dict) -> torch.Tensor:
    """One score per class per image (crops pooled with log-mean-exp)."""
    return log_mean_exp(encoded["logits"]) if encoded["logits"].dim() == 3 else encoded["logits"]


def set_trainable_blocks(model: ViTBranch) -> tuple[list, list]:
    """Unfreeze the last ADAPT_BLOCKS blocks + final LayerNorm (backbone group) and the head (head group)."""
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    backbone = list(model.encoder.encoder.layers[-ADAPT_BLOCKS:].parameters()) + list(model.encoder.encoder.ln.parameters())
    head = list(model.head.parameters())
    for parameter in backbone + head:
        parameter.requires_grad_(True)
    return backbone, head


def preadapt_branch(branch: str, original: ViTBranch, batch_size: int) -> tuple[dict, list, int]:
    """Fine-tune one branch on the preadapt cohort; return (best state, history, selected epoch)."""
    student = copy.deepcopy(original)
    backbone, head = set_trainable_blocks(student)
    optimizer = torch.optim.AdamW([{"params": backbone, "lr": LR_BACKBONE}, {"params": head, "lr": LR_HEAD}], weight_decay=1e-4)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == "cuda")
    loader = DataLoader(ChestXrayDataset(COHORTS["preadapt"], branch, train=True), batch_size=batch_size, shuffle=True,
                        num_workers=WORKERS, drop_last=True, generator=torch.Generator().manual_seed(SEED))

    def select_auroc(model):
        return macro_auc(YS["select"], branch_scores(encode_images(model, COHORTS["select"], branch)).numpy())

    def task_loss(pixels, labels):
        batch = pixels.flatten(0, 1) if branch == "patch" else pixels
        output = student(batch)
        with torch.no_grad():
            anchor_features = original(batch)["zc"]
        logits = log_mean_exp(output["logits"].view(len(pixels), -1, N_CLASSES)) if branch == "patch" else output["logits"]
        anchor = F.cosine_similarity(output["zc"].float(), anchor_features.float(), dim=-1).mean()
        return masked_bce(logits, labels), anchor

    best = {"auroc": select_auroc(original), "epoch": 0, "state": copy.deepcopy(student.state_dict())}
    history = [{"branch": branch, "epoch": 0, "select_auroc": best["auroc"]}]
    print(f"[{branch}] trainable {sum(p.numel() for p in backbone + head):,} params | SELECT AUROC before = {best['auroc']:.4f}")
    for epoch in range(1, ADAPT_EPOCHS + 1):
        student.train()
        losses, anchors = [], []
        for step, (pixels, labels) in enumerate(loader, 1):
            enforce_runtime_budget(f"stage A {branch} epoch {epoch} step {step}")
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                task, anchor = task_loss(pixels.to(DEVICE), labels.to(DEVICE))
                loss = task + LAMBDA_ANCHOR * (1 - anchor)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"[{branch}] non-finite loss at epoch {epoch} step {step}")
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(backbone + head, 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
            anchors.append(anchor.item())
            if SMOKE and step >= SMOKE_MAX_STEPS:
                break
        auroc = select_auroc(student)
        history.append({"branch": branch, "epoch": epoch, "select_auroc": auroc,
                        "loss": float(np.mean(losses)), "anchor_cos": float(np.mean(anchors))})
        print(f"[{branch}] epoch {epoch}/{ADAPT_EPOCHS} loss={np.mean(losses):.4f} anchor_cos={np.mean(anchors):.4f} "
              f"SELECT AUROC={auroc:.4f} | {elapsed()}")
        if np.isfinite(auroc) and auroc > best["auroc"]:
            best = {"auroc": auroc, "epoch": epoch, "state": copy.deepcopy(student.state_dict())}
    print(f"[{branch}] selected epoch {best['epoch']} (SELECT {best['auroc']:.4f})"
          + ("" if best["epoch"] else " -> original weights kept"))
    return {k: v.detach().cpu() for k, v in best["state"].items()}, history, best["epoch"]


CKPT_SHA = {"whole": file_sha256(T0_WHOLE_PATH), "patch": file_sha256(T0_PATCH_PATH)}
STAGE_A_FINGERPRINT = sha_of({"ckpt": CKPT_SHA, "cohorts": {s: sorted(COHORTS[s].patient_id.astype(str)) for s in FEATURE_SPLITS},
                              "recipe": [ADAPT_EPOCHS, ADAPT_BLOCKS, LR_BACKBONE, LR_HEAD, LAMBDA_ANCHOR, BATCH_WHOLE,
                                         BATCH_PATCH, TARGET_CROP_BOXES, RUN_MODE, SEED]})[:12]
feature_cache_name = f"encoder_features_{STAGE_A_FINGERPRINT}.pt"
cached = find_saved_file(feature_cache_name)
if cached is not None:
    saved = torch.load(cached, map_location="cpu", weights_only=False)
    ENC, SELECTED_EPOCHS = saved["features"], saved["selected_epochs"]
    print(f"Restored Stage A features from {cached} (fingerprint {STAGE_A_FINGERPRINT})")
else:
    ENC = {encoder_set: {"whole": {}, "patch": {}} for encoder_set in ENCODER_SETS}   # ENC[set][branch][split]
    SELECTED_EPOCHS, stage_a_history = {}, []
    for branch, path, batch_size in (("whole", T0_WHOLE_PATH, BATCH_WHOLE), ("patch", T0_PATCH_PATH, BATCH_PATCH)):
        progress(f"Stage A starting {branch} encoder; checkpoint: {path.name}")
        original = load_branch(path)
        best_state, history, SELECTED_EPOCHS[branch] = preadapt_branch(branch, original, batch_size)
        stage_a_history += history
        adapted = copy.deepcopy(original)
        adapted.load_state_dict(best_state)
        adapted.eval()
        torch.save({"model": best_state, "selected_epoch": SELECTED_EPOCHS[branch], "source_ckpt_sha": CKPT_SHA[branch]},
                   RUN_DIR / f"T1_CXP_{branch}_vit.pt")
        for encoder_set, model in (("original", original), ("preadapted", adapted)):
            for split in FEATURE_SPLITS:
                ENC[encoder_set][branch][split] = encode_images(model, COHORTS[split], branch)
        del original, adapted
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print(f"[{branch}] features cached for {ENCODER_SETS} x {FEATURE_SPLITS} | {elapsed()}")
    pd.DataFrame(stage_a_history).to_csv(RUN_DIR / "stage_a_history.csv", index=False)
    atomic_torch_save({"features": ENC, "selected_epochs": SELECTED_EPOCHS, "fingerprint": STAGE_A_FINGERPRINT},
                      RUN_DIR / feature_cache_name)
print("Selected pre-adaptation epochs:", SELECTED_EPOCHS, "| Cell 6 ready.", elapsed())


Restored Stage A features from /kaggle/input/datasets/omarradee/trace-output/agpquant_v6_shard_3_T5_trace_distance/full/encoder_features_f21534f61a8c.pt (fingerprint f21534f61a8c)
Selected pre-adaptation epochs: {'whole': 2, 'patch': 2} | Cell 6 ready. t+1.4 min


In [7]:
# =============================================================================
# Cell 7 — Class-wise whole <-> patch routing, routed features and mixture views
#   Routed feature (pure state input): w_patch * attention-pooled patch + (1 - w_patch) * whole, per pathology.
#   Mixture views (mixed state input), one builder per ablation:
#     routed_views      : 6 views, each w_patch * crop_k + (1 - w_patch) * whole   (main grid)
#     patch_only_views  : 6 views, crop_k alone                                     (A3a)
#     whole_patch_pair  : 2 views, whole and attention-pooled patch                 (A3b)
#   Mixture weights are that pathology's crop attention (softmax of its crop logits); for the pair they
#   are (1 - w_patch, w_patch). Routed views share the whole-image component, so their mixed states are
#   close to pure — Cell 10 reports the purity so this can be checked rather than assumed.
# =============================================================================
def advantage_weight(auroc_whole: float, auroc_patch: float, deadband: float = ROUTING_DEADBAND) -> float:
    """Patch weight from each branch's advantage over chance; advantages inside the deadband count as zero."""
    advantage_whole = (auroc_whole - 0.5) if (auroc_whole - 0.5) > deadband else 0.0
    advantage_patch = (auroc_patch - 0.5) if (auroc_patch - 0.5) > deadband else 0.0
    total = advantage_patch + advantage_whole
    return advantage_patch / total if total > 1e-6 else 0.5


def routing_from_csv(path: Path) -> pd.DataFrame:
    table = pd.read_csv(path)
    table = table[table["tier"].astype(str).eq("T0") & table["stage"].astype(str).str.contains("validation", case=False)]
    pivot = table.pivot_table(index="pathology", columns="branch", values="auroc")
    missing = [name for name in CLASSES if name not in pivot.index]
    if missing or not {"whole", "patch"} <= set(pivot.columns):
        raise ValueError(f"routing CSV lacks {missing or 'whole/patch columns'}")
    return pd.DataFrame([{"class": name, "auroc_whole": pivot.loc[name, "whole"], "auroc_patch": pivot.loc[name, "patch"],
                          "patch_weight": advantage_weight(pivot.loc[name, "whole"], pivot.loc[name, "patch"])}
                         for name in CLASSES])


def routing_from_select(encoder_set: str) -> pd.DataFrame:
    labels = YS["select"]
    whole = branch_scores(ENC[encoder_set]["whole"]["select"]).numpy()
    patch = branch_scores(ENC[encoder_set]["patch"]["select"]).numpy()
    rows = []
    for class_index, name in enumerate(CLASSES):
        known = two_class_mask(labels, whole, class_index)
        auroc_whole = roc_auc_score(labels[known, class_index], whole[known, class_index]) if known is not None else 0.5
        auroc_patch = roc_auc_score(labels[known, class_index], patch[known, class_index]) if known is not None else 0.5
        rows.append({"class": name, "auroc_whole": auroc_whole, "auroc_patch": auroc_patch,
                     "patch_weight": advantage_weight(auroc_whole, auroc_patch)})
    return pd.DataFrame(rows)


ROUTING = {}
for encoder_set in ENCODER_SETS:
    if ROUTING_SOURCE == "nih_val_csv":
        ROUTING[encoder_set] = routing_from_csv(ROUTING_CSV)
    elif ROUTING_SOURCE == "select":
        ROUTING[encoder_set] = routing_from_select(encoder_set)
    elif ROUTING_SOURCE == "equal":
        ROUTING[encoder_set] = pd.DataFrame({"class": CLASSES, "patch_weight": [0.5] * N_CLASSES})
    else:
        raise ValueError(ROUTING_SOURCE)
    print(f"=== ROUTING [{encoder_set}] source={ROUTING_SOURCE} ===\n" + ROUTING[encoder_set].round(3).to_string(index=False))
PATCH_WEIGHT = {s: torch.tensor(ROUTING[s].patch_weight.to_numpy(), dtype=torch.float32) for s in ENCODER_SETS}


def _branches(encoder_set: str, split: str):
    """Whole feature (N, 128), crop features (N, 6, 128), crop attention per pathology (N, 5, 6)."""
    whole = ENC[encoder_set]["whole"][split]["zc"].clone()
    patch = ENC[encoder_set]["patch"][split]
    attention = torch.softmax(patch["logits"].clone(), dim=1).permute(0, 2, 1).contiguous()
    return whole, patch["zc"].clone(), attention


def routed_features(encoder_set: str, split: str) -> torch.Tensor:
    """(N, 5, 128) one routed feature per patient and pathology."""
    whole, crops, attention = _branches(encoder_set, split)
    pooled_patch = torch.einsum("nck,nkd->ncd", attention, crops)
    weight = PATCH_WEIGHT[encoder_set][None, :, None]
    return F.normalize(weight * pooled_patch + (1 - weight) * whole[:, None, :], dim=-1)


def routed_views(encoder_set: str, split: str):
    whole, crops, attention = _branches(encoder_set, split)
    weight = PATCH_WEIGHT[encoder_set][None, :, None, None]
    views = F.normalize(weight * crops[:, None] + (1 - weight) * whole[:, None, None, :], dim=-1)
    return views, attention


def patch_only_views(encoder_set: str, split: str):
    _, crops, attention = _branches(encoder_set, split)
    return F.normalize(crops, dim=-1)[:, None].expand(-1, N_CLASSES, -1, -1).contiguous(), attention


def whole_patch_pair(encoder_set: str, split: str):
    whole, crops, attention = _branches(encoder_set, split)
    pooled_patch = F.normalize(torch.einsum("nck,nkd->ncd", attention, crops), dim=-1)
    whole_view = F.normalize(whole, dim=-1)[:, None].expand(-1, N_CLASSES, -1)
    views = torch.stack([whole_view, pooled_patch], dim=2)                               # (N, 5, 2, 128)
    weight = PATCH_WEIGHT[encoder_set]
    weights = torch.stack([1 - weight, weight], -1)[None].expand(len(whole), -1, -1).contiguous()
    return views, weights


VIEW_BUILDERS = {"routed_views": routed_views, "patch_only_views": patch_only_views, "whole_patch_pair": whole_patch_pair}
FEATS = {s: {split: routed_features(s, split) for split in FEATURE_SPLITS} for s in ENCODER_SETS}
FEATURE_SHA = {s: {split: tensor_sha256(FEATS[s][split]) for split in FEATURE_SPLITS} for s in ENCODER_SETS}


def model_inputs(encoder_set: str, view_builder: str, split: str, rows=None):
    """(features, views, view_weights) for one split, optionally restricted to some rows."""
    views, weights = VIEW_BUILDERS[view_builder](encoder_set, split)
    features = FEATS[encoder_set][split]
    if rows is not None:
        rows = torch.as_tensor(np.asarray(rows), dtype=torch.long)
        features, views, weights = features[rows], views[rows], weights[rows]
    return features, views, weights


audit_rows = []
for encoder_set, split in product(ENCODER_SETS, [s for s in FEATURE_SPLITS if s not in ("preadapt", "support")]):
    whole_score = branch_scores(ENC[encoder_set]["whole"][split]).numpy()
    patch_score = branch_scores(ENC[encoder_set]["patch"][split]).numpy()
    weight = PATCH_WEIGHT[encoder_set].numpy()
    audit_rows.append({"encoder_set": encoder_set, "split": split, "whole": macro_auc(YS[split], whole_score),
                       "patch": macro_auc(YS[split], patch_score),
                       "routed_score_mix": macro_auc(YS[split], weight * patch_score + (1 - weight) * whole_score)})
FROZEN_AUDIT = pd.DataFrame(audit_rows)
FROZEN_AUDIT.to_csv(RUN_DIR / "frozen_encoder_audit.csv", index=False)
print("=== FROZEN ENCODER AUDIT, macro AUROC (no circuit, no K-shot) ===\n" + FROZEN_AUDIT.round(4).to_string(index=False))
print("Cell 7 ready.", elapsed())


=== ROUTING [original] source=nih_val_csv ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541
=== ROUTING [preadapted] source=nih_val_csv ===
           class  auroc_whole  auroc_patch  patch_weight
     Atelectasis        0.781        0.790         0.508
    Cardiomegaly        0.857        0.821         0.474
   Consolidation        0.691        0.698         0.509
           Edema        0.843        0.786         0.454
Pleural Effusion        0.777        0.826         0.541
=== FROZEN ENCODER AUDIT, macro AUROC (no circuit, no K-shot) ===
encoder_set   split  whole  patch  routed_score_mix
   original gallery 0.5089 0.5149            0.5063
   original  select 0.5262 0.4676            0.5059


In [8]:
# =============================================================================
# Cell 8 — CheXzero zero-shot prior (fixed, not pre-adapted)
#   PRIOR[split] is one zero-shot score per patient and pathology, shared by every arm. It is used
#   only for the "+prior" classification rows and the P0_prior_only reference, never for retrieval.
#   Disclosure for the paper: confirm whether this checkpoint was ever selected on CheXpert validation
#   data; if so, expert_valid is not fully independent of the prior.
# =============================================================================
CHEXZERO_MEAN, CHEXZERO_STD = 101.48761, 83.43944
CHEXZERO_RESIZE = T.Resize((224, 224), interpolation=T.InterpolationMode.BICUBIC)


def load_chexzero(path: Path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and isinstance(state.get("state_dict"), dict):
        state = state["state_dict"]
    if isinstance(state, dict) and isinstance(state.get("model"), dict):
        state = state["model"]
    if not isinstance(state, dict):
        state = state.state_dict()
    state = {k.removeprefix("module."): v for k, v in state.items() if torch.is_tensor(v)}
    model = build_model_from_openai_state_dict(state).float().to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def pad_to_square(path: str, size: int = 320) -> Image.Image:
    with Image.open(path) as handle:
        image = handle.convert("L")
    width, height = image.size
    scale = size / max(width, height)
    image = image.resize((max(1, round(width * scale)), max(1, round(height * scale))), Image.BICUBIC)
    canvas = Image.new("L", (size, size), 0)
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas


class ChexzeroDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.paths = frame.resolved_path.tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = CHEXZERO_RESIZE(pad_to_square(self.paths[index]))
        return ((TF.pil_to_tensor(image).float() - CHEXZERO_MEAN) / CHEXZERO_STD).expand(3, -1, -1).contiguous()


@torch.inference_mode()
def encode_chexzero(visual, frame: pd.DataFrame) -> torch.Tensor:
    embeddings = []
    progress(f"CheXzero prior: {len(frame)} images")
    for batch_number, pixels in enumerate(DataLoader(ChexzeroDataset(frame), batch_size=64, shuffle=False, num_workers=WORKERS), 1):
        enforce_runtime_budget(f"CheXzero batch {batch_number}")
        progress(f"CheXzero: batch {batch_number}/{math.ceil(len(frame)/64)}")
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            embedding = visual(pixels.to(DEVICE))
        embeddings.append(F.normalize(embedding.float(), dim=-1).cpu())
    return torch.cat(embeddings)


clip_model = load_chexzero(CHEXZERO_CKPT)
with torch.no_grad():
    prompts = [name.lower() for name in CLASSES]
    text = clip_model.encode_text(open_clip.tokenize(prompts + [f"no {p}" for p in prompts]).to(DEVICE))
    text = F.normalize(text.float(), dim=-1).cpu()
TEXT_DIRECTION = text[:N_CLASSES] - text[N_CLASSES:]               # "finding" minus "no finding"
chexzero_visual = copy.deepcopy(clip_model.visual).eval()
del clip_model
PRIOR = {split: (100.0 * encode_chexzero(chexzero_visual, COHORTS[split]) @ TEXT_DIRECTION.T).numpy()
         for split in ["support"] + SCORED_SPLITS}
del chexzero_visual
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
prior_select_auroc = macro_auc(YS["select"], PRIOR["select"])
print(f"CheXzero zero-shot SELECT macro AUROC: {prior_select_auroc:.4f}")
if prior_select_auroc < 0.60:
    print("WARNING: low CheXzero zero-shot AUROC -- check the checkpoint before trusting the +prior rows.")
print("Cell 8 ready.", elapsed())


[AGPQuant t+1.5 min] HEARTBEAT | stage: QNode hs | log: /kaggle/working/agpquant_progress.log
[AGPQuant t+1.5 min] CheXzero prior: 320 images
[AGPQuant t+1.5 min] CheXzero: batch 1/5
[AGPQuant t+1.5 min] CheXzero: batch 2/5
[AGPQuant t+1.6 min] CheXzero: batch 3/5
[AGPQuant t+1.6 min] CheXzero: batch 4/5
[AGPQuant t+1.6 min] CheXzero: batch 5/5
[AGPQuant t+1.6 min] CheXzero prior: 80 images
[AGPQuant t+1.6 min] CheXzero: batch 1/2
[AGPQuant t+1.6 min] CheXzero: batch 2/2
[AGPQuant t+1.6 min] CheXzero prior: 80 images
[AGPQuant t+1.6 min] CheXzero: batch 1/2
[AGPQuant t+1.6 min] CheXzero: batch 2/2
[AGPQuant t+1.6 min] CheXzero prior: 80 images
[AGPQuant t+1.6 min] CheXzero: batch 1/2
[AGPQuant t+1.6 min] CheXzero: batch 2/2
CheXzero zero-shot SELECT macro AUROC: 0.9292
Cell 8 ready. t+1.6 min


## Stage B — train the circuit once per (arm, seed)
Only the similarity inside the loss differs between arms: same circuit, same patients, same initial weights and same batch order for a given seed.

In [9]:
# =============================================================================
# Cell 9 — Stage B: train the H1 circuit once per (training arm, model seed)
#   Data   : the 1,600-patient preadapt cohort (routed features from the frozen Stage A encoders)
#   Loss   : T0_readout -> BCE + ranking loss on the <Z> readout (H1's original recipe)
#            every other arm -> the episodic leave-one-out retrieval loss of the classical Q1 notebook:
#              inside a class-balanced batch, each patient is scored against the OTHER batch patients
#              (mean similarity to known positives minus mean similarity to known negatives, per pathology),
#              the score is standardised, turned into a logit by a learned temperature and bias, and fed to BCE.
#            The similarity inside that loss is the arm's metric — the only difference between arms.
#   Epoch  : chosen by SELECT macro AUROC, scoring SELECT patients against a fixed labelled reference set
#            of preadapt patients with the arm's matched rule (the readout arm uses its logits).
#   Fairness: every arm with the same model seed starts from the same initial weights and sees the same
#            batches in the same order, so arm-to-arm differences come from the loss alone.
#   Resume : each finished model is saved; re-running (or RESUME_FROM) skips it.
# =============================================================================
@dataclass(frozen=True)
class ModelSpec:
    arm: str
    variant: str
    seed: int

    @property
    def model_id(self) -> str:
        return f"{self.variant}__{self.arm}__seed{self.seed}"


ALL_VARIANTS = {MAIN_VARIANT.name: MAIN_VARIANT, **ABLATIONS}
ENCODED_PARTS = ("angles", "pure", "density", "readout")
PARTS_FOR = {"readout": ("readout",), "pure": ("pure",), "mixed": ("density",), "angles": ()}


class ClassBalancedBatchSampler(torch.utils.data.Sampler):
    """Every batch holds >= PER_CLASS_QUOTA known positives AND negatives per pathology, no duplicates, rest random.

    Duplicates are excluded because a patient must never be its own support in the leave-one-out loss.
    """

    def __init__(self, labels: np.ndarray, batch_size: int, n_batches: int, seed: int):
        if batch_size < 2 * PER_CLASS_QUOTA * N_CLASSES:
            raise ValueError(f"CIRCUIT_BATCH={batch_size} < 2 x quota x classes = {2 * PER_CLASS_QUOTA * N_CLASSES}")
        self.n_rows, self.batch_size, self.n_batches = len(labels), batch_size, n_batches
        self.rng = np.random.default_rng(seed)
        self.pools = [[np.flatnonzero(mask) for _, mask in label_poles(labels, c)] for c in range(N_CLASSES)]

    def __len__(self):
        return self.n_batches

    def __iter__(self):
        for _ in range(self.n_batches):
            batch, seen = [], set()
            for class_index in self.rng.permutation(N_CLASSES):
                for pool in self.pools[class_index]:
                    picks = [int(i) for i in self.rng.permutation(pool) if int(i) not in seen][:PER_CLASS_QUOTA]
                    batch += picks
                    seen.update(picks)
            for row in self.rng.permutation(self.n_rows):
                if len(batch) >= self.batch_size:
                    break
                if int(row) not in seen:
                    batch.append(int(row))
                    seen.add(int(row))
            yield batch[:self.batch_size]


def training_representation(model: H1Retriever, arm: TrainingArm, features, views, view_weights) -> torch.Tensor:
    """The quantity the arm's loss reads, for a batch: (B, 5, ...)."""
    angles = model.angles(features)
    if arm.representation == "readout":
        return model.readout_logits(angles)
    if arm.representation == "angles":
        return angles
    if arm.representation == "pure":
        return model.pure_states(angles)
    return model.mixed_densities(model.view_angles(views), view_weights)


def batch_similarity(arm: TrainingArm, representation: torch.Tensor, class_index: int) -> torch.Tensor:
    """(B, B) symmetric similarity of distinct batch pairs; the diagonal is zero and never used."""
    data = representation[:, class_index]
    size = len(data)
    precision = shrunk_precision(data.detach()) if arm.metric == "mahalanobis" else None   # batch covariance, no gradient
    upper_i, upper_j = torch.triu_indices(size, size, offset=1, device=data.device)
    values = paired_similarity(arm.metric, arm.representation, data[upper_i], data[upper_j], precision).to(data.device)
    matrix = torch.zeros(size, size, device=data.device, dtype=values.dtype)
    matrix = matrix.index_put((upper_i, upper_j), values)
    return matrix + matrix.T


def leave_one_out_margin(similarity: torch.Tensor, labels: torch.Tensor):
    """Per patient: mean similarity to OTHER known positives minus mean similarity to OTHER known negatives."""
    known = torch.isfinite(labels)
    not_self = ~torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    positives = (known & (labels >= 0.5)).float()[None, :] * not_self
    negatives = (known & (labels < 0.5)).float()[None, :] * not_self
    n_positive, n_negative = positives.sum(1), negatives.sum(1)
    margin = (similarity * positives).sum(1) / n_positive.clamp_min(1) - (similarity * negatives).sum(1) / n_negative.clamp_min(1)
    valid = known & (n_positive > 0) & (n_negative > 0)
    return margin, valid


def training_loss(model: H1Retriever, arm: TrainingArm, features, views, view_weights, labels) -> torch.Tensor:
    representation = training_representation(model, arm, features, views, view_weights)
    if arm.representation == "readout":
        return masked_bce(representation, labels) + READOUT_RANK_WEIGHT * pairwise_rank_loss(representation, labels)
    logits, targets = [], []
    for class_index in range(N_CLASSES):
        margin, valid = leave_one_out_margin(batch_similarity(arm, representation, class_index), labels[:, class_index])
        spread = margin[valid].detach().std() if int(valid.sum()) > 1 else torch.ones((), device=margin.device)
        standardised = margin / torch.nan_to_num(spread, nan=1.0).clamp_min(1e-6)
        logits.append(model.log_temperature.exp() * standardised + model.class_bias[class_index])
        targets.append(labels[:, class_index].masked_fill(~valid, float("nan")))
    return masked_bce(torch.stack(logits, 1), torch.stack(targets, 1))


@torch.no_grad()
def encode_split(model: H1Retriever, variant: Variant, split: str, rows=None, parts=ENCODED_PARTS, batch_size: int = 256) -> dict:
    """Every representation of a split (or some of its rows), produced by the QNodes, stored on CPU."""
    model.eval()
    features, views, view_weights = model_inputs(variant.encoder_set, variant.view_builder, split, rows)
    collected = {key: [] for key in ("angles", "pure", "readout", "density", "view_angles", "view_weights")}
    for start in range(0, len(features), batch_size):
        feature_batch = features[start:start + batch_size].to(DEVICE)
        angles = model.angles(feature_batch)
        collected["angles"].append(angles.cpu())
        if "pure" in parts:
            collected["pure"].append(model.pure_states(angles).cpu())
        if "readout" in parts:
            collected["readout"].append(model.readout_logits(angles).cpu())
        if "density" in parts:
            view_angles = model.view_angles(views[start:start + batch_size].to(DEVICE))
            weights = view_weights[start:start + batch_size].to(DEVICE)
            collected["density"].append(model.mixed_densities(view_angles, weights).cpu())
            collected["view_angles"].append(view_angles.cpu())
            collected["view_weights"].append(weights.cpu())
    return {key: torch.cat(values) for key, values in collected.items() if values}


def prototype_margin_scores(rule_name: str, query: dict, reference: dict, reference_labels: np.ndarray,
                            audit_seed: int = 0) -> np.ndarray:
    """(Q, 5): mean score to the reference's known positives minus mean score to its known negatives."""
    margins = np.full((len(query["angles"]), N_CLASSES), np.nan, dtype=np.float32)
    for class_index in range(N_CLASSES):
        known = np.isfinite(reference_labels[:, class_index])
        positive = known & (reference_labels[:, class_index] >= 0.5)
        negative = known & (reference_labels[:, class_index] < 0.5)
        if positive.any() and negative.any():
            scores = score_matrix(rule_name, query, reference, class_index, audit_seed + class_index)
            margins[:, class_index] = scores[:, positive].mean(1) - scores[:, negative].mean(1)
    return margins


SELECT_REFERENCE_ROWS = np.sort(np.random.default_rng(SEED + 3).choice(
    len(COHORTS["preadapt"]), min(SELECT_REFERENCE_SIZE, len(COHORTS["preadapt"])), replace=False))


def select_auroc(model: H1Retriever, arm: TrainingArm, variant: Variant) -> float:
    """Epoch-selection score: SELECT macro AUROC of the arm's matched rule (or its readout logits)."""
    if arm.representation == "readout":
        return macro_auc(YS["select"], encode_split(model, variant, "select", parts=("readout",))["readout"].numpy())
    parts = PARTS_FOR[arm.representation]
    query = encode_split(model, variant, "select", parts=parts)
    reference = encode_split(model, variant, "preadapt", rows=SELECT_REFERENCE_ROWS, parts=parts)
    reference["precision"] = class_precisions(reference["angles"])
    margins = prototype_margin_scores(arm.matched_rule, query, reference, YS["preadapt"][SELECT_REFERENCE_ROWS])
    return macro_auc(YS["select"], margins)


def training_fingerprint(spec: ModelSpec) -> str:
    variant = ALL_VARIANTS[spec.variant]
    return sha_of({"spec": asdict(spec), "variant": asdict(variant), "arm": asdict(TRAINING_ARMS[spec.arm]),
                   "features": FEATURE_SHA[variant.encoder_set], "patch_weight": PATCH_WEIGHT[variant.encoder_set].tolist(),
                   "circuit": [CIRCUIT_ID, N_QUBITS, VQC_LAYERS],
                   "recipe": ["v6.1_qnode_per_training_batch", CIRCUIT_EPOCHS, CIRCUIT_PATIENCE, CIRCUIT_BATCH, PER_CLASS_QUOTA, CIRCUIT_LR,
                              CIRCUIT_WEIGHT_DECAY, SCALE_LR, GRAD_CLIP, READOUT_RANK_WEIGHT, SELECT_REFERENCE_SIZE]})[:12]


def build_model(spec: ModelSpec, state: dict | None = None) -> H1Retriever:
    seed_all(spec.seed)                           # same seed -> same initial weights for every arm
    model = H1Retriever(ALL_VARIANTS[spec.variant]).to(DEVICE)
    if state is not None:
        model.load_state_dict(state)
    return model


def train_circuit(spec: ModelSpec) -> dict:
    """Train (or restore) one model and return its record: weights, history and cost."""
    arm, variant = TRAINING_ARMS[spec.arm], ALL_VARIANTS[spec.variant]
    checkpoint_name = f"circuit__{spec.model_id}__{training_fingerprint(spec)}.pt"
    saved = find_saved_file(checkpoint_name)
    if saved is not None:
        return torch.load(saved, map_location="cpu", weights_only=False)

    model = build_model(spec)
    scale_parameters = [model.log_temperature, model.class_bias]
    scale_ids = {id(p) for p in scale_parameters}
    circuit_parameters = [p for p in model.parameters() if p.requires_grad and id(p) not in scale_ids]
    optimizer = torch.optim.AdamW([{"params": circuit_parameters, "lr": CIRCUIT_LR, "weight_decay": CIRCUIT_WEIGHT_DECAY},
                                   {"params": scale_parameters, "lr": SCALE_LR, "weight_decay": 0.0}])
    features, views, view_weights = model_inputs(variant.encoder_set, variant.view_builder, "preadapt")
    labels = torch.tensor(YS["preadapt"], dtype=torch.float32)
    sampler = ClassBalancedBatchSampler(YS["preadapt"], CIRCUIT_BATCH, max(1, len(labels) // CIRCUIT_BATCH), spec.seed)
    qnode_calls_before = QSTATS["qnode_calls"]

    best = {"score": select_auroc(model, arm, variant), "epoch": 0, "state": copy.deepcopy(model.state_dict())}
    history = [{"epoch": 0, "select_auroc": best["score"]}]
    stale_epochs, skipped_steps, train_seconds = 0, 0, 0.0
    for epoch in range(1, CIRCUIT_EPOCHS + 1):
        progress(f"Training {spec.model_id}, epoch {epoch}/{CIRCUIT_EPOCHS}")
        model.train()
        start, losses = time.time(), []
        for batch_number, rows in enumerate(sampler, 1):
            if batch_number == 1 or batch_number % 4 == 0:
                progress(f"Training {spec.model_id}, epoch {epoch}, batch {batch_number}/{len(sampler)}")
            enforce_runtime_budget(f"stage B {spec.model_id} epoch {epoch}")
            rows = torch.as_tensor(rows)
            qnode_before_step = QSTATS["qnode_calls"]
            loss = training_loss(model, arm, features[rows].to(DEVICE), views[rows].to(DEVICE),
                                 view_weights[rows].to(DEVICE), labels[rows].to(DEVICE))
            if arm.family == "quantum" and QSTATS["qnode_calls"] <= qnode_before_step:
                raise RuntimeError(f"{spec.model_id}: quantum training batch {batch_number} did not execute a PennyLane QNode")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            gradients_finite = all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
            if arm.family == "quantum" and gradients_finite:
                mapped_grad = [model.angle_maps[c].weight.grad for c in range(N_CLASSES)]
                if not any(g is not None and torch.isfinite(g).all() and float(g.detach().abs().max()) > 0 for g in mapped_grad):
                    raise RuntimeError(f"{spec.model_id}: no gradient reached the quantum angle maps in batch {batch_number}")
            if not torch.isfinite(loss) or not gradients_finite:
                skipped_steps += 1                       # counted and reported; never silently absorbed
                continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            losses.append(loss.item())
        train_seconds += time.time() - start
        score = select_auroc(model, arm, variant)
        history.append({"epoch": epoch, "loss": float(np.mean(losses)) if losses else np.nan, "select_auroc": score,
                        "seconds": time.time() - start})
        if np.isfinite(score) and score > best["score"]:
            best, stale_epochs = {"score": score, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            stale_epochs += 1
        if stale_epochs >= CIRCUIT_PATIENCE:
            break

    if skipped_steps > 0.1 * len(sampler) * len(history):
        raise FloatingPointError(f"{spec.model_id}: {skipped_steps} steps had non-finite gradients")
    record = {"spec": asdict(spec), "state": {k: v.detach().cpu() for k, v in best["state"].items()},
              "history": history, "selected_epoch": best["epoch"], "select_auroc": best["score"],
              "epochs_run": len(history) - 1, "train_seconds": train_seconds,
              "seconds_per_epoch": train_seconds / max(1, len(history) - 1), "skipped_steps": skipped_steps,
              "qnode_calls": QSTATS["qnode_calls"] - qnode_calls_before, "fingerprint": training_fingerprint(spec)}
    atomic_torch_save(record, MODEL_DIR / checkpoint_name)
    return record


def train_many(specs: list[ModelSpec], label: str) -> dict[str, dict]:
    """Train a list of models in order, with a runtime projection after each one."""
    records, durations = {}, []
    for index, spec in enumerate(specs, 1):
        if not stage_budget_left():
            print(f"WARNING: stage budget reached before {spec.model_id}; re-run to continue.")
            break
        progress(f"{label} model {index}/{len(specs)}: {spec.model_id} starting")
        started = time.time()
        records[spec.model_id] = train_circuit(spec)
        durations.append(time.time() - started)
        record = records[spec.model_id]
        remaining_hours = (len(specs) - index) * float(np.mean(durations)) / 3600
        print(f"[{label} {index}/{len(specs)}] {spec.model_id}: SELECT AUROC {record['select_auroc']:.4f} "
              f"(epoch {record['selected_epoch']}/{record['epochs_run']}, {record['seconds_per_epoch']:.0f} s/epoch, "
              f"skipped steps {record['skipped_steps']}) | {elapsed()} | ~{remaining_hours:.2f} h left in this stage")
    return records


for _arm in SHARD_ARMS:
    if _arm not in TRAINING_ARMS:
        raise RuntimeError(f"SHARD_ARMS names {_arm!r}, which is not a training arm: {tuple(TRAINING_ARMS)}")
ACTIVE_TRAINING_ARMS = ("T0_readout", "T1_pure_fidelity", "T2_mixed_uhlmann", "C1_angle_cosine") if SMOKE else tuple(SHARD_ARMS)
print(f"SHARD {SHARD_NAME}: training arms {ACTIVE_TRAINING_ARMS} x seeds {MODEL_SEEDS}")
MAIN_SPECS = [ModelSpec(arm, MAIN_VARIANT.name, seed) for seed in MODEL_SEEDS for arm in ACTIVE_TRAINING_ARMS]  # seed-major
TRAINED = train_many(MAIN_SPECS, "main")
TRAINING_TABLE = pd.DataFrame([{"model_id": model_id, **record["spec"], "select_auroc": record["select_auroc"],
                                "selected_epoch": record["selected_epoch"], "epochs_run": record["epochs_run"],
                                "seconds_per_epoch": record["seconds_per_epoch"], "skipped_steps": record["skipped_steps"]}
                               for model_id, record in TRAINED.items()])
TRAINING_TABLE.to_csv(RUN_DIR / "training_summary.csv", index=False)
print(TRAINING_TABLE.drop(columns=["model_id"]).round(4).to_string(index=False))
print("Cell 9 ready.", elapsed())


SHARD 3_T5_trace_distance: training arms ('T5_mixed_trace_distance',) x seeds (42, 314)
[AGPQuant t+1.6 min] main model 1/2: main__T5_mixed_trace_distance__seed42 starting
[main 1/2] main__T5_mixed_trace_distance__seed42: SELECT AUROC 0.8930 (epoch 1/4, 121 s/epoch, skipped steps 0) | t+1.6 min | ~0.00 h left in this stage
[AGPQuant t+1.6 min] main model 2/2: main__T5_mixed_trace_distance__seed314 starting
[main 2/2] main__T5_mixed_trace_distance__seed314: SELECT AUROC 0.8531 (epoch 0/3, 122 s/epoch, skipped steps 0) | t+1.6 min | ~0.00 h left in this stage
                    arm variant  seed  select_auroc  selected_epoch  epochs_run  seconds_per_epoch  skipped_steps
T5_mixed_trace_distance    main    42        0.8930               1           4           120.8640              0
T5_mixed_trace_distance    main   314        0.8531               0           3           122.2606              0
Cell 9 ready. t+1.6 min


## Stage C — score every trained model with every rule
Every positive query ranks the same full gallery; query sets and K-shot support draws are shared by every model, rule, seed and variant, so all comparisons are paired. SELECT is used only for epoch selection, beta tuning, standardisation and choosing the ablation targets.

In [10]:
# =============================================================================
# Cell 10 — Stage C: evaluate every trained model with every scoring rule
#   Retrieval (primary)      : each positive query of a scored split ranks the full gallery; relevant = same
#                              pathology positive. Depends only on the trained model, so seeds replicate it.
#   K-shot classification    : score = mean rule value to K positive supports minus mean to K negative supports
#                              (supports drawn from the support cohort; identical draws for every model).
#                              Scores are standardised on SELECT and thresholded at the SELECT Youden point.
#                              "+prior" rows add the CheXzero prior with a weight beta tuned on SELECT using
#                              separate tuning draws.
#   Diagnostics              : purity of the mixed states, similarity spread, and Kendall tau between the
#                              rankings of every pair of rules (tau near 1 means two rules cannot differ).
#   Audits                   : every scoring call checks sampled pairs against qml.math; every model also has
#                              sampled pairs measured by the overlap circuit and the 12-qubit Hilbert-Schmidt circuit.
#   Results are saved per model, so a stopped run resumes where it left off.
# =============================================================================
def draw_support(labels: np.ndarray, shot: int, rng: np.random.Generator) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    """K known positives and K known negatives per pathology."""
    draw = {}
    for class_index, name in enumerate(CLASSES):
        (_, positive), (_, negative) = label_poles(labels, class_index)
        positive, negative = np.flatnonzero(positive), np.flatnonzero(negative)
        if min(len(positive), len(negative)) < shot:
            raise RuntimeError(f"{name}: support pool too small for {shot}-shot")
        draw[class_index] = (np.sort(rng.choice(positive, shot, replace=False)), np.sort(rng.choice(negative, shot, replace=False)))
    return draw


def support_rng(shot: int, episode: int, tuning: bool = False) -> np.random.Generator:
    return np.random.default_rng(SEED + 7919 * shot + 101 * episode + (500_000 if tuning else 0))


SUPPORT_DRAWS = {(shot, episode): draw_support(YS["support"], shot, support_rng(shot, episode))
                 for shot in SHOTS for episode in range(EPISODES)}
TUNING_DRAWS = {(shot, episode): draw_support(YS["support"], shot, support_rng(shot, episode, tuning=True))
                for shot in SHOTS for episode in range(TUNING_EPISODES)}
SUPPORT_ROWS = np.unique(np.concatenate([rows for draws in (SUPPORT_DRAWS, TUNING_DRAWS) for draw in draws.values()
                                         for pair in draw.values() for rows in pair]))

# Query sets are fixed and shared by every model, rule, seed and variant, so every comparison is paired.
QUERY_ROWS = {}
for split in SCORED_SPLITS:
    for class_index in range(N_CLASSES):
        positives = np.flatnonzero(np.isfinite(YS[split][:, class_index]) & (YS[split][:, class_index] >= 0.5))
        rng = np.random.default_rng(stable_seed("queries", split, class_index))
        QUERY_ROWS[(split, class_index)] = np.sort(rng.choice(positives, min(len(positives), QUERY_CAP), replace=False))
GALLERY_ROWS = {c: np.flatnonzero(np.isfinite(YS["gallery"][:, c])) for c in range(N_CLASSES)}
GALLERY_RELEVANCE = {c: (YS["gallery"][GALLERY_ROWS[c], c] >= 0.5).astype(np.int8) for c in range(N_CLASSES)}
DIAGNOSTIC_SPLIT = "dev"


def take_rows(encoded: dict, rows) -> dict:
    """Index every per-patient entry by rows; entries such as the precision matrix are copied unchanged."""
    rows = torch.as_tensor(np.asarray(rows), dtype=torch.long)
    return {key: value if key in NOT_ROW_DATA else value[rows] for key, value in encoded.items()}


def join_splits(encoded: dict, splits: list[str]) -> tuple[dict, dict[str, slice]]:
    """Concatenate several splits (one scoring call instead of one per split) and remember each split's rows."""
    joined = {key: torch.cat([encoded[s][key] for s in splits]) for key in encoded[splits[0]] if key not in NOT_ROW_DATA}
    offsets, start = {}, 0
    for split in splits:
        offsets[split] = slice(start, start + len(encoded[split]["angles"]))
        start = offsets[split].stop
    return joined, offsets


def support_rows_and_labels(draw: dict[int, tuple[np.ndarray, np.ndarray]]) -> tuple[np.ndarray, np.ndarray]:
    """Support-cohort rows of one draw; each pathology keeps only its drawn K+K labels, everything else unknown."""
    rows = np.array(sorted({int(r) for pair in draw.values() for part in pair for r in part}))
    index = {row: i for i, row in enumerate(rows)}
    labels = np.full((len(rows), N_CLASSES), np.nan, dtype=np.float32)
    for class_index, (positive, negative) in draw.items():
        labels[[index[int(r)] for r in positive], class_index] = 1.0
        labels[[index[int(r)] for r in negative], class_index] = 0.0
    return rows, labels


def support_reference(encoded_support: dict, draw: dict[int, tuple[np.ndarray, np.ndarray]]):
    """The drawn supports as a small labelled reference set, taken from the encoded SUPPORT_ROWS.
    Mahalanobis uses the (label-free) precision of all encoded support rows, which is stored with them."""
    rows, labels = support_rows_and_labels(draw)
    position = {row: i for i, row in enumerate(SUPPORT_ROWS)}
    return take_rows(encoded_support, [position[int(r)] for r in rows]), labels


def classification_rules_for(arm_name: str | None) -> tuple[str, ...]:
    """Every support-based rule; the readout only for a model whose readout was trained."""
    trained_readout = arm_name is not None and TRAINING_ARMS[arm_name].representation == "readout"
    return ("readout",) if trained_readout else (TRAINING_ARMS[arm_name].matched_rule,)


def evaluate_classification(encoded: dict, rules: tuple[str, ...], identity: dict, audit_seed: int) -> list[dict]:
    """K-shot classification rows (plain and +prior) for every rule, shot and episode.
    The readout uses no supports, so it is reported once with shot = 0 ("no support"), never as a K-shot result."""
    scored, offsets = join_splits(encoded, SCORED_SPLITS)
    rows = []

    def raw_scores(rule: str, draw) -> np.ndarray:
        if rule == "readout":
            return scored["readout"].numpy()
        reference, reference_labels = support_reference(encoded["support"], draw)
        return prototype_margin_scores(rule, scored, reference, reference_labels, audit_seed)

    for rule in rules:
        uses_support = rule != "readout"
        for shot in (SHOTS if uses_support else (0,)):
            # beta for the +prior rows: tuned on SELECT with separate tuning draws
            tuning_draws = [TUNING_DRAWS[(shot, e)] for e in range(TUNING_EPISODES)] if uses_support else [None]
            tuning = [raw_scores(rule, draw)[offsets["select"]] for draw in tuning_draws]
            tuning = [t / np.where(t.std(0) < 1e-12, 1.0, t.std(0)) for t in tuning]
            beta_auc = {beta: np.mean([macro_auc(YS["select"], PRIOR["select"] + beta * t) for t in tuning]) for beta in BETA_GRID}
            beta = max(beta_auc, key=lambda b: (beta_auc[b], -b))
            for episode in range(EPISODES if uses_support else 1):
                rows += classification_rows(raw_scores(rule, SUPPORT_DRAWS[(shot, episode)] if uses_support else None),
                                            offsets, beta, {**identity, "rule": rule, "shot": shot, "episode": episode})
    return rows


def classification_rows(raw: np.ndarray, offsets: dict, beta: float, base: dict) -> list[dict]:
    """Plain and +prior metric rows for every report split, standardised and thresholded on SELECT."""
    select_raw = raw[offsets["select"]]
    mean, std = np.nanmean(select_raw, 0, keepdims=True), np.nanstd(select_raw, 0, keepdims=True)
    std = np.where(std < 1e-12, 1.0, std)
    threshold = youden_thresholds(YS["select"], (select_raw - mean) / std)
    prior_threshold = youden_thresholds(YS["select"], PRIOR["select"] + beta * select_raw / std)
    rows = []
    for split in REPORT_SPLITS:
        split_raw = raw[offsets[split]]
        rows.append({**base, "split": split, "prior": "none",
                     **classification_metrics(YS[split], (split_raw - mean) / std, threshold)})
        rows.append({**base, "split": split, "prior": "chexzero", "beta": beta,
                     **classification_metrics(YS[split], PRIOR[split] + beta * split_raw / std, prior_threshold)})
    return rows


def evaluate_retrieval(encoded: dict, rules: tuple[str, ...], audit_seed: int) -> tuple[dict, dict]:
    """Per-query retrieval arrays for every scored split, rule and pathology, plus the dev scores kept for diagnostics."""
    arrays, diagnostic_scores = {}, {}
    for split, rule, class_index in product(SCORED_SPLITS, rules, range(N_CLASSES)):
        enforce_runtime_budget(f"retrieval {split}/{rule}/{CLASSES[class_index]}")
        if class_index == 0:
            progress(f"Retrieval {split} / {rule}")
        query_rows = QUERY_ROWS[(split, class_index)]
        if not len(query_rows) or GALLERY_RELEVANCE[class_index].sum() == 0:
            continue
        gallery = take_rows(encoded["gallery"], GALLERY_ROWS[class_index])
        scores = score_matrix(rule, take_rows(encoded[split], query_rows), gallery, class_index, audit_seed + class_index)
        if not np.isfinite(scores).all():
            raise FloatingPointError(f"non-finite scores: {rule} {split} {CLASSES[class_index]}")
        for metric, values in per_query_retrieval(scores, GALLERY_RELEVANCE[class_index]).items():
            arrays[f"{split}|{rule}|{class_index}|{metric}"] = values
        if split == DIAGNOSTIC_SPLIT:
            diagnostic_scores[(rule, class_index)] = scores[:DIAGNOSTIC_QUERIES]
    return arrays, diagnostic_scores


def evaluate_diagnostics(encoded: dict, diagnostic_scores: dict, rules: tuple[str, ...]) -> dict:
    """Purity of the gallery's mixed states, per-rule similarity spread, and rank agreement between rules."""
    diagnostics = {}
    if "density" in encoded["gallery"]:
        purity = torch.stack([density_purity(encoded["gallery"]["density"][:, c]) for c in range(N_CLASSES)], 1)   # (N, 5)
        diagnostics["mean_purity"] = float(purity.mean())
        diagnostics.update({f"purity__{name}": float(purity[:, c].mean()) for c, name in enumerate(CLASSES)})
    for rule in rules:
        blocks = [diagnostic_scores[(rule, c)] for c in range(N_CLASSES) if (rule, c) in diagnostic_scores]
        diagnostics[f"spread__{rule}"] = float(np.mean([b.std() for b in blocks])) if blocks else np.nan
    for rule_a, rule_b in combinations(rules, 2):
        taus = [mean_kendall_tau(diagnostic_scores[(rule_a, c)], diagnostic_scores[(rule_b, c)])
                for c in range(N_CLASSES) if (rule_a, c) in diagnostic_scores and (rule_b, c) in diagnostic_scores]
        diagnostics[f"tau__{rule_a}__{rule_b}"] = float(np.nanmean(taus)) if taus else np.nan
    return diagnostics


def save_result(name: str, arrays: dict, meta: dict) -> None:
    path = RESULT_DIR / name
    temporary = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(temporary, __meta__=np.array(json.dumps(meta, default=float)), **arrays)
    os.replace(temporary, path)


def load_result(path: Path) -> tuple[dict, dict]:
    with np.load(path, allow_pickle=False) as stored:
        meta = json.loads(str(stored["__meta__"]))
        arrays = {key: stored[key] for key in stored.files if key != "__meta__"}
    return meta, arrays


def encode_for_evaluation(model: H1Retriever, variant: Variant, parts) -> dict:
    encoded = {split: encode_split(model, variant, split, parts=parts) for split in ["gallery"] + SCORED_SPLITS}
    encoded["support"] = encode_split(model, variant, "support", rows=SUPPORT_ROWS, parts=parts)
    attach_precision(encoded, encoded["support"]["angles"])
    return encoded


def attach_precision(encoded: dict, support_vectors: torch.Tensor) -> None:
    """Label-free Mahalanobis precision per pathology, estimated on the support rows, stored with every reference set."""
    precision = class_precisions(support_vectors)
    for split in encoded:
        encoded[split]["precision"] = precision


def evaluate_model(spec: ModelSpec, record: dict) -> Path:
    """Encode only the representations needed for matched comparisons, then save the result."""
    result_name = f"result__{spec.model_id}__{record['fingerprint']}.npz"
    saved = find_saved_file(result_name)
    if saved is not None:
        return saved
    variant = ALL_VARIANTS[spec.variant]
    model = build_model(spec, record["state"])
    before = {key: QSTATS[key] for key in ("qnode_calls", "audit_pairs", "circuit_audit_pairs")}
    session_audit_gap, QSTATS["audit_max_gap"] = QSTATS["audit_max_gap"], 0.0      # track this model's own worst gap
    started = time.time()
    active_rules = tuple(TRAINING_ARMS[a].matched_rule for a in QUANTUM_ARMS if a in ACTIVE_TRAINING_ARMS) if spec.arm == "T0_readout" else (TRAINING_ARMS[spec.arm].matched_rule,)
    needed_rules = active_rules + classification_rules_for(spec.arm)
    parts = tuple(p for p in ENCODED_PARTS if any(p in PARTS_FOR[SCORING_RULES[r].representation] for r in needed_rules))
    progress(f"Encoding {spec.model_id}: {parts or ('angles',)}")
    encoded = encode_for_evaluation(model, variant, parts)
    encode_seconds = time.time() - started
    circuit_gap = circuit_audit(encoded["gallery"], model.thetas(), variant.entangle, stable_seed("circuit", spec.model_id))
    audit_seed = stable_seed("audit", spec.model_id)
    # Score the matched rule (and T0 controls).
    arrays, diagnostic_scores = evaluate_retrieval(encoded, active_rules, audit_seed)
    identity = {"model_id": spec.model_id, "arm": spec.arm, "variant": spec.variant, "seed": spec.seed,
                "encoder_set": variant.encoder_set}
    classification = evaluate_classification(encoded, classification_rules_for(spec.arm), identity, audit_seed)
    model_audit_gap = QSTATS["audit_max_gap"]
    QSTATS["audit_max_gap"] = max(session_audit_gap, model_audit_gap)
    meta = {**identity, "classification": classification,
            "diagnostics": evaluate_diagnostics(encoded, diagnostic_scores, active_rules),
            "provenance": {"backend": "classical" if TRAINING_ARMS[spec.arm].family == "classical" else "pennylane", "pl_device": PL_DEVICE_NAME, "pl_version": qml.__version__,
                           "circuit": CIRCUIT_ID, "similarity_engine": SIMILARITY_ENGINE,
                           **{f"{key}_in_evaluation": QSTATS[key] - value for key, value in before.items()},
                           "qnode_calls_in_training": record["qnode_calls"], "circuit_audit_gap": circuit_gap,
                           "qml_math_audit_gap": model_audit_gap},
            "cost": {"encode_seconds": encode_seconds, "evaluate_seconds": time.time() - started,
                     "train_seconds": record["train_seconds"], "seconds_per_epoch": record["seconds_per_epoch"]}}
    if meta["provenance"]["backend"] == "pennylane" and meta["provenance"]["qnode_calls_in_evaluation"] == 0:
        raise RuntimeError(f"{spec.model_id}: evaluated without any PennyLane QNode call")
    save_result(result_name, arrays, meta)
    return RESULT_DIR / result_name


def evaluate_reference_raw(encoder_set: str) -> Path:
    """C0: cosine, Euclidean and Mahalanobis on the 128-d routed feature — no circuit, no training (Q1's classical
    rules; same queries, gallery and supports as every model)."""
    result_name = f"result__{encoder_set}__{REFERENCE_RAW}__{sha_of(FEATURE_SHA[encoder_set])[:12]}.npz"
    saved = find_saved_file(result_name)
    if saved is not None:
        return saved
    encoded = {split: {"angles": FEATS[encoder_set][split]} for split in ["gallery"] + SCORED_SPLITS}
    encoded["support"] = {"angles": FEATS[encoder_set]["support"][torch.as_tensor(SUPPORT_ROWS)]}
    attach_precision(encoded, encoded["support"]["angles"])
    arrays, _ = evaluate_retrieval(encoded, REFERENCE_RAW_RULES, 0)
    identity = {"model_id": f"{encoder_set}__{REFERENCE_RAW}", "arm": REFERENCE_RAW,
                "variant": "reference", "seed": -1, "encoder_set": encoder_set}
    meta = {**identity, "classification": evaluate_classification(encoded, REFERENCE_RAW_RULES, identity, 0),
            "diagnostics": {}, "provenance": {"backend": "classical"}, "cost": {}}
    save_result(result_name, arrays, meta)
    return RESULT_DIR / result_name


def evaluate_many(records: dict[str, dict], label: str) -> dict[str, Path]:
    paths, durations = {}, []
    for index, (model_id, record) in enumerate(records.items(), 1):
        if not stage_budget_left():
            print(f"WARNING: stage budget reached before evaluating {model_id}; re-run to continue.")
            break
        progress(f"{label} evaluation {index}/{len(records)}: {model_id} starting")
        started = time.time()
        paths[model_id] = evaluate_model(ModelSpec(**record["spec"]), record)
        durations.append(time.time() - started)
        remaining_hours = (len(records) - index) * float(np.mean(durations)) / 3600
        print(f"[{label} evaluation {index}/{len(records)}] {model_id} | {elapsed()} | ~{remaining_hours:.2f} h left in this stage")
    return paths
print('Cell 10 definitions ready.', elapsed())


Cell 10 definitions ready. t+1.6 min


In [11]:
# =============================================================================
# Cell 10b — EXACT low-rank mixed-state similarity, for EVALUATION only (v6.3: chunked)
#
# Why: rho = sum_k w_k |psi_k><psi_k| = A A^H with A = [sqrt(w_1)|psi_1>, ...] (64 x V),
# so rho has rank <= V = N_VIEWS_DEFAULT, never d = 64. The THEORY table in the last
# cell already prices this: d V^2 + 4 V^3 vs 10 d^3 per pair, i.e. 3,168 vs 2,621,440.
#
#   Uhlmann   spec(rho sigma)\{0} = spec(G G^H), G = A^H B  ->  F = (sum_i s_i(G))^2
#   Bures     unchanged: -sqrt(2 - 2 sqrt(F)) of that same F
#   TraceDist rho - sigma = C J C^H, C = [A|B], J = diag(I_V, -I_V)  (2V x 2V spectrum)
#   HS        Tr(rho sigma) = ||A^H B||_F^2 ;  Tr(rho^2) = ||A^H A||_F^2
#
# These are identities, not approximations. The self-test at the bottom checks them
# against the purification QNode and the dense rules, at the existing AUDIT_TOLERANCE,
# and every scoring call still runs audit_scores against dense_reference_similarity.
#
# TRAINING IS DELIBERATELY LEFT ON THE DENSE PATH. svdvals has no defined gradient at
# repeated singular values, and Cell 7 notes the routed views share the whole-image
# component so these states are close to pure — near-degenerate spectra, exactly the
# unsafe case. Evaluation runs under torch.no_grad(), so it is unaffected.
#
# Set USE_LOWRANK = False in Cell 1b to fall back to the original engine.
# NOTE: this changes SIMILARITY_ENGINE in each result's provenance. It does NOT change
# training_fingerprint, so trained checkpoints stay reusable. Regenerate and re-approve
# the pre-registration hash with Dr. Nimi before reporting.
# =============================================================================
import contextlib

_encode_split_dense = encode_split
_score_matrix_dense = score_matrix
_prototype_margin_scores_dense = prototype_margin_scores
_support_reference_plain = support_reference

LOWRANK_DTYPE = torch.complex128          # the V x V decompositions are tiny; keep full precision


def lowrank_factor(circuit, view_angles: torch.Tensor, view_weights: torch.Tensor) -> torch.Tensor:
    """(N, V, 12), (N, V) -> (N, 64, V) factor A with rho = A A^H, trace 1.

    Normalises the weights exactly as pad_views does, so A A^H is the same rho that
    mixed_densities reads out of the purification circuit.
    """
    n, n_views, _ = view_angles.shape
    states = circuit.states(view_angles.reshape(n * n_views, N_ANGLES)).reshape(n, n_views, STATE_DIM)
    weights = view_weights.double() / view_weights.double().sum(-1, keepdim=True)
    return (states.to(LOWRANK_DTYPE) * weights.sqrt().to(LOWRANK_DTYPE)[..., None]).transpose(-2, -1)


def lowrank_factors(model: H1Retriever, view_angles: torch.Tensor, view_weights: torch.Tensor) -> torch.Tensor:
    """(N, 5, V, 12), (N, 5, V) -> (N, 5, 64, V), one factor per pathology."""
    return torch.stack([lowrank_factor(model.circuits[c], view_angles[:, c], view_weights[:, c])
                        for c in range(N_CLASSES)], 1)


def _cross_gram(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """(Q,d,Va), (R,d,Vb) -> (Q,R,Va,Vb): G = A^H B for every pair."""
    return torch.einsum("qdv,rdw->qrvw", a.conj(), b)


def _self_gram(a: torch.Tensor) -> torch.Tensor:
    """(Q,d,V) -> (Q,V,V): A^H A per patient, never the Q x Q cross product."""
    return torch.einsum("qdv,qdw->qvw", a.conj(), a)


def _hermitian(matrix: torch.Tensor) -> torch.Tensor:
    return (matrix + matrix.conj().transpose(-2, -1)) / 2


# ---- v6.3 FIX: chunked, guarded decompositions --------------------------------------
# Version 3 crashed in K-shot classification (Cell 10). The support cache scores ALL scored
# queries (select + dev + final = 240) against ALL of SUPPORT_ROWS in ONE call, i.e. tens of
# thousands of batched 12 x 12 eigh problems at once. cusolverDnXsyevBatched_bufferSize
# rejected that batch with CUSOLVER_STATUS_INVALID_VALUE. Retrieval (~1k pairs per call) and
# the self-test (4,096 pairs) passed with the same code, so the batch size is the trigger.
# Fix: at most LOWRANK_PAIR_CHUNK pairs per decomposition, an explicit NaN check (so a real
# NaN is reported as such), and a CPU LAPACK fallback if cuSOLVER still refuses a chunk.
# The values are identical; only the batching changes (checked in the self-test below).
LOWRANK_PAIR_CHUNK = 2048
LOWRANK_STATS = {"chunks": 0, "cpu_fallbacks": 0}


def _check_finite(matrix: torch.Tensor, where: str) -> None:
    real = torch.view_as_real(matrix) if matrix.is_complex() else matrix
    if not torch.isfinite(real).all():
        bad = (~torch.isfinite(real)).reshape(len(real), -1).any(-1).nonzero().flatten()[:10].tolist()
        raise FloatingPointError(f"non-finite values in {where} on the low-rank path (first rows: {bad}); "
                                 f"this is a data/weights problem, not the cuSOLVER batch limit")


def _safe_eigh(matrix: torch.Tensor, eigenvectors: bool = True):
    """Hermitian eigendecomposition; falls back to CPU LAPACK if cuSOLVER refuses the batch."""
    matrix = _hermitian(matrix)
    op = torch.linalg.eigh if eigenvectors else torch.linalg.eigvalsh
    try:
        return op(matrix)
    except RuntimeError:                           # torch.linalg.LinAlgError is a RuntimeError
        if matrix.device.type != "cuda":
            raise
        _check_finite(matrix, "eigh input")
        LOWRANK_STATS["cpu_fallbacks"] += 1
        result = op(matrix.cpu())
        return tuple(r.to(matrix.device) for r in result) if eigenvectors else result.to(matrix.device)


def _safe_svdvals(matrix: torch.Tensor) -> torch.Tensor:
    try:
        return torch.linalg.svdvals(matrix)
    except RuntimeError:
        if matrix.device.type != "cuda":
            raise
        _check_finite(matrix, "svdvals input")
        LOWRANK_STATS["cpu_fallbacks"] += 1
        return torch.linalg.svdvals(matrix.cpu()).to(matrix.device)


def _lowrank_matrix_sqrt(matrix: torch.Tensor) -> torch.Tensor:
    values, vectors = _safe_eigh(matrix)
    return (vectors * values.clamp_min(0).sqrt().to(vectors.dtype)[..., None, :]) @ vectors.conj().transpose(-2, -1)


def _lowrank_block(metric: str, a: torch.Tensor, b: torch.Tensor, gram_b_self: torch.Tensor) -> torch.Tensor:
    """(q, d, V), (R, d, V) -> (q, R) for one chunk of query rows (same formulas as v6.2)."""
    if metric in ("uhlmann", "bures"):
        fidelity = _safe_svdvals(_cross_gram(a, b)).sum(-1).square().clamp(0, 1)
        if metric == "uhlmann":
            return fidelity.float()
        return (-torch.sqrt((2 - 2 * torch.sqrt(fidelity + EPSILON)).clamp_min(0) + EPSILON)).float()

    squared_norm = lambda gram: gram.abs().square().sum((-2, -1))

    if metric == "hilbert_schmidt":
        purity_a = squared_norm(_self_gram(a))[:, None]                   # Tr(rho^2), once per patient
        purity_b = squared_norm(gram_b_self)[None, :]
        overlap = squared_norm(_cross_gram(a, b))                         # Tr(rho sigma)
        return (overlap / torch.sqrt(purity_a * purity_b).clamp_min(1e-10)).clamp(0, 1).float()

    if metric == "trace_distance":
        n_query, n_reference, rank = len(a), len(b), a.shape[-1]
        cross = _cross_gram(a, b)
        gram_a = _self_gram(a)[:, None].expand(-1, n_reference, -1, -1)
        gram_b = gram_b_self[None].expand(n_query, -1, -1, -1)
        gram = torch.cat([torch.cat([gram_a, cross], dim=-1),
                          torch.cat([cross.conj().transpose(-2, -1), gram_b], dim=-1)], dim=-2)
        signs = torch.diag(torch.cat([torch.ones(rank), -torch.ones(rank)])).to(gram)
        root = _lowrank_matrix_sqrt(gram)
        return (-0.5 * _safe_eigh(root @ signs @ root, eigenvectors=False).abs().sum(-1)).float()

    raise ValueError(f"lowrank_similarity_matrix has no mixed-state metric {metric!r}")


def lowrank_similarity_matrix(metric: str, query_factor: torch.Tensor, reference_factor: torch.Tensor,
                              pair_chunk: int | None = None) -> torch.Tensor:
    """(Q, d, V), (R, d, V) -> (Q, R), identical to the dense mixed-state rules,
    with at most `pair_chunk` (query, reference) pairs per decomposition call."""
    a, b = query_factor.to(LOWRANK_DTYPE), reference_factor.to(LOWRANK_DTYPE)
    _check_finite(a, "query factor")
    _check_finite(b, "reference factor")
    pair_chunk = LOWRANK_PAIR_CHUNK if pair_chunk is None else pair_chunk
    gram_b_self = _self_gram(b)
    rows_per_chunk = max(1, pair_chunk // max(1, len(b)))
    blocks = []
    for start in range(0, len(a), rows_per_chunk):
        LOWRANK_STATS["chunks"] += 1
        blocks.append(_lowrank_block(metric, a[start:start + rows_per_chunk], b, gram_b_self))
    return torch.cat(blocks)


@torch.no_grad()
def _lowrank_encode_split(model: H1Retriever, variant: Variant, split: str, rows=None,
                          parts=ENCODED_PARTS, batch_size: int = 256) -> dict:
    """The original encoding, plus the low-rank factor. `density` is kept because
    audit_scores and circuit_audit compare against it — that is the correctness gate."""
    encoded = _encode_split_dense(model, variant, split, rows, parts, batch_size)
    if "view_angles" in encoded:
        encoded["factor"] = torch.cat([
            lowrank_factors(model,
                            encoded["view_angles"][s:s + batch_size].to(DEVICE),
                            encoded["view_weights"][s:s + batch_size].to(DEVICE)).cpu()
            for s in range(0, len(encoded["view_angles"]), batch_size)])
    return encoded


def _lowrank_score_matrix(rule_name: str, query: dict, reference: dict, class_index: int,
                          audit_seed: int = 0) -> np.ndarray:
    rule = SCORING_RULES[rule_name]
    if rule.representation != "mixed" or "factor" not in query or "factor" not in reference:
        return _score_matrix_dense(rule_name, query, reference, class_index, audit_seed)
    with torch.no_grad():
        scores = lowrank_similarity_matrix(rule.metric,
                                           query["factor"][:, class_index].to(DEVICE),
                                           reference["factor"][:, class_index].to(DEVICE)).float().cpu()
    audit_scores(rule.metric, "mixed", query["density"][:, class_index],
                 reference["density"][:, class_index], scores, audit_seed)
    return scores.numpy()


# ---- support-matrix cache: every draw's rows are rows of SUPPORT_ROWS, so compute
#      Q x |SUPPORT_ROWS| once per (model, rule, class) and slice columns per draw.
#      60 similarity calls per evaluation -> 5, with bit-identical values.
_SCORE_CACHE: dict = {}


def _lowrank_support_reference(encoded_support: dict, draw):
    """Unchanged return value; also primes the cache and records this draw's columns.
    The returned reference object is held in the cache: the cached path is used ONLY when
    prototype_margin_scores is later called with this exact object (see below)."""
    if _SCORE_CACHE.get("support_id") != id(encoded_support["angles"]):
        _SCORE_CACHE.clear()                       # a different support encoding: start over
        _SCORE_CACHE["support_id"] = id(encoded_support["angles"])
        _SCORE_CACHE["support_encoded"] = encoded_support     # held, so the id cannot be reused
    rows, _ = support_rows_and_labels(draw)
    position = {row: i for i, row in enumerate(SUPPORT_ROWS)}
    _SCORE_CACHE["columns"] = np.array([position[int(r)] for r in rows])
    reference, labels = _support_reference_plain(encoded_support, draw)
    _SCORE_CACHE["reference"] = reference          # identity tag for this draw
    return reference, labels


def _cached_support_scores(rule_name: str, query: dict, class_index: int, audit_seed: int) -> np.ndarray:
    key = (rule_name, class_index, _SCORE_CACHE["support_id"], id(query["angles"]))
    hit = _SCORE_CACHE.get(key)
    if hit is None or hit[0] is not query["angles"]:          # guard against id() reuse
        hit = (query["angles"], score_matrix(rule_name, query, _SCORE_CACHE["support_encoded"],
                                             class_index, audit_seed))
        _SCORE_CACHE[key] = hit
    return hit[1]


def _lowrank_prototype_margin_scores(rule_name: str, query: dict, reference: dict,
                                     reference_labels: np.ndarray, audit_seed: int = 0) -> np.ndarray:
    # FIX (v6.2): the cache is valid only for the reference that support_reference just built.
    # Any other reference (e.g. select_auroc's preadapt rows during ablation training) takes the
    # original path. Previously a cache left primed by Cell 10 hijacked Cell 11's select_auroc
    # with a stale C0 support set -> KeyError: 'density' (and would have scored the wrong rows).
    if _SCORE_CACHE.get("reference") is not reference:
        return _prototype_margin_scores_dense(rule_name, query, reference, reference_labels, audit_seed)
    columns = _SCORE_CACHE["columns"]
    margins = np.full((len(query["angles"]), N_CLASSES), np.nan, dtype=np.float32)
    for class_index in range(N_CLASSES):
        known = np.isfinite(reference_labels[:, class_index])
        positive = known & (reference_labels[:, class_index] >= 0.5)
        negative = known & (reference_labels[:, class_index] < 0.5)
        if positive.any() and negative.any():
            scores = _cached_support_scores(rule_name, query, class_index,
                                            audit_seed + class_index)[:, columns]
            margins[:, class_index] = scores[:, positive].mean(1) - scores[:, negative].mean(1)
    return margins


_DENSE_ENGINE = {"encode_split": _encode_split_dense, "score_matrix": _score_matrix_dense,
                 "support_reference": _support_reference_plain,
                 "prototype_margin_scores": _prototype_margin_scores_dense}


@contextlib.contextmanager
def dense_engine():
    """Run a block (training / epoch selection) on the ORIGINAL dense engine, exactly as the
    main grid in Cell 9 was trained, then restore whatever engine was active."""
    active = {name: globals()[name] for name in _DENSE_ENGINE}
    _SCORE_CACHE.clear()
    globals().update(_DENSE_ENGINE)
    try:
        yield
    finally:
        globals().update(active)
        _SCORE_CACHE.clear()


def selftest_lowrank(n_pairs: int = 64, seed: int = 0) -> float:
    """Check the identities against the purification QNode and the dense rules."""
    generator = torch.Generator().manual_seed(seed)
    circuit = H1Circuit(entangle=True, train_theta=False).to(DEVICE)
    views = (torch.rand(2 * n_pairs, N_VIEWS_DEFAULT, N_ANGLES, generator=generator) * 2 - 1).to(DEVICE)
    weights = torch.softmax(torch.randn(2 * n_pairs, N_VIEWS_DEFAULT, generator=generator), 1).to(DEVICE)
    with torch.no_grad():
        density = circuit.mixed_densities(views, weights)       # 9-qubit purification QNode
        factor = lowrank_factor(circuit, views, weights)        # 6-qubit state QNode + sqrt(w)
    worst = float(((factor @ factor.conj().transpose(-2, -1)).to(density.dtype) - density).abs().max())
    print(f"  [lowrank] max |A A^H - rho from the mixing QNode| = {worst:.3e}")
    left, right = torch.arange(n_pairs), torch.arange(n_pairs, 2 * n_pairs)
    for metric in ("uhlmann", "bures", "trace_distance", "hilbert_schmidt"):
        dense = paired_similarity(metric, "mixed", density[left], density[right]).cpu()
        low = lowrank_similarity_matrix(metric, factor[left], factor[right]).diagonal().cpu()
        gap = float((dense - low).abs().max())
        worst = max(worst, gap)
        print(f"  [lowrank] {metric:16} max |dense - lowrank| = {gap:.3e}")
    for metric in ("uhlmann", "trace_distance", "hilbert_schmidt"):   # v6.3: chunked == one call
        whole = lowrank_similarity_matrix(metric, factor[left], factor[right], pair_chunk=10 ** 9)
        chunked = lowrank_similarity_matrix(metric, factor[left], factor[right], pair_chunk=100)
        gap = float((whole - chunked).abs().max())
        worst = max(worst, gap)
        print(f"  [lowrank] {metric:16} max |one call - chunked| = {gap:.3e}")
    if worst > AUDIT_TOLERANCE:
        raise RuntimeError(f"low-rank similarity disagrees with the dense path by {worst:.2e} "
                           f"(> AUDIT_TOLERANCE {AUDIT_TOLERANCE:.2e}); do not trust this patch")
    return worst


if USE_LOWRANK:
    print("Low-rank mixed-state similarity ENABLED for evaluation; self-test:")
    selftest_lowrank()
    encode_split = _lowrank_encode_split
    score_matrix = _lowrank_score_matrix
    support_reference = _lowrank_support_reference
    prototype_margin_scores = _lowrank_prototype_margin_scores
    SIMILARITY_ENGINE = f"{SIMILARITY_ENGINE}+lowrank_rankV_exact_eval"
    print(f"Low-rank patch verified and active (<= {LOWRANK_PAIR_CHUNK} pairs per decomposition). "
          f"SIMILARITY_ENGINE = {SIMILARITY_ENGINE}")
else:
    print("USE_LOWRANK is False: evaluation stays on the dense qml.math engine.")
print("Cell 10b ready.", elapsed())


Low-rank mixed-state similarity ENABLED for evaluation; self-test:
  [lowrank] max |A A^H - rho from the mixing QNode| = 1.639e-07
  [lowrank] uhlmann          max |dense - lowrank| = 7.451e-08
  [lowrank] bures            max |dense - lowrank| = 1.192e-07
  [lowrank] trace_distance   max |dense - lowrank| = 6.557e-07
  [lowrank] hilbert_schmidt  max |dense - lowrank| = 3.725e-08
  [lowrank] uhlmann          max |one call - chunked| = 0.000e+00
  [lowrank] trace_distance   max |one call - chunked| = 0.000e+00
  [lowrank] hilbert_schmidt  max |one call - chunked| = 0.000e+00
Low-rank patch verified and active (<= 2048 pairs per decomposition). SIMILARITY_ENGINE = pennylane_qml_math+lowrank_rankV_exact_eval
Cell 10b ready. t+1.7 min


In [12]:
RESULT_PATHS = evaluate_many(TRAINED, "main")
REFERENCE_PATHS = {encoder_set: evaluate_reference_raw(encoder_set) for encoder_set in ENCODER_SETS}
print(f"Evaluated {len(RESULT_PATHS)} models + C0 references | audits: {QSTATS['audit_pairs']} qml.math pairs "
      f"(max gap {QSTATS['audit_max_gap']:.1e}), {QSTATS['circuit_audit_pairs']} circuit pairs "
      f"(max gap {QSTATS['circuit_audit_max_gap']:.1e})")
_SCORE_CACHE.clear()          # nothing from evaluation may leak into later training
if USE_LOWRANK:
    print(f"Low-rank engine: {LOWRANK_STATS['chunks']} chunks, {LOWRANK_STATS['cpu_fallbacks']} CPU fallbacks")
print("Cell 10 ready.", elapsed())


[AGPQuant t+1.7 min] main evaluation 1/2: main__T5_mixed_trace_distance__seed42 starting
[AGPQuant t+1.7 min] Encoding main__T5_mixed_trace_distance__seed42: ('density',)
[AGPQuant t+2.0 min] HEARTBEAT | stage: QNode hs | log: /kaggle/working/agpquant_progress.log
[AGPQuant t+2.0 min] Retrieval select / mixed_trace_distance
[AGPQuant t+2.0 min] Retrieval dev / mixed_trace_distance
[AGPQuant t+2.0 min] Retrieval final / mixed_trace_distance
[main evaluation 1/2] main__T5_mixed_trace_distance__seed42 | t+2.1 min | ~0.01 h left in this stage
[AGPQuant t+2.1 min] main evaluation 2/2: main__T5_mixed_trace_distance__seed314 starting
[AGPQuant t+2.1 min] Encoding main__T5_mixed_trace_distance__seed314: ('density',)
[AGPQuant t+2.4 min] Retrieval select / mixed_trace_distance
[AGPQuant t+2.4 min] Retrieval dev / mixed_trace_distance
[AGPQuant t+2.4 min] Retrieval final / mixed_trace_distance
[AGPQuant t+2.5 min] HEARTBEAT | stage: retrieval final/mixed_trace_distance/Pleural Effusion | log: /k

## Ablations

In [13]:
# =============================================================================
# Cell 11 — Ablations A1-A5 on the best arm (chosen on SELECT retrieval, never on DEV/FINAL)
#   A1 no entanglement | A2 frozen theta | A3a patch-only views | A3b whole+patch pair | A5 original encoders
#   are trained and evaluated exactly like the main grid, with every model seed.
#   A4 K-shot fine-tuning: the best arm's trained model is fine-tuned on each 5-shot support draw with its
#   own episodic loss, then scored on the same K-shot classification task (matched rule, no prior).
#   1-shot is excluded because a single positive has no other same-label support, so the loss is undefined.
# =============================================================================
def retrieval_macro(meta_arrays: tuple[dict, dict], split: str, rule: str, metric: str = "map") -> float:
    """Macro (over pathologies) mean of a per-query retrieval metric from one result file."""
    _, arrays = meta_arrays
    values = [arrays[f"{split}|{rule}|{c}|{metric}"].mean() for c in range(N_CLASSES) if f"{split}|{rule}|{c}|{metric}" in arrays]
    return float(np.mean(values)) if values else np.nan


def select_ranking(result_paths: dict[str, Path], arms: tuple[str, ...]) -> pd.DataFrame:
    """SELECT retrieval mAP of each arm's matched rule, mean over the available seeds."""
    rows = []
    for model_id, path in result_paths.items():
        loaded = load_result(path)
        arm = loaded[0]["arm"]
        if arm in arms:
            rows.append({"arm": arm, "seed": loaded[0]["seed"],
                         "select_map": retrieval_macro(loaded, "select", TRAINING_ARMS[arm].matched_rule)})
    return pd.DataFrame(rows).groupby("arm", as_index=False).select_map.mean().sort_values("select_map", ascending=False)


QUANTUM_RANKING = select_ranking(RESULT_PATHS, QUANTUM_ARMS)
if QUANTUM_RANKING.empty:
    raise RuntimeError(f"shard {SHARD_NAME} evaluated no quantum arm, so it has no ablation target; "
                       "ablations A1/A2/A5 need one (add a quantum arm to SHARD_ARMS)")
BEST_QUANTUM_ARM = QUANTUM_RANKING.arm.iloc[0]
_ranked_mixed = QUANTUM_RANKING[QUANTUM_RANKING.arm.isin(MIXED_ARMS)]
# A shard may legitimately hold no mixed-state arm (shard 5). A3a/A3b rebuild the
# MIXTURE of views, so they are undefined without one — not a failure, just N/A.
BEST_MIXED_ARM = None if _ranked_mixed.empty else _ranked_mixed.arm.iloc[0]
print("=== SELECT retrieval mAP of the matched rule (chooses the ablation targets) ===\n"
      + QUANTUM_RANKING.round(4).to_string(index=False))
print(f"Best quantum arm: {BEST_QUANTUM_ARM} | best mixed-state arm: {BEST_MIXED_ARM}")


def ablation_target(variant: Variant) -> str:
    return BEST_MIXED_ARM if variant.applies_to == "best_mixed" else BEST_QUANTUM_ARM


APPLICABLE_ABLATIONS = [v for v in ABLATIONS.values() if ablation_target(v) is not None]
for _variant in ABLATIONS.values():
    if ablation_target(_variant) is None:
        print(f"[ablation] {_variant.name} not applicable in shard {SHARD_NAME}: it varies the mixture of "
              f"views and this shard has no mixed-state arm")
ABLATION_SPECS = [] if SMOKE else [ModelSpec(ablation_target(variant), variant.name, seed)
                  for seed in MODEL_SEEDS[:1] for variant in APPLICABLE_ABLATIONS]
with dense_engine():                     # same selection engine as the main grid (Cell 9)
    ABLATION_TRAINED = train_many(ABLATION_SPECS, "ablation")
_SCORE_CACHE.clear()
ABLATION_PATHS = evaluate_many(ABLATION_TRAINED, "ablation")


# ---- A4: K-shot fine-tuning of the best arm ----
def finetune_on_support(model: H1Retriever, arm: TrainingArm, variant: Variant, draw) -> H1Retriever:
    """A few steps of the arm's own episodic loss on the drawn supports only (their drawn labels only)."""
    rows, labels = support_rows_and_labels(draw)
    labels = torch.tensor(labels)
    features, views, view_weights = model_inputs(variant.encoder_set, variant.view_builder, "support", rows)
    tuned = copy.deepcopy(model)
    tuned.train()
    optimizer = torch.optim.AdamW([p for p in tuned.parameters() if p.requires_grad], lr=A4_FINETUNE_LR)
    for step in range(A4_FINETUNE_STEPS):
        qnode_before_step = QSTATS["qnode_calls"]
        loss = training_loss(tuned, arm, features.to(DEVICE), views.to(DEVICE), view_weights.to(DEVICE), labels.to(DEVICE))
        if QSTATS["qnode_calls"] <= qnode_before_step:
            raise RuntimeError(f"A4 support fine-tuning step {step} skipped the PennyLane circuit")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in tuned.parameters() if p.grad is not None):
            torch.nn.utils.clip_grad_norm_(tuned.parameters(), GRAD_CLIP)
            optimizer.step()
    return tuned.eval()


def evaluate_finetuning(spec: ModelSpec, record: dict) -> Path:
    arm, variant = TRAINING_ARMS[spec.arm], ALL_VARIANTS[spec.variant]
    result_name = f"result__A4_kshot_finetune__{spec.arm}__seed{spec.seed}__{record['fingerprint']}.npz"
    saved = find_saved_file(result_name)
    if saved is not None:
        return saved
    base_model = build_model(spec, record["state"])
    identity = {"model_id": f"A4_kshot_finetune__{spec.arm}__seed{spec.seed}", "arm": spec.arm,
                "variant": "A4_kshot_finetune", "seed": spec.seed, "encoder_set": variant.encoder_set}
    qnode_calls_before, rows = QSTATS["qnode_calls"], []
    for shot, episode in product(A4_FINETUNE_SHOTS, range(EPISODES)):
        enforce_runtime_budget(f"A4 {spec.model_id} episode {episode}")
        draw = SUPPORT_DRAWS[(shot, episode)]
        with dense_engine():
            tuned = finetune_on_support(base_model, arm, variant, draw)
        parts = PARTS_FOR[arm.representation]
        encoded = {split: encode_split(tuned, variant, split, parts=parts) for split in SCORED_SPLITS}
        encoded["support"] = encode_split(tuned, variant, "support", rows=SUPPORT_ROWS, parts=parts)
        scored, offsets = join_splits(encoded, SCORED_SPLITS)
        reference, reference_labels = support_reference(encoded["support"], draw)
        raw = prototype_margin_scores(arm.matched_rule, scored, reference, reference_labels, stable_seed("A4", spec.seed, episode))
        select_raw = raw[offsets["select"]]
        mean, std = np.nanmean(select_raw, 0, keepdims=True), np.nanstd(select_raw, 0, keepdims=True)
        std = np.where(std < 1e-12, 1.0, std)
        threshold = youden_thresholds(YS["select"], (select_raw - mean) / std)
        for split in REPORT_SPLITS:
            rows.append({**identity, "rule": arm.matched_rule, "shot": shot, "episode": episode, "split": split,
                         "prior": "none", **classification_metrics(YS[split], (raw[offsets[split]] - mean) / std, threshold)})
    meta = {**identity, "classification": rows, "diagnostics": {},
            "provenance": {"backend": "pennylane", "qnode_calls_in_evaluation": QSTATS["qnode_calls"] - qnode_calls_before},
            "cost": {}}
    save_result(result_name, {}, meta)
    return RESULT_DIR / result_name


A4_PATHS = {}
for seed in (() if SMOKE else MODEL_SEEDS[:1]):
    spec = ModelSpec(BEST_QUANTUM_ARM, MAIN_VARIANT.name, seed)
    if spec.model_id in TRAINED and stage_budget_left():
        A4_PATHS[f"A4_kshot_finetune__{spec.arm}__seed{seed}"] = evaluate_finetuning(spec, TRAINED[spec.model_id])
        print(f"[A4] {spec.model_id} fine-tuned and scored on {EPISODES} episodes | {elapsed()}")
print("Cell 11 ready.", elapsed())


=== SELECT retrieval mAP of the matched rule (chooses the ablation targets) ===
                    arm  select_map
T5_mixed_trace_distance      0.6962
Best quantum arm: T5_mixed_trace_distance | best mixed-state arm: T5_mixed_trace_distance
[AGPQuant t+2.6 min] ablation model 1/5: A1_no_entanglement__T5_mixed_trace_distance__seed42 starting
[AGPQuant t+3.0 min] HEARTBEAT | stage: similarity mixed/trace_distance 14/80 | log: /kaggle/working/agpquant_progress.log
[AGPQuant t+3.5 min] HEARTBEAT | stage: similarity mixed/trace_distance 6/80 | log: /kaggle/working/agpquant_progress.log
[AGPQuant t+4.0 min] HEARTBEAT | stage: similarity mixed/trace_distance 76/80 | log: /kaggle/working/agpquant_progress.log
[AGPQuant t+4.0 min] Training A1_no_entanglement__T5_mixed_trace_distance__seed42, epoch 1/4
[AGPQuant t+4.0 min] Training A1_no_entanglement__T5_mixed_trace_distance__seed42, epoch 1, batch 1/20
[AGPQuant t+4.3 min] Training A1_no_entanglement__T5_mixed_trace_distance__seed42, epoch 1, 

## Results, tests and provenance

In [14]:
# =============================================================================
# Cell 12 — Tables, pre-registered tests, diagnostics, provenance and manifest
#   Works on whatever results exist (also after a runtime stop). Replication units:
#     retrieval       -> model seeds (the gallery ranking depends only on the trained model)
#     classification  -> K-shot episodes (seed-averaged within each episode)
#   Retrieval decisions need Holm p < alpha in the two-level bootstrap AND the same sign in every seed.
# =============================================================================
ALPHA = PREREGISTRATION["alpha"]

# ---- 1. load every result ----
ALL_RESULT_PATHS = {**RESULT_PATHS, **ABLATION_PATHS, **A4_PATHS,
                    **{f"{s}__{REFERENCE_RAW}": p for s, p in REFERENCE_PATHS.items()}}
LOADED = {model_id: load_result(path) for model_id, path in ALL_RESULT_PATHS.items()}
CONDITION_FIELDS = ["variant", "encoder_set", "arm", "rule"]

RETRIEVAL = {}      # RETRIEVAL[(split, variant, encoder_set, arm, rule)][seed][class_index][metric] = per-query array
classification_rows, diagnostic_rows, provenance_rows = [], [], []
for model_id, (meta, arrays) in LOADED.items():
    for key, values in arrays.items():
        split, rule, class_index, metric = key.split("|")
        condition = (split, meta["variant"], meta["encoder_set"], meta["arm"], rule)
        RETRIEVAL.setdefault(condition, {}).setdefault(meta["seed"], {}).setdefault(int(class_index), {})[metric] = values
    classification_rows += meta["classification"]
    if meta["diagnostics"]:
        diagnostic_rows.append({"model_id": model_id, "variant": meta["variant"], "arm": meta["arm"], "seed": meta["seed"],
                                **meta["diagnostics"]})
    provenance_rows.append({"model_id": model_id, "variant": meta["variant"], "arm": meta["arm"], "seed": meta["seed"],
                            **meta["provenance"], **{f"cost_{k}": v for k, v in meta["cost"].items()}})
CLASSIFICATION = pd.DataFrame(classification_rows)
DIAGNOSTICS = pd.DataFrame(diagnostic_rows)
PROVENANCE_TABLE = pd.DataFrame(provenance_rows)

# P0: the CheXzero prior alone (constant across shots and episodes, repeated so it pairs with every episode)
p0_rows = []
for split in REPORT_SPLITS:
    metrics = classification_metrics(YS[split], PRIOR[split], youden_thresholds(YS["select"], PRIOR["select"]))
    for shot, episode in product(SHOTS, range(EPISODES)):
        p0_rows.append({"model_id": REFERENCE_PRIOR_ONLY, "arm": REFERENCE_PRIOR_ONLY, "variant": "reference", "seed": -1,
                        "encoder_set": "none", "rule": "chexzero_prior", "shot": shot, "episode": episode, "split": split,
                        "prior": "chexzero", **metrics})
CLASSIFICATION = pd.concat([CLASSIFICATION, pd.DataFrame(p0_rows)], ignore_index=True)
CLASSIFICATION.to_csv(RUN_DIR / "classification_per_episode.csv", index=False)

# ---- 2. retrieval helpers ----
CHANCE = pd.DataFrame([{"pathology": name, "gallery_known": len(GALLERY_ROWS[c]), "gallery_positive": int(GALLERY_RELEVANCE[c].sum()),
                        "chance_map": float(GALLERY_RELEVANCE[c].mean()),
                        **{f"queries_{s}": len(QUERY_ROWS[(s, c)]) for s in SCORED_SPLITS}} for c, name in enumerate(CLASSES)])
MACRO_CHANCE = float(CHANCE.chance_map.mean())


def seed_class_arrays(split: str, condition: tuple, metric: str = "map", seeds=None):
    """(seeds, one (U, Q_c) array per pathology, the query rows of each pathology)."""
    by_seed = RETRIEVAL[(split, *condition)]
    seeds = sorted(by_seed) if seeds is None else seeds
    classes = [c for c in range(N_CLASSES) if all(c in by_seed[s] for s in seeds)]
    arrays = [np.stack([by_seed[s][c][metric] for s in seeds]).astype(np.float64) for c in classes]
    return seeds, arrays, [QUERY_ROWS[(split, c)] for c in classes]


def macro_per_seed(arrays: list[np.ndarray]) -> np.ndarray:
    return np.mean([a.mean(1) for a in arrays], 0)


def retrieval_summary(split: str, condition: tuple) -> dict:
    seeds, arrays, rows = seed_class_arrays(split, condition)
    per_seed = macro_per_seed(arrays)
    row = {"split": split, **dict(zip(CONDITION_FIELDS, condition)), "n_seeds": len(seeds), "map": float(per_seed.mean()),
           "map_seed_sd": float(per_seed.std(ddof=1)) if len(seeds) > 1 else np.nan, "chance_map": MACRO_CHANCE}
    if len(seeds) > 1:
        draws = two_level_bootstrap(arrays, rows, len(YS[split]), BOOTSTRAP_REPS, stable_seed("ci", split, *condition))
        row["map_ci_low"], row["map_ci_high"] = np.nanpercentile(draws, [2.5, 97.5])
    for metric in RETRIEVAL_METRICS[1:]:
        row[metric] = float(macro_per_seed(seed_class_arrays(split, condition, metric)[1]).mean())
    return row


def paired_retrieval_test(split: str, condition_a: tuple, condition_b: tuple) -> dict:
    """A - B on per-query mAP, paired by query patient and model seed (a single-seed reference pairs with every seed)."""
    seeds_a, seeds_b = sorted(RETRIEVAL[(split, *condition_a)]), sorted(RETRIEVAL[(split, *condition_b)])
    seeds = seeds_a if seeds_b == [-1] else sorted(set(seeds_a) & set(seeds_b))
    _, arrays_a, rows = seed_class_arrays(split, condition_a, seeds=seeds)
    _, arrays_b, _ = seed_class_arrays(split, condition_b, seeds=[-1] * len(seeds) if seeds_b == [-1] else seeds)
    differences = [a - b for a, b in zip(arrays_a, arrays_b)]
    per_seed = macro_per_seed(differences)
    result = {"split": split, "a": " / ".join(map(str, condition_a)), "b": " / ".join(map(str, condition_b)),
              "n_seeds": len(seeds), "delta_map": float(per_seed.mean()),
              "seed_deltas": " ".join(f"{d:+.4f}" for d in per_seed),
              "same_sign_all_seeds": bool(np.all(per_seed > 0) or np.all(per_seed < 0))}
    if len(seeds) >= 2:
        test_seed = stable_seed("test", split, *condition_a, *condition_b)
        draws = two_level_bootstrap(differences, rows, len(YS[split]), BOOTSTRAP_REPS, test_seed)
        result["ci_low"], result["ci_high"] = np.nanpercentile(draws, [2.5, 97.5])
        result["p_bootstrap"] = bootstrap_p(draws)
    else:
        result["p_bootstrap"] = np.nan
    return result


def main_condition(arm: str, rule: str | None = None) -> tuple:
    """(variant, encoder_set, arm, rule) of a main-grid model; the rule defaults to the arm's matched rule."""
    return (MAIN_VARIANT.name, MAIN_ENCODER_SET, arm, rule or TRAINING_ARMS[arm].matched_rule)


def ablation_condition(variant: Variant) -> tuple:
    arm = ablation_target(variant)
    return (variant.name, variant.encoder_set, arm, TRAINING_ARMS[arm].matched_rule)


C0_CONDITION = ("reference", MAIN_ENCODER_SET, REFERENCE_RAW, "angle_cosine")
RETRIEVAL_ARMS = tuple(arm for arm, spec in TRAINING_ARMS.items() if spec.matched_rule in RETRIEVAL_RULES)

# ---- 3. leaderboard and the cross grid ----
LEADERBOARD = pd.DataFrame([retrieval_summary(split, condition) for (split, *condition) in RETRIEVAL
                            if split in REPORT_SPLITS]).sort_values(["split", "variant", "map"], ascending=[True, True, False])
LEADERBOARD.to_csv(RUN_DIR / "table_retrieval_leaderboard.csv", index=False)
main_rows = LEADERBOARD[(LEADERBOARD.variant == MAIN_VARIANT.name) & (LEADERBOARD.encoder_set == MAIN_ENCODER_SET)]
CROSS_GRID = {split: main_rows[main_rows.split == split].pivot_table(index="arm", columns="rule", values="map")
              .reindex(index=list(TRAINING_ARMS), columns=list(RETRIEVAL_RULES)) for split in REPORT_SPLITS}
for split, grid in CROSS_GRID.items():
    grid.to_csv(RUN_DIR / f"table_cross_grid_{split}.csv")
def is_matched_main(row) -> bool:
    """A main-grid row scored with the rule its arm was trained with."""
    return (row.variant == MAIN_VARIANT.name and row.encoder_set == MAIN_ENCODER_SET and row.arm in RETRIEVAL_ARMS
            and row.rule == TRAINING_ARMS[row.arm].matched_rule)


is_raw_reference = (LEADERBOARD.arm == REFERENCE_RAW) & (LEADERBOARD.encoder_set == MAIN_ENCODER_SET)
MATCHED = LEADERBOARD[LEADERBOARD.apply(is_matched_main, axis=1) | is_raw_reference]
MATCHED.to_csv(RUN_DIR / "table_matched_diagonal.csv", index=False)

# ---- 4. pre-registered retrieval families (Holm within family x split) ----
family_pairs = {
    "F1_quantum_vs_classical": [(main_condition(q), main_condition("C1_angle_cosine")) for q in QUANTUM_ARMS],
    "F2_training_effect": [(main_condition(q), main_condition("T0_readout", TRAINING_ARMS[q].matched_rule))
                           for q in QUANTUM_ARMS],
    "F3_no_circuit_control": [(main_condition(a), C0_CONDITION) for a in RETRIEVAL_ARMS],
    "F4_quantum_pairwise": [(main_condition(a), main_condition(b)) for a, b in combinations(QUANTUM_ARMS, 2)],
    "ablations": [(ablation_condition(variant), main_condition(ablation_target(variant)))
                  for variant in ABLATIONS.values() if ablation_target(variant) is not None],
}
test_rows = []
for family, pairs in family_pairs.items():
    for split in REPORT_SPLITS:
        for condition_a, condition_b in pairs:
            if (split, *condition_a) in RETRIEVAL and (split, *condition_b) in RETRIEVAL:
                test_rows.append({"family": family, **paired_retrieval_test(split, condition_a, condition_b)})
RETRIEVAL_TESTS = pd.DataFrame(test_rows)
if not RETRIEVAL_TESTS.empty:
    RETRIEVAL_TESTS["p_holm"] = np.nan
    for _, index in RETRIEVAL_TESTS.groupby(["family", "split"]).groups.items():
        RETRIEVAL_TESTS.loc[list(index), "p_holm"] = holm(RETRIEVAL_TESTS.loc[list(index), "p_bootstrap"].to_numpy())
    significant = (RETRIEVAL_TESTS.p_holm < ALPHA) & RETRIEVAL_TESTS.same_sign_all_seeds
    RETRIEVAL_TESTS["decision"] = np.where(~significant, "no significant difference",
                                           np.where(RETRIEVAL_TESTS.delta_map > 0, "A retrieves better", "B retrieves better"))
    confirmatory_family = RETRIEVAL_TESTS.family.isin(["F1_quantum_vs_classical", "F2_training_effect", "F3_no_circuit_control"])
    confirmatory = confirmatory_family & (RETRIEVAL_TESTS.split == PRIMARY_SPLIT)
    RETRIEVAL_TESTS["status"] = np.where(confirmatory, "confirmatory", "exploratory")
    if SMOKE:
        RETRIEVAL_TESTS["decision"] = "SMOKE: not evaluated"
RETRIEVAL_TESTS.to_csv(RUN_DIR / "table_retrieval_tests.csv", index=False)

# ---- 5. K-shot classification: summaries and tests (replication unit = episode, seeds averaged) ----
EPISODE_KEYS = ["split", "variant", "encoder_set", "arm", "rule", "shot", "prior", "episode"]
CLASSIFICATION_METRICS = ["macro_auroc", "macro_auprc", "macro_balanced_accuracy"]
EPISODE_MEANS = CLASSIFICATION.groupby(EPISODE_KEYS, as_index=False)[CLASSIFICATION_METRICS].mean()
CLASSIFICATION_SUMMARY = (EPISODE_MEANS.groupby(EPISODE_KEYS[:-1])
                          .agg(macro_auroc=("macro_auroc", "mean"), macro_auroc_ci=("macro_auroc", ci95_t),
                               macro_auprc=("macro_auprc", "mean"), macro_balanced_accuracy=("macro_balanced_accuracy", "mean"),
                               n_episodes=("episode", "nunique")).reset_index())
CLASSIFICATION_SUMMARY.to_csv(RUN_DIR / "table_classification_summary.csv", index=False)


def episode_series(split, shot, prior, variant, encoder_set, arm, rule) -> pd.Series:
    selected = EPISODE_MEANS[(EPISODE_MEANS.split == split) & (EPISODE_MEANS.shot == shot) & (EPISODE_MEANS.prior == prior)
                             & (EPISODE_MEANS.variant == variant) & (EPISODE_MEANS.encoder_set == encoder_set)
                             & (EPISODE_MEANS.arm == arm) & (EPISODE_MEANS.rule == rule)]
    return selected.set_index("episode").macro_auroc


def paired_episode_test(family, split, shot, prior_a, a, prior_b, b) -> dict:
    difference = (episode_series(split, shot, prior_a, *a) - episode_series(split, shot, prior_b, *b)).dropna()
    return {"family": family, "split": split, "shot": shot, "a": f"{' / '.join(map(str, a))} [{prior_a}]",
            "b": f"{' / '.join(map(str, b))} [{prior_b}]", "n_episodes": len(difference),
            "delta_auroc": float(difference.mean()) if len(difference) else np.nan, "ci95": ci95_t(difference),
            "episode_wins": int((difference > 0).sum()),
            "p_signflip": signflip_p(difference.to_numpy(), stable_seed(family, split, shot, a, b))}


P0_IDENTITY = ("reference", "none", REFERENCE_PRIOR_ONLY, "chexzero_prior")
classification_tests = []
for split, shot in product(REPORT_SPLITS, SHOTS):
    for arm in (a for a in RETRIEVAL_ARMS if a in ACTIVE_TRAINING_ARMS):
        identity = main_condition(arm)
        classification_tests.append(paired_episode_test("F5_prior", split, shot, "chexzero", identity, "chexzero", P0_IDENTITY))
    for arm in (a for a in QUANTUM_ARMS if a in ACTIVE_TRAINING_ARMS):
        classification_tests.append(paired_episode_test("F1c_quantum_vs_classical_classification", split, shot, "none",
                                                        main_condition(arm), "none", main_condition("C1_angle_cosine")))
    if not SMOKE and shot in A4_FINETUNE_SHOTS:
        best = main_condition(BEST_QUANTUM_ARM)
        classification_tests.append(paired_episode_test("A4_kshot_finetune", split, shot, "none",
                                                        ("A4_kshot_finetune", *best[1:]), "none", best))
CLASSIFICATION_TESTS = pd.DataFrame(classification_tests).dropna(subset=["delta_auroc"])
if not CLASSIFICATION_TESTS.empty:
    CLASSIFICATION_TESTS["p_holm"] = np.nan
    for _, index in CLASSIFICATION_TESTS.groupby(["family", "split", "shot"]).groups.items():
        CLASSIFICATION_TESTS.loc[list(index), "p_holm"] = holm(CLASSIFICATION_TESTS.loc[list(index), "p_signflip"].to_numpy())
CLASSIFICATION_TESTS.to_csv(RUN_DIR / "table_classification_tests.csv", index=False)

# ---- 6. diagnostics: are the mixed states mixed, and do the rules rank differently? ----
if not DIAGNOSTICS.empty:
    tau_columns = [c for c in DIAGNOSTICS if c.startswith("tau__")]
    DIAGNOSTIC_SUMMARY = DIAGNOSTICS.groupby(["variant", "arm"], as_index=False)[
        ["mean_purity"] + [c for c in DIAGNOSTICS if c.startswith("spread__")] + tau_columns].mean()
    DIAGNOSTIC_SUMMARY.to_csv(RUN_DIR / "table_diagnostics.csv", index=False)
    main_diagnostics = DIAGNOSTICS[DIAGNOSTICS.variant == MAIN_VARIANT.name]
    TAU_MATRIX = pd.DataFrame(np.eye(len(RETRIEVAL_RULES)), index=RETRIEVAL_RULES, columns=RETRIEVAL_RULES)
    for column in tau_columns:
        _, rule_a, rule_b = column.split("__")
        TAU_MATRIX.loc[rule_a, rule_b] = TAU_MATRIX.loc[rule_b, rule_a] = main_diagnostics[column].mean()
    TAU_MATRIX.to_csv(RUN_DIR / "table_rank_agreement_kendall_tau.csv")

# ---- 7. provenance: every quantum number must come from PennyLane ----
quantum_models = PROVENANCE_TABLE[PROVENANCE_TABLE.backend == "pennylane"]
problems = []
if QSTATS["torch_reference_calls"]:
    problems.append("the torch reference simulator ran outside the equivalence gate")
if (quantum_models.qnode_calls_in_evaluation <= 0).any():
    problems.append("some evaluated models have no QNode calls")
circuit_trained = quantum_models[quantum_models.arm.map(lambda a: a in TRAINING_ARMS and TRAINING_ARMS[a].representation != "angles")
                                 & (quantum_models.variant != "A4_kshot_finetune")]
if (circuit_trained.qnode_calls_in_training <= 0).any():
    problems.append("some circuit-trained models have no QNode calls during training")
evaluated = quantum_models[quantum_models.variant != "A4_kshot_finetune"]      # A4 re-scores without separate audits
audit_pairs = int(evaluated.audit_pairs_in_evaluation.sum())
circuit_audit_pairs = int(evaluated.circuit_audit_pairs_in_evaluation.sum())
audit_gap, circuit_gap = float(evaluated.qml_math_audit_gap.max()), float(evaluated.circuit_audit_gap.max())
if audit_gap > AUDIT_TOLERANCE or circuit_gap > AUDIT_TOLERANCE:
    problems.append("an audit gap exceeded the tolerance")
if audit_pairs == 0 or circuit_audit_pairs == 0:
    problems.append("no audits were recorded")
if problems:
    raise RuntimeError("Provenance check failed: " + "; ".join(problems))
PROVENANCE = {"backend": "pennylane", "device": PL_DEVICE_NAME, "pennylane": qml.__version__, "circuit": CIRCUIT_ID,
              "qubits": N_QUBITS, "layers": VQC_LAYERS, "qnode_torch_device": str(QNODE_DEVICE),
              "similarity_engine": SIMILARITY_ENGINE, "models_verified": int(len(quantum_models)),
              "qnode_calls_this_session": int(QSTATS["qnode_calls"]), "qnode_samples_this_session": int(QSTATS["qnode_samples"]),
              "qnode_calls_in_verified_results": int(quantum_models.qnode_calls_in_evaluation.sum()
                                                     + circuit_trained.qnode_calls_in_training.sum()),
              "torch_reference_calls_outside_gate": int(QSTATS["torch_reference_calls"]),
              "qml_math_audit_pairs": audit_pairs, "qml_math_audit_max_gap": audit_gap,
              "circuit_audit_pairs": circuit_audit_pairs, "circuit_audit_max_gap": circuit_gap}
(RUN_DIR / "quantum_provenance.json").write_text(json.dumps(PROVENANCE, indent=1))

# ---- 8. manifest ----
MANIFEST = {
    "design": "H1 circuit trained once per (training arm, seed) on the preadapt cohort; the arm's similarity is inside the "
              "training loss; every trained model is scored with every rule; headline = matched diagonal",
    "preregistration_sha256": PREREG_SHA, "preregistration": PREREGISTRATION,
    "best_quantum_arm_on_select": BEST_QUANTUM_ARM, "best_mixed_arm_on_select": BEST_MIXED_ARM,
    "models": {"main": len(RESULT_PATHS), "ablations": len(ABLATION_PATHS), "a4": len(A4_PATHS)},
    "stage_a_selected_epochs": SELECTED_EPOCHS, "stage_a_fingerprint": STAGE_A_FINGERPRINT,
    "execution": f"PennyLane {qml.__version__} {PL_DEVICE_NAME} exact statevector simulation; no hardware, no finite-shot sampling",
    "wording": "'K-shot after supervised target pre-adaptation' (the encoders and the circuit saw 1,600 labelled CheXpert "
               "patients before any K-shot step), not 'few-shot'",
    "classical_components": "ViT encoders, CheXzero prior, the 128->12 angle maps and the cosine / Euclidean / Mahalanobis rules",
    "quantum_similarities": "qml.math.fidelity_statevector, qml.math.fidelity, qml.math.trace_distance, qml.math.purity "
                            "(training loss and every score); states from QNodes incl. qml.density_matrix",
    "caveats": ["circuits are trained on preadapt features, which the Stage A encoders were fitted on",
                "the last layer's theta and final CNOT ring are common to every state, so they cannot change any quantum similarity",
                "Uhlmann and Bures rank identically; T3 differs from T2 only through training",
                "CheXzero checkpoint provenance must be disclosed (possible CheXpert-validation selection)",
                "FINAL is scored once, in FULL, and reported separately from DEV"],
    "circuit_approval": CIRCUIT_APPROVAL, "evidence_rule": "only a completed FULL run is evidence; SMOKE verifies execution",
    "review": "all results require Dr. Nimi's review before any claim",
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__,
                 "pennylane": qml.__version__, "open_clip": open_clip.__version__},
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2, default=str))

# ---- 9. report (primary split; every split is in the CSV tables) ----
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)


def show(title: str, table: pd.DataFrame, columns: list[str] | None = None, index: bool = False) -> None:
    table = table if columns is None else table[[c for c in columns if c in table]]
    print(f"\n=== {title} ===\n" + table.round(4).to_string(index=index))


def is_matched_classification(row) -> bool:
    if row.arm == REFERENCE_PRIOR_ONLY:
        return True
    return row.variant == MAIN_VARIANT.name and row.arm in TRAINING_ARMS and row.rule == TRAINING_ARMS[row.arm].matched_rule


primary_classification = CLASSIFICATION_SUMMARY[CLASSIFICATION_SUMMARY.split == PRIMARY_SPLIT]
show("RETRIEVAL CHANCE LEVEL (gallery prevalence)", CHANCE)
show(f"HEADLINE: matched diagonal, retrieval macro mAP ({PRIMARY_SPLIT})", MATCHED[MATCHED.split == PRIMARY_SPLIT],
     ["arm", "rule", "n_seeds", "map", "map_ci_low", "map_ci_high", "map_seed_sd", "chance_map", "p_at_10", "mrr"])
show(f"CROSS GRID ({PRIMARY_SPLIT}): rows = trained with, columns = scored with", CROSS_GRID[PRIMARY_SPLIT], index=True)
show(f"RETRIEVAL TESTS ({PRIMARY_SPLIT})", RETRIEVAL_TESTS[RETRIEVAL_TESTS.split == PRIMARY_SPLIT],
     ["family", "status", "a", "b", "delta_map", "ci_low", "ci_high", "seed_deltas", "p_holm", "decision"])
show(f"K-SHOT CLASSIFICATION ({PRIMARY_SPLIT}, matched rule, macro AUROC over episodes)",
     primary_classification[primary_classification.apply(is_matched_classification, axis=1)],
     ["arm", "rule", "shot", "prior", "macro_auroc", "macro_auroc_ci", "macro_auprc", "n_episodes"])
show(f"CLASSIFICATION TESTS ({PRIMARY_SPLIT})", CLASSIFICATION_TESTS[CLASSIFICATION_TESTS.split == PRIMARY_SPLIT],
     ["family", "shot", "a", "b", "delta_auroc", "ci95", "episode_wins", "p_holm"])
if not DIAGNOSTICS.empty:
    show("DIAGNOSTICS: purity of the mixed states (1 = pure)", DIAGNOSTIC_SUMMARY, ["variant", "arm", "mean_purity"])
    show("RANK AGREEMENT between scoring rules (Kendall tau, main models; 1 = identical ranking)", TAU_MATRIX, index=True)
print("\n=== QUANTUM PROVENANCE ===\n" + json.dumps(PROVENANCE, indent=1))
print(f"\nTables written to {RUN_DIR} | {elapsed()} | quantum-vs-classical claims need Dr. Nimi's review.")



=== RETRIEVAL CHANCE LEVEL (gallery prevalence) ===
       pathology  gallery_known  gallery_positive  chance_map  queries_select  queries_dev  queries_final
     Atelectasis             28                16      0.5714              14           15             14
    Cardiomegaly             35                21      0.6000              15           12             16
   Consolidation             34                15      0.4412               9           10             10
           Edema             32                15      0.4688              18           20             16
Pleural Effusion             52                28      0.5385              24           24             24

=== HEADLINE: matched diagonal, retrieval macro mAP (final) ===
                    arm                 rule  n_seeds    map  map_ci_low  map_ci_high  map_seed_sd  chance_map  p_at_10    mrr
T5_mixed_trace_distance mixed_trace_distance        2 0.7099      0.6429       0.7913       0.0677       0.524   0.6892

## Complexity and efficiency

In [15]:
if False:  # separate optional profiling run, disabled in the timed notebook
    # =============================================================================
    # Cell 13 — Complexity and efficiency of each retrieval rule
    #   Four views of cost, each written to table_complexity.csv:
    #     1. Theory      : operations per (query, gallery) pair and per patient, for what this notebook executes
    #                      (qml.math on dense 64 x 64 density matrices), with the cheaper low-rank alternative noted
    #     2. Indexing    : whether the rule is an inner product (approximate-nearest-neighbour search possible)
    #                      or only a metric / neither (every query must be compared with the whole gallery)
    #     3. Hardware    : qubits, native gates and depth of the circuit that measures the rule (from qml.specs),
    #                      and the shots needed to estimate one pair to +/- EPSILON
    #     4. Measured    : microseconds per pair with PennyLane qml.math (GPU and CPU) and with simulator circuits,
    #                      peak GPU memory, and training seconds per epoch
    #   The accuracy-cost figure plots the matched rule's retrieval mAP against measured cost per pair.
    # =============================================================================
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # noqa: E402

    EPSILON, DELTA = 0.01, 0.05                          # target precision and failure probability for shot counts
    SHOTS_PER_PROBABILITY = math.ceil(math.log(2 / DELTA) / (2 * EPSILON ** 2))   # Hoeffding bound
    D, V, A = STATE_DIM, N_VIEWS_DEFAULT, N_ANGLES
    GATES_PER_STATE = VQC_LAYERS * (4 * N_QUBITS + N_QUBITS)                     # 90 for 6 qubits x 3 layers

    # ---- 1-3. theory, indexing and hardware ----
    # Complex multiply-adds (real for the angle rules). Dense n x n eigen/singular-value decompositions count as 4 n^3.
    # Per patient, a pure state costs G gate applications on d amplitudes; a mixed state is prepared once by the
    # purification circuit (V views on 3 ancillas) and stored as a d x d matrix.
    THEORY = {
        "angle_cosine": dict(pair_formula="2A", pair_ops=2 * A, index="yes: inner product of unit vectors (exact ANN / FAISS)",
                             low_rank_note="", circuit=None),
        "angle_euclidean": dict(pair_formula="2A", pair_ops=2 * A, index="yes: L2 metric (exact ANN / FAISS)",
                                low_rank_note="", circuit=None),
        "angle_mahalanobis": dict(pair_formula="A^2 + A", pair_ops=A * A + A,
                                  index="yes: Euclidean after whitening by P^(1/2) (exact ANN / FAISS)", low_rank_note="", circuit=None),
        "pure_fidelity": dict(pair_formula="d", pair_ops=D, index="yes: <vec(|a><a|), vec(|b><b|)> in d^2 real dims",
                              low_rank_note="", circuit="overlap"),
        "mixed_hilbert_schmidt": dict(pair_formula="d^2 (purities once per patient)", pair_ops=D * D,
                                      index="yes: cosine of vec(rho), vec(sigma) in d^2 real dims",
                                      low_rank_note=f"d V^2 = {D * V * V} with rank-V factors", circuit="hs"),
        "mixed_uhlmann": dict(pair_formula="2 x 4d^3 (sqrt(rho), eig) + 2d^3 (products)", pair_ops=10 * D ** 3,
                              index="no: neither an inner product nor a metric - brute force",
                              low_rank_note=f"d V^2 + 4V^3 = {D * V * V + 4 * V ** 3} with rank-V factors", circuit=None),
        "mixed_bures": dict(pair_formula="same call as Uhlmann", pair_ops=10 * D ** 3,
                            index="metric only: metric trees, no inner-product ANN",
                            low_rank_note=f"d V^2 + 4V^3 = {D * V * V + 4 * V ** 3} with rank-V factors", circuit=None),
        "mixed_trace_distance": dict(pair_formula="4d^3 (eigenvalues of rho - sigma)", pair_ops=4 * D ** 3,
                                     index="metric only: metric trees, no inner-product ANN",
                                     low_rank_note=f"d V^2 + 9(2V)^3 = {D * V * V + 9 * (2 * V) ** 3} with rank-V factors",
                                     circuit=None),
    }
    PATIENT_OPS = {"angles": A * FEATURE_DIM, "pure": GATES_PER_STATE * D, "mixed": V * GATES_PER_STATE * D + D * D}
    HARDWARE_NOTE = {
        "pure_fidelity": "exact: compute-uncompute circuit, P(0^n) = |<a|b>|^2",
        "mixed_hilbert_schmidt": "exact: 12-qubit multiplexed circuit, P(0^n) = Tr(rho sigma); purities once per patient",
        "mixed_uhlmann": "no exact circuit: needs tomography (exponential in n) or variational fidelity estimation",
        "mixed_bures": "no exact circuit: derived from Uhlmann fidelity",
        "mixed_trace_distance": "no exact circuit: needs tomography or variational state diagonalisation",
        "angle_cosine": "classical (no circuit)", "angle_euclidean": "classical (no circuit)",
        "angle_mahalanobis": "classical (no circuit)",
    }


    def circuit_resources(kind: str) -> dict:
        """Wires, native gate count and depth of one circuit after decomposition for default.qubit."""
        theta = torch.zeros(VQC_LAYERS, N_QUBITS, 2, dtype=torch.float64)
        angles = torch.zeros(N_ANGLES, dtype=torch.float64)
        views = torch.zeros(MAX_VIEW_SLOTS, N_ANGLES, dtype=torch.float64)
        arguments = {"state": (angles, theta), "overlap": (angles, angles, theta),
                     "mixing": (torch.ones(MAX_VIEW_SLOTS, dtype=torch.complex128) / math.sqrt(MAX_VIEW_SLOTS), views, theta),
                     "hs": (torch.ones(MAX_VIEW_SLOTS ** 2, dtype=torch.complex128) / MAX_VIEW_SLOTS, views, views, theta)}[kind]
        resources = qml.specs(h1_qnode(kind), level="device")(*arguments)["resources"]
        return {"qubits": resources.num_wires, "native_gates": resources.num_gates, "depth": resources.depth,
                "two_qubit_gates": int(resources.gate_types.get("CNOT", 0))}


    RETRIEVAL_PAIRS_PER_MODEL = sum(len(QUERY_ROWS[(split, c)]) * len(GALLERY_ROWS[c])
                                    for split in REPORT_SPLITS for c in range(N_CLASSES))
    PREPARATION = circuit_resources("state")

    # ---- 4. measured cost ----
    def time_call(function, repeats: int = 3) -> float:
        """Median wall-clock seconds of function() after one warm-up call."""
        function()
        times = []
        for _ in range(repeats):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            started = time.perf_counter()
            function()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - started)
        return float(np.median(times))


    def random_block(rule: ScoringRule, n: int, device: torch.device, generator: torch.Generator) -> torch.Tensor:
        if rule.representation == "angles":
            return (torch.rand(n, A, generator=generator) * 2 - 1).to(device)
        states = F.normalize(torch.randn(n, V, D, dtype=torch.complex64, generator=generator), dim=-1)
        if rule.representation == "pure":
            return states[:, 0].to(device)
        weights = torch.softmax(torch.randn(n, V, generator=generator), 1).to(torch.complex64)
        return torch.einsum("nk,nki,nkj->nij", weights, states, states.conj()).to(device)


    def measure_rule(rule_name: str, n_queries: int = 16, n_gallery: int = 128) -> dict:
        """Microseconds per pair of the scoring path used in this notebook (qml.math for quantum rules)."""
        rule = SCORING_RULES[rule_name]
        generator = torch.Generator().manual_seed(SEED)
        measured = {}
        for label, device in (("gpu", DEVICE), ("cpu", torch.device("cpu"))):
            if label == "gpu" and DEVICE.type != "cuda":
                continue
            query, gallery = random_block(rule, n_queries, device, generator), random_block(rule, n_gallery, device, generator)
            precision = torch.eye(A, device=device) if rule.metric == "mahalanobis" else None
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
            with torch.no_grad():
                seconds = time_call(lambda: similarity_matrix(rule.metric, rule.representation, query, gallery, precision))
            measured[f"us_per_pair_{label}"] = 1e6 * seconds / (n_queries * n_gallery)
            if device.type == "cuda":
                measured["peak_gpu_mb"] = torch.cuda.max_memory_allocated() / 2 ** 20
        return measured


    def measure_circuits() -> dict:
        generator = torch.Generator().manual_seed(SEED)
        theta = torch.randn(VQC_LAYERS, N_QUBITS, 2, generator=generator) * 0.1
        angles_a, angles_b = torch.rand(256, A, generator=generator), torch.rand(256, A, generator=generator)
        views, weights = torch.rand(8, V, A, generator=generator), torch.softmax(torch.randn(8, V, generator=generator), 1)
        overlap_seconds = time_call(lambda: circuit_overlap(angles_a, angles_b, theta, True), 1)
        hilbert_schmidt_seconds = time_call(lambda: circuit_hilbert_schmidt_overlap(views, weights, views, weights, theta, True), 1)
        return {"pure_fidelity": 1e6 * overlap_seconds / 256, "mixed_hilbert_schmidt": 1e6 * hilbert_schmidt_seconds / 8}


    CIRCUIT_TIMING = measure_circuits()
    seconds_per_epoch = TRAINING_TABLE.groupby("arm").seconds_per_epoch.mean() if not TRAINING_TABLE.empty else pd.Series(dtype=float)
    matched_map = {row.rule: row for row in MATCHED[MATCHED.split == PRIMARY_SPLIT].itertuples()
                   if row.variant == MAIN_VARIANT.name}                  # the trained arms only (C0 is a separate reference)
    arm_for_rule = {arm.matched_rule: name for name, arm in TRAINING_ARMS.items()}

    complexity_rows = []
    for rule_name in RETRIEVAL_RULES:
        theory, rule = THEORY[rule_name], SCORING_RULES[rule_name]
        row = {"rule": rule_name, "family": rule.family, "representation": rule.representation,
               "pair_ops_formula": theory["pair_formula"], "pair_ops": theory["pair_ops"],
               "low_rank_alternative": theory["low_rank_note"], "patient_ops": PATIENT_OPS[rule.representation],
               "ann_indexable": theory["index"], "hardware": HARDWARE_NOTE[rule_name], **measure_rule(rule_name)}
        if theory["circuit"] is not None:
            resources = circuit_resources(theory["circuit"])
            row.update({f"circuit_{k}": v for k, v in resources.items()})
            row["shots_per_pair"] = SHOTS_PER_PROBABILITY
            row["shots_per_model_retrieval"] = SHOTS_PER_PROBABILITY * RETRIEVAL_PAIRS_PER_MODEL
            row["us_per_pair_simulated_circuit_cpu"] = CIRCUIT_TIMING[rule_name]
        if rule.representation in ("pure", "mixed"):
            row["state_prep_qubits"], row["state_prep_gates"], row["state_prep_depth"] = (
                PREPARATION["qubits"], PREPARATION["native_gates"], PREPARATION["depth"])
        arm = arm_for_rule.get(rule_name)
        row["train_seconds_per_epoch"] = float(seconds_per_epoch.get(arm, np.nan)) if arm else np.nan
        match = matched_map.get(rule_name)
        row["matched_map"] = float(match.map) if match is not None else np.nan
        row["map_ci_low"] = float(getattr(match, "map_ci_low", np.nan)) if match is not None else np.nan
        row["map_ci_high"] = float(getattr(match, "map_ci_high", np.nan)) if match is not None else np.nan
        fastest = row.get("us_per_pair_gpu", row["us_per_pair_cpu"])
        row["seconds_per_query_gallery_10k"] = fastest * 1e4 / 1e6
        row["seconds_per_query_gallery_1m"] = fastest * 1e6 / 1e6
        complexity_rows.append(row)
    COMPLEXITY = pd.DataFrame(complexity_rows)
    COMPLEXITY.to_csv(RUN_DIR / "table_complexity.csv", index=False)

    # ---- accuracy-cost figure ----
    cost_column = "us_per_pair_gpu" if "us_per_pair_gpu" in COMPLEXITY else "us_per_pair_cpu"
    plot_rows = COMPLEXITY.dropna(subset=["matched_map"])
    colours = {"quantum": "#2a78d6", "classical": "#eb6834"}            # validated categorical pair (CVD-safe)
    markers = {"quantum": "o", "classical": "s"}                         # shape repeats the colour meaning
    figure, axis = plt.subplots(figsize=(7.5, 4.8), facecolor="#fcfcfb")
    axis.set_facecolor("#fcfcfb")
    for family in ("quantum", "classical"):
        subset = plot_rows[plot_rows.family == family]
        if {"map_ci_low", "map_ci_high"} <= set(subset):
            axis.errorbar(subset[cost_column], subset.matched_map, fmt="none", ecolor=colours[family], elinewidth=1.2,
                          yerr=np.nan_to_num([subset.matched_map - subset.map_ci_low, subset.map_ci_high - subset.matched_map]), zorder=2)
        axis.scatter(subset[cost_column], subset.matched_map, s=64, color=colours[family], marker=markers[family],
                     edgecolor="#fcfcfb", linewidth=2, label=f"{family} rule (95% CI over seeds and queries)", zorder=3)
    for row in plot_rows.itertuples():
        below = row.family == "classical"                                  # classical labels below, quantum above: no collisions
        axis.annotate(row.rule.replace("mixed_", "").replace("_", " "), (getattr(row, cost_column), row.matched_map),
                      xytext=(6, -12 if below else 5), textcoords="offset points", fontsize=8, color="#52514e")
    axis.margins(x=0.18)
    axis.axhline(MACRO_CHANCE, color="#898781", linewidth=1, linestyle="--", zorder=1)
    axis.annotate("chance (gallery prevalence)", (plot_rows[cost_column].min(), MACRO_CHANCE), xytext=(0, 4),
                  textcoords="offset points", fontsize=8, color="#898781")
    axis.set_xscale("log")
    hardware = "GPU" if "gpu" in cost_column else "CPU"
    axis.set_xlabel(f"cost per (query, gallery) pair, microseconds ({hardware}, PennyLane qml.math / torch)", color="#52514e")
    axis.set_ylabel(f"retrieval macro mAP, matched rule ({PRIMARY_SPLIT})", color="#52514e")
    axis.set_title("Accuracy vs cost of each retrieval rule", color="#0b0b0b", loc="left")
    axis.grid(True, color="#e1e0d9", linewidth=0.6)
    for spine in axis.spines.values():
        spine.set_color("#c3c2b7")
    axis.tick_params(colors="#898781")
    axis.legend(frameon=False, loc="center left", labelcolor="#52514e", fontsize=8)
    figure.tight_layout()
    figure.savefig(RUN_DIR / "figure_accuracy_vs_cost.png", dpi=160)
    plt.close(figure)

    shown_columns = ["rule", "pair_ops_formula", "pair_ops", cost_column, "low_rank_alternative", "ann_indexable",
                     "circuit_qubits", "circuit_native_gates", "circuit_depth", "shots_per_pair", "matched_map"]
    print(f"=== COMPLEXITY AND EFFICIENCY (d={D}, V={V} views, {GATES_PER_STATE} gates per state; "
          f"{RETRIEVAL_PAIRS_PER_MODEL:,} retrieval pairs per model) ===")
    print(COMPLEXITY[[c for c in shown_columns if c in COMPLEXITY]].round(3).to_string(index=False))
    print(f"\nHardware: one H1 state needs {PREPARATION['qubits']} qubits, {PREPARATION['native_gates']} native gates, depth "
          f"{PREPARATION['depth']}. Estimating one probability to +/-{EPSILON} with {1 - DELTA:.0%} confidence needs "
          f"{SHOTS_PER_PROBABILITY:,} shots (Hoeffding), so one model's retrieval workload would need "
          f"{SHOTS_PER_PROBABILITY * RETRIEVAL_PAIRS_PER_MODEL:.2e} shots for pure fidelity or Hilbert-Schmidt; "
          "Uhlmann, Bures and trace distance have no exact measurement circuit at all. Gate counts are after decomposition "
          "for default.qubit; the ancilla StatePrep of the Hilbert-Schmidt circuit is counted as one gate.")
    print(f"Figure: {RUN_DIR / 'figure_accuracy_vs_cost.png'} | Cell 13 ready. {elapsed()}")
else:
    progress("Complexity microbenchmarks skipped to protect the seven-hour ceiling")


[AGPQuant t+102.8 min] Complexity microbenchmarks skipped to protect the seven-hour ceiling
